# snake-arena · MuZero — a mesma busca, sobre um modelo aprendido

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/voaneves/snake-arena/blob/main/notebooks/07_muzero.ipynb)

Deve perder para o AlphaZero — o simulador aqui é exato e gratuito. O que se mede é quanto custa não tê-lo, e a comparação é limpa porque o algoritmo de busca é **o mesmo objeto**: muda só o que a árvore percorre.

E é exatamente por ser o mesmo objeto que os três defeitos achados na primeira execução do AlphaZero (§2.27–§2.29) estavam aqui também: o PUCT dando `Q = 0` a filho não visitado — com o valor positivo que este jogo produz, a busca só confirmava a rede em vez de discordar dela —, o alvo de valor não normalizado dominando o tronco, e a temperatura transformando o alvo de política em rótulo duro. Os consertos já são o padrão aqui: ao contrário do AlphaZero, o MuZero nunca rodou sob o contrato, então não havia execução de controle a preservar. Ver [`docs/BUSCA_DEGENERADA.md`](https://github.com/voaneves/snake-arena/blob/main/docs/BUSCA_DEGENERADA.md).

**Duas colunas.** A curva oficial mede a política pura de `h`+`f`, sem busca. A célula *Veredito com busca* mede o agente como ele de fato joga, no mesmo protocolo, e grava em `meta["com_busca"]`.

**Este notebook é autocontido.** Não precisa clonar nada: o ambiente, a rede, o protocolo
de avaliação e o agente estão todos aqui dentro. O código do núcleo é **gerado a partir do
pacote** ([`voaneves/snake-arena`](https://github.com/voaneves/snake-arena)) e é byte a byte igual
em todos os notebooks — é isso que torna as curvas comparáveis.

`Runtime → Change runtime type → GPU (T4)` antes de rodar.

Assinatura do código gerado: `adbd46d52abbddf4`


In [ ]:
# @title Ambiente
import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import json, math, time, glob, csv, platform, subprocess, sys, shutil, argparse
from dataclasses import dataclass, field, asdict

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers, ops, regularizers

print("TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| backend", keras.backend.backend())
GPUS = tf.config.list_physical_devices("GPU")
print("GPU:", GPUS or "nenhuma — vai rodar em CPU, muito mais lento")
for g in GPUS:
    tf.config.experimental.set_memory_growth(g, True)


## O núcleo, gerado a partir do pacote

A célula abaixo é **gerada**. Editá-la aqui não muda o repositório e faz o teste
`tests/test_notebooks.py` acusar divergência — o que é de propósito: é o que garante que
os 21 notebooks rodem exatamente o mesmo jogo, com a mesma régua.

Para mudar algo aqui, mude no pacote e rode `python tools/gerar_notebooks.py`.


In [ ]:
# ==== GERADO A PARTIR DO PACOTE — NÃO EDITE AQUI ====
# assinatura: adbd46d52abbddf4

from __future__ import annotations

# --- snakeai/plataforma.py ---
"""Onde o notebook está rodando — Colab, Kaggle ou máquina local.

Por que isto existe
-------------------
O mesmo `.ipynb` precisa rodar nos dois serviços gratuitos, e eles diferem exatamente nos
três pontos que decidem se um treino de horas sobrevive:

======================  ==============================  ==============================
                        Colab                           Kaggle
======================  ==============================  ==============================
pasta que persiste      Google Drive, montado à mão     ``/kaggle/working``, automático
retomar depois da queda  o Drive continua lá             anexar a saída da execução
                                                        anterior em ``/kaggle/input``
baixar o resultado      ``google.colab.files.download``  painel *Output*, sem código
======================  ==============================  ==============================

A detecção é por **capacidade observada**, não por variável de ambiente decorada: `kaggle`
só se `/kaggle/working` for gravável, `colab` só se além de `google.colab` importar
existir um `/content` gravável. Um notebook rodando em qualquer outro lugar cai no caso
`local` e continua funcionando — o que também é o que faz a suíte de testes conseguir
exercitar isto aqui.

E a ordem importa, por um motivo que só aparece rodando: **o Kaggle também consegue
importar `google.colab`**. Ele traz um módulo de compatibilidade cujo `drive.mount` existe
e levanta `NotImplementedError: Mounting drive is unsupported in this environment`. Uma
detecção que perguntasse "`google.colab` importa?" primeiro chamaria o Kaggle de Colab,
tentaria montar o Drive, falharia, e cairia no fallback `/content` — que no Kaggle não é
só volátil, é **invisível**: o painel *Output* mostra `/kaggle/working` e mais nada. O
treino roda até o fim e o resultado não existe em lugar nenhum. Daí a regra: quem tem
pasta persistente própria é perguntado primeiro, e importabilidade de um stub nunca conta
como capacidade.

O problema que o Kaggle resolve
-------------------------------
No Colab a sessão cai por inatividade e o teto de uso é opaco. O Kaggle tem cota semanal
de GPU declarada e um caminho **headless**: *Save Version → Save & Run All* roda o notebook
inteiro sem aba aberta, e a saída vira um artefato versionado. Para um treino de 5 M passos
que leva ~40 minutos, isso é a diferença entre "torcer para não cair" e "enfileirar e
buscar depois".

A contrapartida é que `/kaggle/working` **não** volta sozinho na sessão seguinte: ele vira
a *saída* daquela versão. Para continuar de onde parou, anexe a saída anterior como
entrada (*Add Input → Your Work → Notebook Output*) e `semear_checkpoints` faz o resto.
"""


import os
import shutil

__all__ = ["detecta", "pasta_de_trabalho", "semear_checkpoints", "entregar_arquivo",
           "resumo_plataforma", "COLAB", "KAGGLE", "LOCAL"]

COLAB, KAGGLE, LOCAL = "colab", "kaggle", "local"


def _gravavel(caminho):
    return os.path.isdir(caminho) and os.access(caminho, os.W_OK)


def detecta():
    """`"colab"`, `"kaggle"` ou `"local"`, por capacidade observada.

    O Kaggle é testado **primeiro** de propósito: ele importa `google.colab` (um módulo de
    compatibilidade cujo `drive.mount` só levanta `NotImplementedError`), então perguntar
    pelo Colab antes o classificaria errado e mandaria o treino escrever em `/content` —
    fora do painel *Output*, ou seja, resultado nenhum no fim. Importar não é capacidade;
    ter pasta persistente é.
    """
    if _gravavel("/kaggle/working"):
        return KAGGLE
    try:
        import google.colab  # noqa: F401,PLC0415
    except Exception:
        return LOCAL
    return COLAB if _gravavel("/content") else LOCAL


def pasta_de_trabalho(usar_drive="auto", nome="snake-arena", verbose=True):
    """A pasta onde checkpoints, `runs/` e export vão viver. **Sem nada para configurar.**

    `usar_drive="auto"` é o padrão e resolve tudo sozinho: no Colab tenta montar o Drive,
    no Kaggle usa `/kaggle/working`, no local usa o diretório atual. Passar `True` ou
    `False` força o comportamento no Colab e não faz diferença nos outros dois — o mesmo
    notebook roda nos três sem editar célula, que é o ponto.

    Se a montagem do Drive falhar (o usuário recusa a autorização, ou a sessão não tem
    navegador), cai para `/content` **avisando alto**: ali o treino roda, mas a queda da
    sessão leva os checkpoints junto, e descobrir isso depois de três horas é pior do que
    ler um aviso agora.
    """
    onde = detecta()

    if onde == COLAB and usar_drive is not False:
        try:
            from google.colab import drive  # noqa: PLC0415

            drive.mount("/content/drive")
            raiz = os.path.join("/content/drive/MyDrive", nome)
        except Exception as e:
            if usar_drive is True:
                raise
            print(f"AVISO: não consegui montar o Drive ({type(e).__name__}: {e}).")
            print("       Usando /content, que NÃO sobrevive à queda da sessão —")
            print("       se o treino cair, ele recomeça do zero.")
            raiz = os.path.join("/content", nome)
    elif onde == COLAB:
        raiz = os.path.join("/content", nome)
    elif onde == KAGGLE:
        raiz = os.path.join("/kaggle/working", nome)
    else:
        raiz = os.path.abspath(nome)

    os.makedirs(raiz, exist_ok=True)
    if verbose:
        print(f"plataforma: {onde} · pasta: {raiz}")
        if onde == KAGGLE:
            print("  lembre: /kaggle/working vira a SAÍDA desta versão. Para continuar "
                  "depois,\n  anexe esta saída como entrada da próxima execução.")
    return raiz


def semear_checkpoints(ckpt_dir, verbose=True):
    """Traz checkpoints de execuções anteriores anexadas em `/kaggle/input`.

    É isto que faz "retomar" funcionar no Kaggle. A sessão nova nasce com
    `/kaggle/working` vazio; o que sobreviveu está montado **somente leitura** em
    `/kaggle/input/<algum-nome>/`. Copiamos para `ckpt_dir` só o que ainda não existe lá —
    um checkpoint desta sessão sempre vence o de uma anterior, senão retomar andaria para
    trás.

    Devolve a lista do que foi copiado. Fora do Kaggle, lista vazia e nenhum efeito.
    """
    if detecta() != KAGGLE or not os.path.isdir("/kaggle/input"):
        return []

    os.makedirs(ckpt_dir, exist_ok=True)
    copiados = []
    for raiz, _, arquivos in os.walk("/kaggle/input"):
        if os.path.basename(raiz) != "checkpoints":
            continue
        for nome in arquivos:
            if not nome.endswith((".keras", ".json")):
                continue
            destino = os.path.join(ckpt_dir, nome)
            if os.path.exists(destino):
                continue
            shutil.copyfile(os.path.join(raiz, nome), destino)
            copiados.append(destino)

    if verbose and copiados:
        print(f"  [retomada] {len(copiados)} arquivo(s) de checkpoint vieram de "
              f"/kaggle/input")
    return copiados


def entregar_arquivo(caminho, verbose=True):
    """Entrega o arquivo ao usuário, do jeito que a plataforma permite.

    No Colab dispara o download pelo navegador — que só funciona com a aba aberta. No
    Kaggle não há o que disparar: o que está em `/kaggle/working` aparece sozinho no painel
    *Output*, e é justamente por isso que o Kaggle aguenta execução headless. No local, o
    arquivo já está no disco.

    Devolve `True` só quando um download foi realmente disparado.
    """
    onde = detecta()
    if onde == COLAB:
        try:
            from google.colab import files  # noqa: PLC0415

            files.download(caminho)
            return True
        except Exception as e:                       # aba fechada, sessão sem navegador
            if verbose:
                print(f"download automático não rolou ({type(e).__name__}: {e})")
    elif onde == KAGGLE and verbose:
        print("no Kaggle não há download automático: o arquivo já está no painel "
              "**Output**,\nà direita, e é baixável de lá mesmo com a aba fechada.")
    if verbose:
        print(f"arquivo: {caminho}")
    return False


def resumo_plataforma():
    """Dicionário com plataforma e aceleradores visíveis — vai para o `meta` do registro.

    O nome é longo de propósito. No notebook gerado todos os módulos viram **um espaço de
    nomes só**, e um `resumo()` aqui colidiria com o `resumo()` de `snakeai/nets/registry.py`
    — o último inlinado venceria e o outro sumiria sem erro nenhum.
    `tests/test_notebooks.py::test_no_two_inlined_modules_define_the_same_name` tranca isso.
    """
    info = {"plataforma": detecta()}
    try:
        import tensorflow as tf  # noqa: PLC0415

        gpus = tf.config.list_physical_devices("GPU")
        info["gpus"] = [g.name for g in gpus]
        info["n_gpus"] = len(gpus)
    except Exception:
        info["gpus"], info["n_gpus"] = [], 0
    return info


# --- snakeai/env/vec_snake.py ---
"""`VecSnake` — Snake vetorizado, N tabuleiros independentes evoluindo em lote.

Este módulo é **a fonte única de verdade do ambiente**. Todo algoritmo do `snake-arena`
treina e é avaliado aqui, sem exceção — é isso que torna as curvas comparáveis. Ele não
importa TensorFlow nem Keras: é NumPy puro, roda em qualquer lugar e é rápido o bastante
para que o gargalo do treino seja a GPU, não o jogo.

O truque que faz ser rápido: em vez de uma lista de posições por cobra, guardamos uma
grade `occ` de inteiros onde `occ[n, y, x]` é **quantos passos faltam para aquela célula
ficar livre**. A cabeça recebe `occ = comprimento`; a cada passo o mundo inteiro decrementa
em 1 e a cauda some sozinha. Tudo vira operação NumPy em lote sobre `(N, B, B)` — nada de
laço Python por cobra.

Como bônus, essa grade *já é* a feature mais informativa que existe para Snake: normalizada
por comprimento, ela diz à rede **quando** cada célula vai desocupar, que é exatamente a
informação necessária para a cobra passar rente ao próprio corpo sem se prender.

Convenções fixadas pelo contrato de comparabilidade (`docs/COMPARABILITY.md`):

* tabuleiro 10x10, `starve_base = 100`;
* observação `(N, B, B, 5)` egocêntrica;
* 3 ações relativas com máscara de morte imediata;
* recompensa `+1` comer, `-1` morrer, `0` passo;
* **score = comida comida**, começando em zero. Nunca comprimento.
"""


import numpy as np

__all__ = ["VecSnake", "DIRS", "TURN", "N_ACTIONS", "N_CHANNELS",
           "N_CHANNELS_COM_FOME", "DEFAULT_SEED"]

# Direções: 0=cima(-y), 1=direita(+x), 2=baixo(+y), 3=esquerda(-x)  (sentido horário)
DIRS = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]], dtype=np.int32)
# Ações relativas: 0=vira à esquerda, 1=segue reto, 2=vira à direita
TURN = np.array([-1, 0, 1], dtype=np.int32)

N_ACTIONS = 3

#: O contrato oficial. Não mude este número — versione o contrato.
N_CHANNELS = 5

#: Com o canal de fome ligado (`VecSnake(canal_fome=True)`). **Fora do contrato**, e é para
#: ser: qualquer execução assim tem que ir para a arena com `comparable=False`.
N_CHANNELS_COM_FOME = 6

DEFAULT_SEED = 42


class VecSnake:
    """`num_envs` tabuleiros independentes de Snake evoluindo em lote.

    Observação: `(num_envs, B, B, 5)` float32, **egocêntrica** — o tabuleiro é rotacionado
    para que a cobra sempre olhe para cima. Isso colapsa as 4 simetrias de rotação e deixa
    a rede ~4x mais eficiente em amostras.

    Canais
    ------
    0. corpo (binário, sem a cabeça)
    1. cabeça
    2. decaimento da cauda: `occ / comprimento` em (0, 1]
    3. comida
    4. plano constante = comprimento / B**2  (a rede precisa saber o quão longa está)

    Parâmetros
    ----------
    num_envs : int
        Quantos tabuleiros correm em paralelo.
    board_size : int
        Lado do tabuleiro. O contrato oficial usa 10.
    starve_base : int, opcional
        Paciência base antes de morrer de fome; o limite efetivo é
        `starve_base + 2 * comprimento`. Padrão: `board_size ** 2`.
    rng : np.random.Generator, opcional
        Gerador próprio. Passe um com semente fixa para reprodutibilidade.
    canal_fome : bool, opcional
        Liga um **sexto** canal com o relógio da fome. Padrão `False`, que é o contrato.

        Existe porque os 5 canais do contrato não contêm o contador de fome, e o limite é
        `starve_base + 2·comprimento` passos sem comer. Ou seja: dois estados visualmente
        idênticos, um com fome 5 e outro com fome 105, valem coisas diferentes e a rede não
        tem como saber. Os algoritmos sem modelo toleram — o retorno **real** pune andar em
        círculo, mesmo que o crítico não veja a causa. Um modelo do mundo não tem essa
        sorte: ele só pode sonhar o que a observação carrega, e inanição simplesmente não
        existe no sonho.

        Ligar isto **quebra a comparabilidade** com todas as curvas de 5 canais: a entrada
        da rede muda. Use apenas em execuções marcadas `comparable=False`, com o motivo em
        `caveat`.
    """

    def __init__(self, num_envs=256, board_size=10, starve_base=None, rng=None,
                 canal_fome=False):
        if board_size < 6:
            raise ValueError("tabuleiro pequeno demais para o corpo inicial (mínimo 6)")
        self.canal_fome = bool(canal_fome)
        self.n_channels = N_CHANNELS_COM_FOME if self.canal_fome else N_CHANNELS
        self.n = int(num_envs)
        self.b = int(board_size)
        self.cells = self.b * self.b
        self.starve_base = self.cells if starve_base is None else int(starve_base)
        self.rng = rng if rng is not None else np.random.default_rng(DEFAULT_SEED)

        self.occ = np.zeros((self.n, self.b, self.b), dtype=np.int32)
        self.head = np.zeros((self.n, 2), dtype=np.int32)
        self.food = np.zeros((self.n, 2), dtype=np.int32)
        self.dir = np.zeros(self.n, dtype=np.int32)
        self.length = np.zeros(self.n, dtype=np.int32)
        self.steps = np.zeros(self.n, dtype=np.int32)
        self.hunger = np.zeros(self.n, dtype=np.int32)
        self.score = np.zeros(self.n, dtype=np.int32)

        self._reset_idx(np.arange(self.n))

    # ------------------------------------------------------------------- reset
    def _reset_idx(self, idx):
        """Reinicia apenas os ambientes em `idx`, em lote."""
        if idx.size == 0:
            return
        k = idx.size
        b = self.b
        self.occ[idx] = 0
        # cabeça longe das bordas para caber o corpo inicial de 3
        self.head[idx] = self.rng.integers(2, b - 2, size=(k, 2), dtype=np.int32)
        self.dir[idx] = self.rng.integers(0, 4, size=k, dtype=np.int32)
        self.length[idx] = 3
        self.steps[idx] = 0
        self.hunger[idx] = 0
        self.score[idx] = 0

        d = DIRS[self.dir[idx]]                       # (k, 2)
        for back, ttl in ((0, 3), (1, 2), (2, 1)):    # cabeça, meio, cauda
            p = self.head[idx] - back * d
            np.clip(p, 0, b - 1, out=p)
            self.occ[idx, p[:, 0], p[:, 1]] = ttl

        self._spawn_food(idx)

    def _spawn_food(self, idx):
        """Sorteia comida uniformemente entre as células livres (vetorizado)."""
        if idx.size == 0:
            return
        free = self.occ[idx].reshape(idx.size, -1) == 0
        r = self.rng.random((idx.size, self.cells))
        r[~free] = -1.0
        flat = r.argmax(axis=1)
        self.food[idx, 0] = flat // self.b
        self.food[idx, 1] = flat % self.b

    def reset(self):
        """Reinicia todos os ambientes. Retorna `(obs, mask)`."""
        self._reset_idx(np.arange(self.n))
        return self.obs(), self.action_mask()

    # -------------------------------------------------------------- observação
    def limite_de_fome(self):
        """`(N,)` passos sem comer que matam. Cresce com o corpo: comer fica mais difícil."""
        return self.starve_base + 2 * self.length

    def _raw_planes(self):
        """Os 5 canais no referencial do tabuleiro (6 com `canal_fome`), antes da rotação."""
        b, n = self.b, self.n
        occ = self.occ
        body = (occ > 0).astype(np.float32)
        head = np.zeros((n, b, b), dtype=np.float32)
        rows = np.arange(n)
        head[rows, self.head[:, 0], self.head[:, 1]] = 1.0
        body -= head                                   # cabeça sai do canal de corpo
        decay = occ.astype(np.float32) / self.length[:, None, None].astype(np.float32)
        food = np.zeros((n, b, b), dtype=np.float32)
        food[rows, self.food[:, 0], self.food[:, 1]] = 1.0
        lenpl = np.broadcast_to(
            (self.length.astype(np.float32) / self.cells)[:, None, None], (n, b, b)
        )
        planos = [body, head, decay, food, lenpl]

        if self.canal_fome:
            # Plano constante com a fração do relógio já gasta: 0 = acabou de comer,
            # 1 = morre de fome neste passo. Normalizado pelo limite **efetivo**, que
            # cresce com o corpo — assim o número significa a mesma coisa do começo ao fim,
            # que é a mesma razão de o canal de comprimento existir.
            fome = self.hunger.astype(np.float32) / np.maximum(
                self.limite_de_fome().astype(np.float32), 1.0)
            planos.append(np.broadcast_to(fome[:, None, None], (n, b, b)))

        return np.stack(planos, axis=-1)

    def obs(self):
        """Planos rotacionados para o referencial da cabeça (sempre olhando p/ cima)."""
        raw = self._raw_planes()
        out = np.empty_like(raw)
        for k in range(4):
            m = self.dir == k
            if m.any():
                out[m] = np.rot90(raw[m], k=k, axes=(1, 2))
        return out

    # ----------------------------------------------------------------- máscara
    def _next_head(self, actions):
        """Posição e direção da cabeça se `actions` fosse aplicada agora."""
        nd = (self.dir + TURN[actions]) % 4
        return self.head + DIRS[nd], nd

    def _lethal(self, pos):
        """True onde a posição mata (parede ou corpo que ainda não desocupou)."""
        b = self.b
        oob = (pos[:, 0] < 0) | (pos[:, 0] >= b) | (pos[:, 1] < 0) | (pos[:, 1] >= b)
        safe_pos = np.where(oob[:, None], 0, pos)
        # a cauda vai embora neste passo -> célula com occ<=1 estará livre
        hit = self.occ[np.arange(self.n), safe_pos[:, 0], safe_pos[:, 1]] > 1
        return oob | (hit & ~oob)

    def _raw_mask(self):
        """`(N, 3)` bool sem o *override* de beco sem saída — a verdade nua."""
        mask = np.empty((self.n, N_ACTIONS), dtype=bool)
        for a in range(N_ACTIONS):
            pos, _ = self._next_head(np.full(self.n, a, dtype=np.int32))
            mask[:, a] = ~self._lethal(pos)
        return mask

    def dead_ends(self):
        """`(N,)` bool: True onde **todas** as três ações matam.

        Existe porque `action_mask()` não permite descobrir isso — lá, um beco sem saída
        aparece como "tudo liberado". Quem precisa distinguir (testes, diagnóstico, o
        filtro de segurança) pergunta aqui.
        """
        return ~self._raw_mask().any(axis=1)

    def action_mask(self):
        """`(N, 3)` bool: True = ação não mata imediatamente.

        Se as três matam, liberamos todas (a cobra morreu de qualquer jeito) — assim a
        distribuição nunca fica sem suporte e o log-prob não vira NaN. Use `dead_ends()`
        para saber quando esse caso ocorreu.
        """
        mask = self._raw_mask()
        mask[~mask.any(axis=1)] = True
        return mask

    # -------------------------------------------------------------------- step
    def step(self, actions, shaping_coef=0.0, gamma=0.99):
        """Avança todos os ambientes um passo.

        Retorna `(obs, mask, reward, done, info)`. Ambientes terminados são resetados
        automaticamente; `obs` já é o do episódio novo, e `info` guarda as estatísticas
        do episódio que acabou.

        `info` contém:
            scores      : score final dos episódios encerrados neste passo
            lengths     : duração em passos desses episódios
            wins        : quantos encheram o tabuleiro
            deaths      : quantos morreram por colisão
            starved     : quantos foram truncados por fome
            trunc_idx   : índices dos truncados por fome
            final_obs   : observação terminal dos truncados (para bootstrap do valor)
            final_mask  : máscara terminal dos truncados
            phi_old     : potencial `-dist(cabeça, comida)/b` **antes** do passo, `(N,)`
            phi_new     : o mesmo **depois** do passo, `(N,)`
            shaping_valido : onde o delta de potencial vale, `~(morreu|venceu|comeu)`

        As três últimas são informação, não comportamento: a recompensa devolvida é
        exatamente a mesma com ou sem elas. Servem a quem precisa aplicar **mais de um**
        coeficiente de shaping sobre o mesmo passo — ver a nota no `info`.
        """
        n, b = self.n, self.b
        rows = np.arange(n)
        actions = np.asarray(actions, dtype=np.int32)

        d_old = np.abs(self.head - self.food).sum(axis=1).astype(np.float32)

        new_head, new_dir = self._next_head(actions)
        dead = self._lethal(new_head)
        new_head = np.where(dead[:, None], self.head, new_head)  # congela quem morreu

        ate = (
            (~dead)
            & (new_head[:, 0] == self.food[:, 0])
            & (new_head[:, 1] == self.food[:, 1])
        )

        # cauda anda quando não comeu
        moved = ~ate & ~dead
        self.occ[moved] = np.maximum(self.occ[moved] - 1, 0)

        self.length += ate.astype(np.int32)
        self.score += ate.astype(np.int32)
        alive = ~dead
        self.head[alive] = new_head[alive]
        self.dir[alive] = new_dir[alive]
        self.occ[rows[alive], self.head[alive, 0], self.head[alive, 1]] = self.length[alive]

        self.steps += 1
        self.hunger = np.where(ate, 0, self.hunger + 1)

        won = self.length >= self.cells
        need_food = ate & ~won
        if need_food.any():
            self._spawn_food(np.nonzero(need_food)[0])

        starved = (self.hunger >= self.limite_de_fome()) & ~dead & ~won

        # ---- recompensa
        reward = np.zeros(n, dtype=np.float32)
        reward += ate.astype(np.float32)
        reward -= dead.astype(np.float32)
        reward += won.astype(np.float32) * 2.0
        reward -= starved.astype(np.float32) * 0.5
        # O potencial é calculado **sempre**, mesmo com `shaping_coef = 0`, porque ele
        # também sai no `info`. Custa uma subtração vetorizada por passo e é o que permite
        # a um agente montar o shaping por conta própria — ver as chaves `phi_*` abaixo.
        # O delta só faz sentido quando a comida não mudou de lugar.
        d_new = np.abs(self.head - self.food).sum(axis=1).astype(np.float32)
        phi_old = (-d_old / b).astype(np.float32)
        phi_new = (-d_new / b).astype(np.float32)
        shaping_valido = ~(dead | won | ate)
        if shaping_coef > 0.0:
            delta = np.where(shaping_valido, gamma * phi_new - phi_old, 0.0)
            reward += shaping_coef * delta

        done = dead | won | starved

        # Truncamento por fome: o episódio *continuaria*, então precisamos do valor do
        # estado final para fazer bootstrap. Como o env reseta sozinho, guardamos a
        # observação terminal antes do reset (custa uma passada extra, só quando ocorre).
        starved_idx = np.nonzero(starved)[0]
        final_obs = final_mask = None
        if starved_idx.size:
            final_obs = self.obs()[starved_idx]
            final_mask = self.action_mask()[starved_idx]

        info = {
            "scores": self.score[done].copy(),
            "lengths": self.steps[done].copy(),
            "wins": int(won.sum()),
            "deaths": int(dead.sum()),
            "starved": int(starved.sum()),
            "trunc_idx": starved_idx,
            "final_obs": final_obs,
            "final_mask": final_mask,
            # --- potencial do shaping, cru. Existe para quem precisa de **mais de um**
            # coeficiente de shaping sobre o mesmo passo: com `shaping_coef=0` a
            # recompensa devolvida é a esparsa pura, e o consumidor monta a sua com
            #
            #     r_i = reward + coef_i * shaping_valido * (γ_i * phi_new - phi_old)
            #
            # O ambiente continua dono do potencial (é ele que sabe quando a comida
            # trocou de lugar); o agente só recombina. É o que o LBC usa para dar a cada
            # política da população o seu próprio `(γ_i, RS_i)` — o eixo `H` do paper —
            # sem que o `VecSnake` precise saber que existe uma população.
            "phi_old": phi_old,
            "phi_new": phi_new,
            "shaping_valido": shaping_valido,
        }
        self._reset_idx(np.nonzero(done)[0])
        return self.obs(), self.action_mask(), reward, done, info

    # -------------------------------------------------------------- utilidades
    def free_space_from(self, env_i, pos):
        """Flood-fill: quantas células livres são alcançáveis a partir de `pos`.

        Usado só no filtro de segurança da inferência, nunca no treino.
        """
        b = self.b
        occ = self.occ[env_i]
        seen = np.zeros((b, b), dtype=bool)
        stack = [(int(pos[0]), int(pos[1]))]
        seen[pos[0], pos[1]] = True
        count = 0
        while stack:
            y, x = stack.pop()
            count += 1
            for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                ny, nx = y + dy, x + dx
                if 0 <= ny < b and 0 <= nx < b and not seen[ny, nx] and occ[ny, nx] <= 1:
                    seen[ny, nx] = True
                    stack.append((ny, nx))
        return count

    # -------------------------------------------------------- estado serializável
    #: Os campos que definem completamente o estado do jogo. Nada fora desta lista
    #: influencia o futuro — é o que torna a busca em árvore possível.
    CAMPOS_ESTADO = ("occ", "head", "food", "dir", "length", "steps", "hunger", "score")

    def get_state(self):
        """Cópia do estado de todos os ambientes, como dicionário de arrays.

        Existe para a busca em árvore: o MCTS precisa voltar a um nó anterior, e a única
        forma honesta de fazer isso é restaurar o estado exato. Snake é determinístico e de
        informação perfeita, então este dicionário *é* o nó da árvore.
        """
        return {c: getattr(self, c).copy() for c in self.CAMPOS_ESTADO}

    def set_state(self, estado):
        """Restaura o estado. Não valida por desempenho — use `check_invariants` em teste."""
        for c in self.CAMPOS_ESTADO:
            getattr(self, c)[...] = estado[c]
        return self

    def estado_de(self, i):
        """O estado de um único ambiente, como dicionário de arrays sem eixo de lote."""
        return {c: getattr(self, c)[i].copy() for c in self.CAMPOS_ESTADO}

    def escrever_estado(self, i, estado_unico):
        for c in self.CAMPOS_ESTADO:
            getattr(self, c)[i] = estado_unico[c]

    # ------------------------------------------------------------- introspecção
    def check_invariants(self):
        """Levanta `AssertionError` se o estado interno estiver inconsistente.

        Barato o bastante para rodar em testes e em depuração; nunca no laço de treino.
        """
        assert (self.occ >= 0).all(), "occ negativo"
        assert ((self.occ > 0).sum(axis=(1, 2)) == self.length).all(), \
            "número de células ocupadas não bate com o comprimento"
        assert (self.occ.reshape(self.n, -1).max(axis=1) == self.length).all(), \
            "a cabeça deveria ser a célula de maior occ"
        rows = np.arange(self.n)
        assert (self.occ[rows, self.head[:, 0], self.head[:, 1]] == self.length).all(), \
            "occ na posição da cabeça não é o comprimento"
        occupied_food = self.occ[rows, self.food[:, 0], self.food[:, 1]] > 0
        assert not occupied_food.any() or (self.length >= self.cells).any(), \
            "comida dentro do corpo"
        assert (self.score == self.length - 3).all(), \
            "score deve ser comprimento - 3"

    def __repr__(self):
        extra = ", canal_fome=True" if self.canal_fome else ""
        return (
            f"VecSnake(num_envs={self.n}, board_size={self.b}, "
            f"starve_base={self.starve_base}{extra})"
        )


# --- snakeai/otimizadores.py ---
"""Otimizadores — o eixo de ablação de primeira ordem.

Onde foi parar o K-FAC
----------------------
Quatro notebooks do `colab-rl` tentaram K-FAC e nenhum roda: dependiam de
`tensorflow.contrib.kfac`, que não existe desde o TensorFlow 2. Ele **voltou**, mas não
para cá: mora em `snakeai/kfac.py` e é usado pelo `ACKTR` (`snakeai/agents/acktr.py`).

O motivo de não estar neste eixo é estrutural, não histórico. `cria_otimizador` recebe um
nome e um learning rate; um `keras.optimizers.Optimizer` recebe pares `(gradiente,
variável)`. O K-FAC precisa das **ativações de entrada** e dos **gradientes de
pré-ativação** de cada camada — coisas que só existem durante o forward/backward e que
nenhum otimizador do Keras enxerga. Espremê-lo nesta assinatura exigiria refazer o forward
por dentro do otimizador, que foi o que a API Keras do `tensorflow/kfac` fazia (arquivada
em 19/04/2026).

A **pergunta** por trás daqueles notebooks continua sendo boa: *o otimizador importa?* Este
módulo é a resposta de primeira ordem — um eixo de ablação com otimizadores que existem,
funcionam em Keras 3 e cobrem escolhas de projeto diferentes. A resposta de segunda ordem é
a curva do ACKTR ao lado da do A2C, que é o mesmo algoritmo com a curvatura ligada:

===========  ==============================================================
nome         o que muda
===========  ==============================================================
``rmsprop``  o que o repositório antigo usava na maioria dos experimentos
``adam``     momento + escala adaptativa; o padrão de fato em RL
``adamw``    Adam com decaimento de peso desacoplado — regulariza sem mexer
             na escala adaptativa, ao contrário do `weight_decay` clássico
``lion``     só o **sinal** do momento; usa muito menos memória de estado e
             costuma preferir LR ~10x menor
``sgd``      o controle: momento e nada mais. Se o eixo não separar nada,
             este aqui denuncia
===========  ==============================================================

Todos entram pelo mesmo lugar: `cfg.optimizer = "adamw"`. O resto do experimento não muda,
que é o que torna a comparação uma ablação e não uma anedota.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras

__all__ = ["OTIMIZADORES", "cria_otimizador", "LR_SUGERIDO"]

OTIMIZADORES = ("adam", "adamw", "rmsprop", "lion", "sgd")

#: Multiplicador de learning rate típico de cada otimizador, relativo ao Adam. O Lion usa
#: só o sinal do momento, então o passo tem magnitude constante e o LR precisa ser bem
#: menor; o SGD, sem escala adaptativa, precisa de bem maior. Comparar otimizadores com o
#: mesmo LR não mede otimizador — mede quem tolera aquele LR específico.
LR_SUGERIDO = {"adam": 1.0, "adamw": 1.0, "rmsprop": 1.0, "lion": 0.1, "sgd": 30.0}


def cria_otimizador(nome, learning_rate, clipnorm=None, weight_decay=1e-4, **kw):
    """Devolve um `keras.optimizers.Optimizer` pelo nome.

    `learning_rate` é o valor **base**; aplique `LR_SUGERIDO[nome]` por fora se quiser a
    escala típica de cada um. Deixar isso explícito é de propósito: um experimento que
    ajusta o LR junto com o otimizador está medindo os dois ao mesmo tempo, e precisa
    dizer isso.
    """
    nome = nome.lower()
    comum = {"learning_rate": learning_rate}
    if clipnorm is not None:
        comum["clipnorm"] = clipnorm

    if nome == "adam":
        return keras.optimizers.Adam(epsilon=1e-5, **comum, **kw)
    if nome == "adamw":
        return keras.optimizers.AdamW(epsilon=1e-5, weight_decay=weight_decay,
                                      **comum, **kw)
    if nome == "rmsprop":
        return keras.optimizers.RMSprop(rho=0.95, epsilon=1e-5, **comum, **kw)
    if nome == "lion":
        return keras.optimizers.Lion(beta_1=0.9, beta_2=0.99, **comum, **kw)
    if nome == "sgd":
        # `setdefault` e não literal: o ACKTR precisa medir o momento como variável
        # (§2.36), e um literal aqui daria `got multiple values for 'momentum'`.
        kw.setdefault("momentum", 0.9)
        kw.setdefault("nesterov", True)
        return keras.optimizers.SGD(**comum, **kw)
    raise ValueError(f"otimizador desconhecido: {nome!r}. Use um de {OTIMIZADORES}")


# --- snakeai/eval.py ---
"""Avaliação — o protocolo oficial do benchmark.

Este módulo responde à única pergunta que importa: **quanto esse agente tira, de verdade?**
Ele é deliberadamente independente de TensorFlow e Keras — recebe uma *função de política*,
não um modelo. Isso permite avaliar qualquer coisa pelo mesmo caminho: uma rede Keras, uma
tabela, uma heurística escrita à mão, ou a política aleatória que define o piso. E permite
testar a avaliação sem GPU.

Protocolo fixado pelo contrato de comparabilidade (`docs/COMPARABILITY.md`):

* 1.000 episódios, tabuleiro 10x10;
* política **greedy** (sem exploração);
* `seed = 123`;
* **sem** filtro de segurança na curva principal;
* métrica = `score` (comida comida), nunca comprimento.

Sobre o viés que este módulo corrige
------------------------------------
A forma ingênua de avaliar é rodar N ambientes em paralelo e parar assim que 1.000
episódios terminarem. Isso **subestima o agente**: episódios curtos terminam primeiro e
entram na amostra, enquanto os longos — que são justamente os bons — ainda estão correndo
quando a contagem fecha. Quanto melhor o agente, pior o viés.

A correção é simples: cada ambiente contribui com o mesmo número de episódios (os
primeiros que ele terminar), em vez de a amostra ser "os primeiros a terminar no total".
"""


import math

import numpy as np


__all__ = [
    "MASK_NEG",
    "evaluate",
    "random_baseline",
    "random_policy",
    "keras_policy",
    "apply_safety_filter",
    "verdict",
]

MASK_NEG = -1e9

#: Piso documentado no README: política aleatória com máscara, 1.000 episódios, 10x10.
PISO_ALEATORIO_10X10 = 1.08


# --------------------------------------------------------------------- políticas
def random_policy(rng=None):
    """Política uniforme sobre as ações permitidas — o piso do benchmark.

    Não é "aleatória pura": ela respeita a máscara, ou seja, já evita a morte imediata.
    É o piso honesto, porque qualquer agente do benchmark também tem a máscara.
    """
    rng = rng if rng is not None else np.random.default_rng(0)

    def politica(obs, mask):
        return np.where(mask, rng.random(mask.shape), -np.inf).astype(np.float32)

    return politica


def keras_policy(model, batch_size=None):
    """Embrulha um modelo Keras (actor-critic) numa função de política.

    O import de TensorFlow acontece aqui dentro, de propósito: quem só quer avaliar uma
    heurística não precisa ter TF instalado.
    """
    import tensorflow as tf  # noqa: PLC0415  (lazy de propósito)

    @tf.function(reduce_retracing=True)
    def _forward(obs, mask):
        saida = model(obs, training=False)
        logits = saida[0] if isinstance(saida, (list, tuple)) else saida
        return tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))

    def politica(obs, mask):
        return _forward(
            tf.convert_to_tensor(obs), tf.convert_to_tensor(mask)
        ).numpy()

    return politica


# ------------------------------------------------------------- filtro de segurança
def apply_safety_filter(env: VecSnake, logits, margin=1.0, penalty=50.0):
    """Penaliza ações que deixariam a cobra num bolso menor que o próprio corpo.

    Pós-processamento de inferência, **não aprendido**: entre as ações que a rede
    considera boas, desencoraja as que se fecham num espaço sem saída (flood-fill a partir
    da nova cabeça). Por isso ele nunca entra na curva principal do benchmark — vira
    coluna separada da tabela.

    A penalidade é grande mas finita: se *todas* as opções forem ruins, a ordem relativa
    que a rede preferia é preservada e o agente escolhe a menos pior.
    """
    out = np.array(logits, dtype=np.float32, copy=True)
    for a in range(N_ACTIONS):
        pos, _ = env._next_head(np.full(env.n, a, dtype=np.int32))
        lethal = env._lethal(pos)
        for i in range(env.n):
            if lethal[i]:
                out[i, a] = MASK_NEG
            elif env.free_space_from(i, pos[i]) < margin * env.length[i]:
                out[i, a] -= penalty
    return out


# ------------------------------------------------------------------- avaliação
def evaluate(
    policy,
    board_size=10,
    episodes=1000,
    num_envs=250,
    greedy=True,
    safety=False,
    seed=123,
    max_steps=200_000,
    rng=None,
    canal_fome=False,
):
    """Roda o protocolo oficial e devolve `(stats, scores)`.

    Parâmetros
    ----------
    policy : callable
        `policy(obs, mask) -> logits (N, 3)`. Já deve aplicar a máscara aos logits;
        `evaluate` não confia nisso e reaplica de qualquer forma.
    episodes : int
        Quantos episódios compõem a amostra. O contrato usa 1.000.
    greedy : bool
        `True` = argmax (o padrão do benchmark). `False` = amostra da softmax.
    safety : bool
        Liga o flood-fill. Fora da curva principal, por construção.
    seed : int
        Semente do ambiente. Fixa em 123 no contrato, para que a sequência de comidas
        seja a mesma para todos os algoritmos.
    canal_fome : bool
        Constrói o ambiente de avaliação com o 6º canal. **Fora do contrato** — só existe
        para que uma política treinada com `canal_fome=True` receba a observação que a
        rede espera. Se ficar `False` com uma rede de 6 canais, o erro aparece na primeira
        chamada da política, como incompatibilidade de forma (5 vs 6). A dinâmica do jogo
        é idêntica nos dois casos: o canal só muda a observação, não o ambiente, então o
        número continua sendo do mesmo protocolo — o que muda é a entrada da rede, e por
        isso a curva não é comparável com as de 5 canais.

    Cada ambiente contribui com o mesmo número de episódios — ver a nota sobre viés no
    topo do módulo.
    """
    env = VecSnake(num_envs, board_size, rng=np.random.default_rng(seed),
                   canal_fome=canal_fome)
    rng = rng if rng is not None else np.random.default_rng(seed + 1)
    obs, mask = env.reset()
    apos_passo = getattr(policy, "apos_passo", None)

    por_env = math.ceil(episodes / num_envs)
    coletados = [[] for _ in range(num_envs)]
    #: Por que cada episódio da amostra terminou. Score sozinho não distingue "o agente
    #: joga mal" de "o agente anda em círculo": um DQN greedy no começo do treino tira
    #: 0,05 morrendo **100% por fome**, e a leitura correta disso não é "não aprendeu", é
    #: "a política determinística entrou em ciclo". São problemas diferentes.
    motivos = {"fome": 0, "colisao": 0, "tabuleiro_cheio": 0}
    faltam = num_envs
    passos = 0

    while faltam > 0 and passos < max_steps:
        logits = np.asarray(policy(obs, mask), dtype=np.float32)
        logits = np.where(mask, logits, MASK_NEG)
        if safety:
            logits = apply_safety_filter(env, logits)

        if greedy:
            acoes = logits.argmax(axis=1).astype(np.int32)
        else:
            z = logits - logits.max(axis=1, keepdims=True)
            p = np.exp(z)
            p /= p.sum(axis=1, keepdims=True)
            acoes = (p.cumsum(axis=1) > rng.random((num_envs, 1))).argmax(axis=1).astype(np.int32)

        obs, mask, r, done, info = env.step(acoes)
        passos += 1

        # Políticas com estado recorrente (DreamerV3) precisam saber o que de fato
        # aconteceu: a ação escolhida — que pode não ser o argmax, se o filtro de
        # segurança agiu — e onde o episódio terminou, para zerar o estado latente ali.
        # Políticas sem memória simplesmente não expõem este método.
        if apos_passo is not None:
            apos_passo(acoes, done)

        # `info["scores"]` é o score **final** do episódio, já contando a comida do
        # último passo. Ler `env.score` antes do passo perde exatamente um ponto nos
        # episódios que terminam comendo — que são precisamente as vitórias. Ver
        # `test_eval.py::test_a_winning_episode_scores_the_last_apple`.
        truncados = set(info["trunc_idx"].tolist())
        for j, i in enumerate(np.nonzero(done)[0]):
            if len(coletados[i]) < por_env:
                s_final = int(info["scores"][j])
                coletados[i].append(s_final)
                if i in truncados:
                    motivos["fome"] += 1
                elif s_final == board_size * board_size - 3:
                    motivos["tabuleiro_cheio"] += 1
                else:
                    motivos["colisao"] += 1
                if len(coletados[i]) == por_env:
                    faltam -= 1

    scores = np.array([s for lista in coletados for s in lista][:episodes], dtype=np.int32)
    if scores.size == 0:
        raise RuntimeError("nenhum episódio terminou — aumente `max_steps`")

    perfeito = board_size * board_size - 3
    # A taxa de vitória sai da **amostra coletada**, não de um contador do laço: o laço
    # continua rodando os ambientes que já cumpriram a cota, e somar as vitórias deles
    # daria uma taxa que não corresponde aos episódios de fato medidos.
    stats = {
        "episodes": int(scores.size),
        "score_mean": float(scores.mean()),
        "score_median": float(np.median(scores)),
        "score_std": float(scores.std()),
        "score_max": int(scores.max()),
        "score_p95": float(np.percentile(scores, 95)),
        "win_rate": float((scores == perfeito).mean()),
        "perfect_possible": perfeito,
        "env_steps_used": int(passos),
        "completo": bool(faltam == 0),
    }
    total_motivos = max(1, sum(motivos.values()))
    stats.update({f"fim_{k}": v / total_motivos for k, v in motivos.items()})
    return stats, scores


def random_baseline(board_size=10, episodes=1000, num_envs=250, seed=123):
    """O piso: política uniforme sobre as ações permitidas.

    É o número contra o qual todo resultado do benchmark é lido. Num 10x10 ele vale
    ~1,08 — qualquer coisa que não esteja bem acima disso não aprendeu nada.
    """
    stats, _ = evaluate(
        random_policy(np.random.default_rng(seed)),
        board_size=board_size,
        episodes=episodes,
        num_envs=num_envs,
        greedy=False,
        seed=seed,
    )
    return stats["score_mean"]


# ---------------------------------------------------------------------- veredito
def verdict(policy, board_size=10, episodes=1000, num_envs=250, com_filtro=True, seed=123,
            canal_fome=False):
    """A resposta objetiva para "aprendeu mesmo?".

    Roda, na mesma execução, três regimes e devolve a tabela:

    ===========================  =============================================
    regime                       o que mede
    ===========================  =============================================
    aleatório com máscara        o piso — quanto se tira sem aprender nada
    agente (greedy)              a política pura, sem nenhuma ajuda externa
    agente + filtro de segurança o teto prático, com o flood-fill ligado
    ===========================  =============================================

    Se a linha do meio não estiver bem acima do piso, não aprendeu — e aí o problema é de
    hiperparâmetro ou de tempo de treino, não do código.

    `canal_fome` acompanha o ambiente de treino — ver `evaluate`. O piso não muda: a
    política aleatória não olha a observação, e o canal extra não altera a dinâmica.
    """
    linhas = []

    piso = random_baseline(board_size, episodes, num_envs, seed)
    linhas.append({"regime": "aleatório com máscara", "score_mean": piso})

    st, sc = evaluate(policy, board_size=board_size, episodes=episodes,
                      num_envs=num_envs, greedy=True, seed=seed,
                      canal_fome=canal_fome)
    linhas.append({"regime": "agente (greedy)", "scores": sc, **st})

    if com_filtro:
        # o flood-fill é laço Python: menos ambientes, para não ficar lento
        stf, scf = evaluate(policy, board_size=board_size, episodes=episodes,
                            num_envs=min(num_envs, 64), greedy=True, safety=True,
                            seed=seed, canal_fome=canal_fome)
        linhas.append({"regime": "agente + filtro de segurança", "scores": scf, **stf})

    return {
        "piso": piso,
        "perfeito": board_size * board_size - 3,
        "ganho_sobre_o_piso": linhas[1]["score_mean"] / max(piso, 1e-9),
        "linhas": linhas,
    }


def format_verdict(resultado):
    """Formata o retorno de `verdict` como tabela de texto."""
    larg = 30
    out = [f"{'regime':<{larg}}{'média':>8}{'mediana':>9}{'máx':>6}{'cheio':>8}", "-" * (larg + 31)]
    for ln in resultado["linhas"]:
        med = f"{ln['score_median']:.0f}" if "score_median" in ln else "-"
        mx = f"{ln['score_max']}" if "score_max" in ln else "-"
        wr = f"{ln['win_rate']:.1%}" if "win_rate" in ln else "-"
        out.append(f"{ln['regime']:<{larg}}{ln['score_mean']:>8.2f}{med:>9}{mx:>6}{wr:>8}")
    out.append("-" * (larg + 31))
    out.append(
        f"score perfeito: {resultado['perfeito']}   |   "
        f"ganho sobre o piso: {resultado['ganho_sobre_o_piso']:.1f}x"
    )
    ag = resultado["linhas"][1]
    if "fim_fome" in ag:
        out.append(
            f"como terminou: fome {ag['fim_fome']:.0%} · colisão {ag['fim_colisao']:.0%}"
            f" · tabuleiro cheio {ag['fim_tabuleiro_cheio']:.0%}"
        )
        # Morrer de fome é o fim NORMAL aqui: a máscara de morte impede a colisão, então
        # até a política aleatória termina 85% dos episódios por fome. O que denuncia o
        # ciclo é a combinação — quase nenhuma colisão **e** score abaixo do piso, ou seja,
        # a cobra anda para sempre sem nunca comer.
        if ag["fim_colisao"] < 0.05 and ag["score_mean"] < resultado["piso"]:
            out.append(
                "  ⚠ nunca colide e não come: a política determinística entrou em ciclo.\n"
                "    Não é 'jogou mal' — é falta de exploração na hora de agir. Normal cedo\n"
                "    num DQN greedy, e é por isso que o score de TREINO (ε-greedy) fica\n"
                "    acima do de AVALIAÇÃO (greedy) nesta fase."
            )
    return "\n".join(out)


# --- snakeai/record.py ---
"""Registro de execuções — o esquema do `history.json` e o validador do contrato.

Este módulo é o porteiro do benchmark. Toda execução de todo algoritmo escreve o mesmo
arquivo, com os mesmos campos, e passa pela mesma validação antes de virar uma linha no
gráfico. **Um resultado que não valida não entra na arena** — não porque seja ruim, mas
porque não é comparável, que é pior.

A regra vale inclusive para as curvas históricas do `colab-rl`: elas são convertidas para
este mesmo esquema, mas com `comparable=False` e o motivo registrado em `caveat`. Assim
elas aparecem no gráfico como contexto (tracejado cinza) sem nunca serem confundidas com
um competidor.

Sem dependências além da biblioteca padrão e do NumPy — o validador roda no CI em segundos.
"""


import json
import os
import platform
import subprocess
import sys
import time
from dataclasses import asdict, dataclass, field

import numpy as np

__all__ = [
    "SCHEMA_VERSION",
    "CONTRATO",
    "ORCAMENTO_OFICIAL",
    "SEMENTES_OFICIAIS",
    "ContractViolation",
    "RunRecord",
    "Recorder",
    "validate",
    "save",
    "load",
    "load_all",
    "configuracoes_incompletas",
    "from_legacy_csv",
]

#: 2 — `busca` virou campo de primeira classe; antes era `meta["com_busca"]`,
#: gravado com `skip_validation=True` e portanto fora de qualquer conferência.
#: `load` migra o lugar antigo, então nenhum `history.json` precisou ser reescrito.
SCHEMA_VERSION = 2

#: Os valores que **todos** os resultados oficiais precisam compartilhar.
#: Espelha a tabela do README; mudar aqui é mudar o contrato, e invalida o histórico.
CONTRATO = {
    "env": "VecSnake",
    "board_size": 10,
    "starve_base": 100,
    "n_channels": 5,
    "n_actions": 3,
    "obs": "egocentric",
    "metric": "score",
    "reward_food": 1.0,
    "reward_death": -1.0,
    "eval_episodes": 1000,
    "eval_seed": 123,
    "eval_greedy": True,
    "eval_safety": False,
}

#: Orçamento oficial, em passos de ambiente. Fica fora do `CONTRATO` porque não descreve o
#: ambiente, mas é igualmente obrigatório: comparar um algoritmo que treinou 5 M passos com
#: outro que treinou 500 mil não mede algoritmo, mede paciência. Validado a partir de
#: `config["total_steps"]`.
ORCAMENTO_OFICIAL = 5_000_000

#: Sementes por configuração. Era convenção escrita no `COMPARABILITY.md` e nada mais —
#: e foi assim que a arena publicou um ACKTR de **uma** semente ao lado de um PPO de três,
#: com a amplitude entre sementes do PPO valendo 19 pontos. Convenção que ninguém confere
#: não é contrato. Ver `configuracoes_incompletas`.
SEMENTES_OFICIAIS = 3

#: Piso e teto do 10x10, medidos e documentados no README.
PISO_ALEATORIO = 1.21
SCORE_PERFEITO = 97


class ContractViolation(Exception):
    """Levantada quando um registro não obedece ao contrato de comparabilidade."""


# ------------------------------------------------------------------- estrutura
@dataclass
class RunRecord:
    """Uma execução completa de um algoritmo, com curva e resultado final.

    Campos
    ------
    algo, variant, seed
        Identidade da execução. `runs/<algo>/<variant>/seed<N>/history.json`.
    net, params
        Tronco usado e número de parâmetros treináveis — o eixo "arquitetura importa?".
    config
        Hiperparâmetros do agente, como dicionário livre. Não é validado; é documentação.
    env_spec
        O recorte do contrato que esta execução usou. **É** validado.
    curve
        Lista de pontos ao longo do treino. Cada ponto tem, no mínimo, `global_step`;
        `eval_score_mean` aparece só nos passos em que a avaliação rodou.
    final
        O `stats` de `snakeai.eval.evaluate` para o modelo do **último** passo. É este que
        entra na curva e na arena.
    melhor
        O mesmo `stats`, para o **melhor checkpoint** já visto, mais o `global_step` em que
        ele apareceu. RL profundo não melhora monotonicamente — não há garantia nenhuma
        fora do caso tabular — e uma execução pode terminar pior do que já esteve. Guardar
        os dois separa duas perguntas diferentes: *como o algoritmo terminou* (final) e
        *o melhor que ele produziu* (melhor). A primeira é a da arena; a segunda é a de
        quem vai levar o modelo para o jogo. Ver `docs/COMPARABILITY.md`.
    busca
        Os mesmos `stats`, para o agente medido **com a máquina que ele usa para jogar** —
        a busca em árvore do AlphaZero e do MuZero. Um dicionário `"<checkpoint>_sims<N>"`
        → `stats`, porque um agente pode ser medido em mais de um orçamento de busca e em
        mais de um checkpoint.

        Mora aqui, e não em `meta`, porque **é um resultado**, e resultado se valida. Fica
        numa coluna separada de `final` pelo motivo do `docs/COMPARABILITY.md`: a busca
        gasta dezenas de avaliações de rede por jogada contra uma do PPO, então ela não
        divide eixo com a curva oficial. As três colunas respondem a três perguntas:
        *como o algoritmo terminou* (`final`), *o melhor que ele produziu* (`melhor`) e
        *o que você levaria para jogar* (`busca`).

        Cada entrada carrega `num_simulations`, `checkpoint` e `episodes`. Só as que
        cumprem o protocolo do contrato (1000 episódios, `completo=True`) contam para a
        arena — as demais ficam gravadas, marcadas, como o que são: uma espiada.
    comparable, caveat
        `False` marca uma curva que entra no gráfico como contexto histórico, com o
        motivo em `caveat`. Toda execução nova nasce `True`.
    """

    algo: str
    variant: str = "default"
    seed: int = 0
    net: str = ""
    params: int = 0
    config: dict = field(default_factory=dict)
    env_spec: dict = field(default_factory=lambda: dict(CONTRATO))
    curve: list = field(default_factory=list)
    final: dict = field(default_factory=dict)
    melhor: dict = field(default_factory=dict)
    busca: dict = field(default_factory=dict)
    comparable: bool = True
    caveat: str = ""
    meta: dict = field(default_factory=dict)
    schema_version: int = SCHEMA_VERSION

    # ---------------------------------------------------------------- derivados
    @property
    def run_id(self):
        return f"{self.algo}/{self.variant}/seed{self.seed}"

    @property
    def rel_path(self):
        return os.path.join("runs", self.algo, self.variant, f"seed{self.seed}",
                            "history.json")

    def steps(self):
        return np.array([p["global_step"] for p in self.curve], dtype=np.int64)

    @property
    def oficial(self):
        """Pode competir na arena? Comparável **e** sem violação de contrato registrada.

        Separado de `comparable` de propósito: uma execução de fumaça não é uma curva
        histórica. Ela não compete, mas também não vira contexto — simplesmente não
        aparece, e o motivo fica em `meta["contract_violations"]`.
        """
        return self.comparable and not self.meta.get("contract_violations")

    @property
    def busca_oficial(self):
        """As entradas de `busca` que cumprem o protocolo do contrato.

        Mesma régua de `final`: 1000 episódios e `completo=True`. Uma medição de 200
        episódios tem erro padrão ~2× o da oficial e uma que estourou o teto de tempo é
        uma amostra enviesada para episódios **curtos** — justamente os ruins. As duas
        ficam gravadas, e nenhuma das duas entra na arena.
        """
        return {k: st for k, st in (self.busca or {}).items()
                if isinstance(st, dict)
                and st.get("episodes") == CONTRATO["eval_episodes"]
                and st.get("completo", False)}

    def melhor_com_busca(self, checkpoint=None):
        """A melhor entrada oficial de `busca`, ou `None`.

        `checkpoint` filtra por `"last"`/`"best"`; sem ele, o melhor de qualquer um. Não
        há escolha "correta" entre os dois — quem leva o modelo para jogar leva o melhor
        que tem —, e é por isso que o critério fica explícito aqui em vez de implícito
        num max espalhado pelo gráfico.
        """
        itens = [(k, st) for k, st in self.busca_oficial.items()
                 if checkpoint is None or st.get("checkpoint") == checkpoint
                 or (st.get("checkpoint") is None and k.startswith(f"{checkpoint}_"))]
        if not itens:
            return None
        return max((st for _k, st in itens), key=lambda st: st.get("score_mean", -1.0))

    def eval_curve(self):
        """`(passos, scores)` só dos pontos em que a avaliação rodou."""
        pts = self._pontos_de_eval()
        x = np.array([p["global_step"] for p in pts], dtype=np.int64)
        y = np.array([p["eval_score_mean"] for p in pts], dtype=np.float64)
        return x, y

    def _pontos_de_eval(self):
        return [p for p in self.curve if p.get("eval_score_mean") is not None]

    def eval_curve_tempo(self):
        """`(horas, scores)` — a mesma curva no eixo de **custo**, não de dados.

        O eixo oficial da arena são passos de ambiente, que igualam os *dados vistos* e
        escondem o *esforço gasto*: o AlphaZero faz busca em árvore a cada passo e custa
        ordens de grandeza mais que o DQN para chegar ao mesmo x. Este eixo mostra a outra
        metade.

        O `wall_s` inclui as avaliações periódicas, e isso é proposital: elas são custo
        real de quem roda. Mas veja `mesmo_hardware` — comparar tempo entre execuções
        feitas em GPUs diferentes não significa nada.
        """
        pts = self._pontos_de_eval()
        h = np.array([p.get("wall_s", np.nan) for p in pts], dtype=np.float64) / 3600.0
        y = np.array([p["eval_score_mean"] for p in pts], dtype=np.float64)
        return h, y

    def passos_ate(self, limiar):
        """Primeiro passo **medido** em que a avaliação atingiu `limiar`. `None` se nunca.

        Sem interpolação, de propósito: a resolução é a cadência de avaliação
        (`eval_every_steps`), e interpolar inventaria uma precisão que a amostragem não
        tem. O número devolvido é um passo em que a medição de fato aconteceu.
        """
        x, y = self.eval_curve()
        atingiu = np.nonzero(y >= limiar)[0]
        return int(x[atingiu[0]]) if atingiu.size else None

    @property
    def hardware(self):
        """Identidade do que rodou isto, para o eixo de tempo saber quando calar a boca."""
        gpus = self.meta.get("gpus") or []
        return f"{self.meta.get('plataforma', '?')}/{','.join(gpus) or 'cpu'}"


# -------------------------------------------------------------------- gravação
class Recorder:
    """Acumula a curva durante o treino e grava o `history.json` no fim.

    Uso típico, dentro do laço de treino::

        rec = Recorder("ppo", variant="resnet_small", seed=0, net="resnet_small",
                       params=model.count_params(), config=asdict(cfg))
        ...
        rec.log(global_step=n, episodes=e, train_score_mean=m)
        rec.log(global_step=n, eval_score_mean=stats["score_mean"])   # nos passos de eval
        ...
        rec.finish(stats)
        rec.save()            # valida antes de escrever; levanta se violar o contrato
    """

    def __init__(self, algo, variant="default", seed=0, net="", params=0,
                 config=None, env_spec=None, root="runs"):
        self.root = root
        self.t0 = time.perf_counter()
        self.record = RunRecord(
            algo=algo, variant=variant, seed=seed, net=net, params=int(params),
            config=dict(config or {}),
            env_spec=dict(env_spec or CONTRATO),
            meta=_ambiente(),
        )

    def log(self, global_step, **metrics):
        """Anexa um ponto à curva. `global_step` é o eixo oficial."""
        ponto = {"global_step": int(global_step),
                 "wall_s": round(time.perf_counter() - self.t0, 3)}
        for k, v in metrics.items():
            ponto[k] = _jsonable(v)
        self.record.curve.append(ponto)
        return ponto

    def finish(self, final_stats, comparable=True, caveat="", melhor_stats=None):
        self.record.final = {k: _jsonable(v) for k, v in dict(final_stats).items()}
        if melhor_stats is not None:
            self.record.melhor = {k: _jsonable(v) for k, v in dict(melhor_stats).items()}
        self.record.comparable = bool(comparable)
        self.record.caveat = str(caveat)
        self.record.meta["wall_s_total"] = round(time.perf_counter() - self.t0, 3)
        return self.record

    def save(self, path=None, skip_validation=False):
        if not skip_validation:
            problemas = validate(self.record)
            if problemas:
                raise ContractViolation(
                    f"{self.record.run_id} viola o contrato:\n  - "
                    + "\n  - ".join(problemas)
                )
        destino = path or os.path.join(self.root, self.record.algo,
                                       self.record.variant,
                                       f"seed{self.record.seed}", "history.json")
        return save(self.record, destino)


def save(record: RunRecord, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(asdict(record), f, ensure_ascii=False, indent=2)
    return path


def load(path) -> RunRecord:
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    d.pop("schema_version", None)
    # v1 → v2: a coluna com busca morava em `meta["com_busca"]`, gravada com
    # `skip_validation=True`. Migrar na leitura em vez de reescrever os arquivos mantém
    # os registros já publicados byte a byte iguais — a assinatura de um `history.json`
    # é parte do que torna uma execução citável. O lugar antigo continua legível; o novo
    # é o que a arena consulta.
    if not d.get("busca"):
        d["busca"] = dict(d.get("meta", {}).get("com_busca") or {})
    return RunRecord(**d, schema_version=SCHEMA_VERSION)


def configuracoes_incompletas(registros, minimo=SEMENTES_OFICIAIS):
    """As configurações `(algo, variant)` com menos de `minimo` sementes distintas.

    É uma propriedade do **conjunto**, não de uma execução — por isso não cabe em
    `validate`, que olha uma por vez. A arena chama isto para não publicar uma linha de
    uma semente com a mesma tipografia de uma de três.
    """
    grupos = {}
    for r in registros:
        grupos.setdefault((r.algo, r.variant), set()).add(r.seed)
    return [{"algo": a, "variant": v, "sementes": len(s), "faltam": minimo - len(s)}
            for (a, v), s in sorted(grupos.items()) if len(s) < minimo]


def load_all(root="runs"):
    """Carrega todo `history.json` sob `root`, ordenado por algoritmo/variante/seed."""
    achados = []
    for base, _, arquivos in os.walk(root):
        for nome in arquivos:
            if nome == "history.json":
                achados.append(load(os.path.join(base, nome)))
    achados.sort(key=lambda r: (r.algo, r.variant, r.seed))
    return achados


# ------------------------------------------------------------------- validação
def validate(record: RunRecord, strict_eval=True):
    """Devolve a lista de violações do contrato. Lista vazia = pode entrar na arena.

    Curvas marcadas `comparable=False` só precisam ter identidade, curva e um `caveat`
    explicando por que não competem — o resto do contrato não se aplica a elas.
    """
    p = []

    if not record.algo:
        p.append("`algo` vazio")
    if record.schema_version != SCHEMA_VERSION:
        p.append(f"schema_version {record.schema_version} != {SCHEMA_VERSION}")
    if not record.curve:
        p.append("curva vazia")
    else:
        steps = [pt.get("global_step") for pt in record.curve]
        if any(s is None for s in steps):
            p.append("ponto da curva sem `global_step`")
        elif list(steps) != sorted(steps):
            p.append("`global_step` não é monotônico")

    if not record.comparable:
        if not record.caveat:
            p.append("`comparable=False` exige um `caveat` explicando por quê")
        return p

    # --- daqui para baixo, só para execuções que querem competir
    for chave, esperado in CONTRATO.items():
        obtido = record.env_spec.get(chave, "<ausente>")
        if obtido != esperado:
            p.append(f"env_spec['{chave}'] = {obtido!r}, contrato exige {esperado!r}")

    if not record.final:
        p.append("`final` vazio — falta o resultado do protocolo de avaliação")
    elif strict_eval:
        f = record.final
        if f.get("episodes") != CONTRATO["eval_episodes"]:
            p.append(f"avaliação final com {f.get('episodes')} episódios, "
                     f"contrato exige {CONTRATO['eval_episodes']}")
        if not f.get("completo", True):
            p.append("avaliação final incompleta (bateu `max_steps`)")
        # As chaves de causa de fim entraram junto com a correção do protocolo — o mesmo
        # commit que passou a contar a maçã final do episódio vencedor. Um registro sem
        # elas foi medido com a régua **anterior**, e `episodes`/`completo` continuam
        # iguais nos dois casos: é a única marca que distingue. Ver
        # `docs/ANTES_DO_ARTIGO.md`.
        faltando = [k for k in ("fim_fome", "fim_colisao", "fim_tabuleiro_cheio")
                    if k not in f]
        if faltando:
            p.append(f"`final` sem {', '.join(faltando)} — medido com um protocolo "
                     "anterior ao atual; remeça com o `evaluate` desta versão")
        media = f.get("score_mean")
        if media is None:
            p.append("`final.score_mean` ausente")
        elif not (0.0 <= media <= SCORE_PERFEITO):
            p.append(f"score_mean fora da faixa possível: {media}")

    # A coluna com busca é opcional — a maioria dos algoritmos não tem máquina além da
    # rede. Mas uma entrada que **existe** tem de dizer o que é: sem `num_simulations` o
    # número não é interpretável, e sem `checkpoint` não se sabe se mediu o `last` ou o
    # `best`. O tamanho da amostra **não** é violação: uma espiada de 200 episódios é
    # legítima, só não entra na arena (ver `busca_oficial`).
    for chave, st in (record.busca or {}).items():
        if not isinstance(st, dict):
            p.append(f"`busca['{chave}']` não é um dicionário de stats")
            continue
        if st.get("num_simulations") is None:
            p.append(f"`busca['{chave}']` sem `num_simulations` — o número só significa "
                     "alguma coisa junto com o orçamento de busca que o produziu")
        media = st.get("score_mean")
        if media is None:
            p.append(f"`busca['{chave}'].score_mean` ausente")
        elif not (0.0 <= media <= SCORE_PERFEITO):
            p.append(f"`busca['{chave}'].score_mean` fora da faixa possível: {media}")

    orcamento = record.config.get("total_steps")
    if orcamento is None:
        p.append("`config['total_steps']` ausente — o orçamento é parte do contrato")
    elif int(orcamento) != ORCAMENTO_OFICIAL:
        p.append(f"orçamento de {int(orcamento):,} passos; o contrato exige "
                 f"{ORCAMENTO_OFICIAL:,}. Comparar treinos de tamanhos diferentes mede "
                 "paciência, não algoritmo")

    # O `config` diz o orçamento **pretendido**; a curva diz o que foi **gasto**. Conferir
    # só o primeiro deixava passar uma execução interrompida na metade com o `config`
    # intacto — `train(ate_passos=...)` faz exatamente isso, e uma sessão do Colab caindo
    # também. Ver `docs/REVISAO_ALGORITMOS.md` §1.3.
    gasto = max((int(ponto.get("global_step", 0)) for ponto in record.curve), default=0)
    if gasto < ORCAMENTO_OFICIAL:
        p.append(f"a curva vai até {gasto:,} passos, abaixo dos {ORCAMENTO_OFICIAL:,} do "
                 "contrato — o orçamento declarado em `config` não é o que foi gasto")

    if record.params <= 0:
        p.append("`params` deve ser o número de parâmetros treináveis")
    if not record.net:
        p.append("`net` vazio — a arquitetura é um eixo de comparação")

    return p


def assert_valid(record: RunRecord, **kw):
    problemas = validate(record, **kw)
    if problemas:
        raise ContractViolation(
            f"{record.run_id} viola o contrato:\n  - " + "\n  - ".join(problemas)
        )
    return record


# ---------------------------------------------------------------------- legado
def from_legacy_csv(path, algo="dqn-legacy", variant=None, caveat=None):
    """Converte um CSV de treino do `colab-rl` para o esquema do repositório.

    Os CSVs antigos têm colunas sem nome: `índice, comprimento, passos, loss, reward`.
    O comprimento vira score pela regra `score = comprimento - 3`, e o registro nasce
    `comparable=False` — foi medido em outro ambiente, com outra recompensa e outra
    unidade de tempo. Ele é contexto histórico, não competidor.
    """
    import csv

    linhas = []
    with open(path, newline="", encoding="utf-8") as f:
        leitor = csv.reader(f)
        cabecalho = next(leitor, None)
        for row in leitor:
            if len(row) < 5:
                continue
            try:
                ep = int(float(row[0]))
                comprimento = float(row[1])
                passos = float(row[2])
                perda = float(row[3])
                recompensa = float(row[4])
            except ValueError:
                continue
            linhas.append((ep, comprimento, passos, perda, recompensa))

    if not linhas:
        raise ValueError(f"nenhuma linha aproveitável em {path} (cabeçalho: {cabecalho})")

    # Nos CSVs originais o nome do arquivo é sempre `keras_training_data.csv` e quem
    # identifica a variante é a pasta; nos normalizados de `results/legacy/` é o
    # contrário. Aceita os dois.
    if variant is None:
        raiz = os.path.splitext(os.path.basename(path))[0]
        variant = (os.path.basename(os.path.dirname(path))
                   if raiz in ("keras_training_data", "training_data") else raiz)
    curva = [
        {
            "global_step": ep,               # aqui o eixo é episódio, não passo — ver caveat
            "episodes": ep,
            "train_score_mean": comprimento - 3.0,
            "train_length_mean": comprimento,
            "episode_steps": passos,
            "loss": perda,
            "reward": recompensa,
        }
        for ep, comprimento, passos, perda, recompensa in linhas
    ]

    scores = np.array([c["train_score_mean"] for c in curva], dtype=np.float64)
    rec = RunRecord(
        algo=algo,
        variant=variant,
        seed=0,
        net="cnn-legado",
        params=0,
        env_spec={"env": "snake-on-pygame (legado)"},
        curve=curva,
        final={
            "episodes": len(curva),
            "score_mean": float(scores[-100:].mean()),
            "score_max": float(scores.max()),
        },
        comparable=False,
        caveat=(
            caveat
            or "Medido no ambiente antigo (snake-on-pygame): recompensa +comprimento ao "
               "comer, estado ordinal de 1 canal com a cabeça sobrescrita, 5 ações "
               "absolutas, eixo em episódios. Convertido por score = comprimento - 3 "
               "apenas para posicionar a curva; não é comparável com as execuções novas."
        ),
        meta={"fonte": os.path.basename(path)},
    )
    return rec


# ------------------------------------------------------------------ utilidades
def _jsonable(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, np.ndarray):
        return v.tolist()
    if isinstance(v, (np.bool_,)):
        return bool(v)
    return v


def _ambiente():
    """Carimbo de proveniência: sem isto, um número no gráfico não é rastreável."""
    meta = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    }
    try:
        meta["commit"] = subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL,
            text=True, timeout=5,
        ).strip()
    except Exception:
        meta["commit"] = "desconhecido"

    # No Colab e no Kaggle não existe clone git: o `commit` sai "desconhecido" e a curva
    # fica sem procedência — que é justamente onde quase todas as execuções deste
    # repositório nascem. O gerador de notebooks injeta `ASSINATURA_PACOTE` no bloco de
    # código gerado, e ela identifica o pacote inteiro (é o hash do fonte embutido). Como
    # o notebook é um namespace só, esta busca a encontra lá e não a encontra aqui.
    assinatura = globals().get("ASSINATURA_PACOTE")
    if assinatura:
        meta["assinatura_pacote"] = str(assinatura)
    for mod in ("tensorflow", "keras"):
        try:
            meta[mod] = __import__(mod).__version__
        except Exception:
            pass
    return meta


# --- snakeai/env/render.py ---
"""Ver a cobra jogar — GIF de um episódio, sem pygame.

O Colab não tem display, então renderizar pelo jogo original não é opção. Aqui o episódio
vira uma sequência de imagens direto da grade `occ` do `VecSnake`, e o GIF é o artefato
que se olha para entender *como* o agente joga — coisa que nenhuma curva conta.

Vale mais do que parece: um agente com score médio 20 que morre sempre se prendendo no
próprio corpo e outro que morre por fome têm a mesma linha no gráfico e problemas
completamente diferentes.
"""


import numpy as np


__all__ = ["quadros_do_episodio", "render_episode", "PALETA_JOGO"]

#: Fundo, corpo, comida, cabeça — nas cores do gráfico da arena, para o GIF e as figuras
#: parecerem do mesmo projeto.
PALETA_JOGO = np.array(
    [
        [26, 26, 25],      # fundo (a superfície escura do gráfico)
        [27, 175, 122],    # corpo (aqua da paleta)
        [235, 104, 52],    # comida (laranja da paleta)
        [252, 252, 251],   # cabeça (tinta clara)
    ],
    dtype=np.uint8,
)


def _quadro(env: VecSnake, i=0, escala=16):
    grade = np.zeros((env.b, env.b), dtype=np.int32)
    grade[env.occ[i] > 0] = 1
    grade[env.food[i, 0], env.food[i, 1]] = 2
    grade[env.head[i, 0], env.head[i, 1]] = 3
    return PALETA_JOGO[grade].repeat(escala, 0).repeat(escala, 1)


def quadros_do_episodio(politica, board_size=10, safety=False, max_steps=2000,
                        seed=7, escala=16, canal_fome=False):
    """Roda um episódio com `politica` e devolve `(quadros, score, motivo)`.

    `politica` é a mesma interface de `snakeai.eval`: `politica(obs, mask) -> logits`.
    Assim o GIF mostra exatamente a política que o benchmark mediu, sem caminho paralelo.
    """

    # `canal_fome` tem que acompanhar o ambiente de treino: uma rede de 6 canais
    # recebendo observação de 5 quebra aqui, e o GIF é gerado no fim do treino — tarde
    # demais para descobrir. Ver `snakeai.eval.evaluate`.
    env = VecSnake(1, board_size, rng=np.random.default_rng(seed),
                   canal_fome=canal_fome)
    obs, mask = env.reset()
    quadros = [_quadro(env, escala=escala)]
    score, motivo = 0, "limite de passos"
    # Políticas com memória — o `PoliticaRecorrente` do DreamerV3, o `PoliticaComOpcoes`
    # do SOAP — precisam saber qual ação de fato saiu para avançar o estado interno.
    # `snakeai.eval` já respeitava este contrato; aqui não, e o resultado era um GIF
    # gravado com o latente congelado no valor inicial: o agente do vídeo não era o
    # agente da curva, e o vídeo é justamente o artefato que se olha para entender *como*
    # ele joga. Políticas sem memória não expõem o método e nada muda para elas.
    apos_passo = getattr(politica, "apos_passo", None)

    for _ in range(max_steps):
        logits = np.asarray(politica(obs, mask), dtype=np.float32)
        logits = np.where(mask, logits, MASK_NEG)
        if safety:
            logits = apply_safety_filter(env, logits)
        a = logits.argmax(axis=1).astype(np.int32)

        score_antes = int(env.score[0])
        comprimento_antes = int(env.length[0])
        fome_antes = int(env.hunger[0])
        obs, mask, r, d, info = env.step(a)
        if apos_passo is not None:
            apos_passo(a, d)
        quadros.append(_quadro(env, escala=escala))

        if d[0]:
            # `info["scores"]` é o score **final**, já com a maçã do último passo. Ler
            # `env.score` antes do passo perde exatamente um ponto nos episódios que
            # terminam comendo — que são precisamente as vitórias, e o GIF de uma vitória
            # saía rotulado com 96 num tabuleiro cujo perfeito é 97. Mesmo defeito que o
            # `eval.py` corrige e trava com teste.
            finais = info.get("scores")
            score = int(finais[0]) if finais is not None and len(finais) else score_antes
            if comprimento_antes >= board_size * board_size - 1:
                motivo = "tabuleiro cheio"
            elif fome_antes + 1 >= env.starve_base + 2 * comprimento_antes:
                motivo = "fome"
            else:
                motivo = "colisão"
            break
    else:
        score = int(env.score[0])

    return quadros, score, motivo


def render_episode(politica, caminho="episodio.gif", fps=15, **kw):
    """Grava o GIF e devolve `(caminho, score, motivo)`."""
    import imageio.v2 as imageio

    quadros, score, motivo = quadros_do_episodio(politica, **kw)
    imageio.mimsave(caminho, quadros, fps=fps, loop=0)
    return caminho, score, motivo


# --- snakeai/export.py ---
"""Exportar o modelo — `.keras` para retomar treino, TFLite para embarcar.

Uma armadilha silenciosa do Keras 3, registrada aqui para ninguém repetir
--------------------------------------------------------------------------
Converter para TFLite **precisa passar por um SavedModel**.
`TFLiteConverter.from_concrete_functions(...)` compila sem erro, gera um arquivo
minúsculo — e **não captura os pesos**. A inferência devolve NaN, sem nenhum aviso. O
sintoma é um `.tflite` de poucos KB quando deveria ter centenas.

Por isso `export_model` sempre passa por `model.export(dir, format="tf_saved_model")`, e
sempre valida a paridade contra o modelo original antes de declarar sucesso.

A segunda armadilha: "a saída da rede" não tem uma forma só
-----------------------------------------------------------
A paridade compara a **ação escolhida**, e por muito tempo esse cálculo assumiu que a
saída da política é `(lote, ações)`. Neste repositório ela é isso em três dos formatos e
outra coisa nos demais:

======================================  ==============================  ================
construtor                              saída de política                quem usa
======================================  ==============================  ================
``build_actor_critic``                  ``(lote, ações)``                PPO, A2C, ACKTR…
``build_q_network`` (sem C51)           ``(lote, ações)``                DQN
``build_q_network`` (``n_atoms > 0``)   ``(lote, ações, átomos)``        Rainbow, C51
``build_actor_critic_populacao``        ``(lote, políticas, ações)``     LBC
``build_policy_q``                      ``(lote, ações)`` **duas vezes** ACER
======================================  ==============================  ================

O eixo das ações muda de lugar, e no ACER duas saídas diferentes têm exatamente a mesma
forma. Por isso a redução para "um escore por ação" mora em `_escores_por_acao`, e a
escolha de qual tensor do `.tflite` corresponde à política mora em `_indice_da_politica`
— as duas aplicadas **do mesmo jeito nos dois lados** da comparação. Reduzir só o lado
Keras é o defeito que quebrava o Rainbow no fim de um treino inteiro:

    ValueError: operands could not be broadcast together with shapes (200,) (200,121)

É o mesmo erro de `DQN.politica_do_modelo` (§2.17) um passo adiante — lá o C51 quebrava a
avaliação do checkpoint, aqui quebrava a exportação. Ver `docs/REVISAO_ALGORITMOS.md`.
"""


import os
import shutil
import time

import numpy as np


__all__ = ["export_model", "medir_latencia", "conferir_paridade", "canais_do_modelo"]


def canais_do_modelo(modelo, padrao=N_CHANNELS):
    """Quantos canais a rede espera na entrada — **perguntando à rede**, não à constante.

    O contrato são 5 canais, mas uma execução com `canal_fome=True` treina uma rede de 6.
    Exportar essa rede alimentando-a com a constante quebra na primeira inferência, com
    uma mensagem sobre formas — e isso acontece **depois** do treino inteiro, na última
    célula do notebook. Ver `snakeai.eval.evaluate`, que tem o mesmo cuidado.
    """
    try:
        forma = modelo.input_shape
        if isinstance(forma, (list, tuple)) and forma and isinstance(forma[0], (list, tuple)):
            forma = forma[0]                      # modelos de múltiplas entradas
        canais = forma[-1]
        return int(canais) if canais else padrao
    except Exception:                             # rede sem `input_shape` conhecido
        return padrao


def medir_latencia(fn, board_size=10, repeticoes=200, aquecimento=20, canais=N_CHANNELS):
    """Latência de inferência com lote 1 — o que importa se o modelo for para o jogo."""
    x = np.zeros((1, board_size, board_size, canais), dtype=np.float32)
    for _ in range(aquecimento):
        fn(x)
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        fn(x)
    return (time.perf_counter() - t0) / repeticoes * 1000.0


def _q_de_logits_c51(logits):
    """`(…, ações, átomos)` de logits → `(…, ações)`, para **escolher a ação**.

    O `Q` do C51 é `Σ_z p(z)·z` com `z` no suporte `linspace(v_min, v_max, n_atoms)`. O
    exportador não conhece `v_min`/`v_max` — e não precisa: como o suporte é afim e
    crescente no índice do átomo (`z_i = v_min + i·Δz`, com `Δz > 0`), vale

        argmax_a Σ_i p(a,i)·z_i  =  argmax_a Σ_i p(a,i)·i

    ou seja, a **ação escolhida** não depende do suporte, só da esperança do índice. É por
    isso que esta função devolve o índice esperado em vez do `Q` de verdade: o número não
    é o `Q`, mas o `argmax` é o mesmo, e é só o `argmax` que esta comparação usa.

    A média simples dos logits, que estava aqui antes, **não** tem essa propriedade — ela
    ignora a softmax e pode trocar a ação escolhida.
    """
    z = logits - logits.max(axis=-1, keepdims=True)
    p = np.exp(z)
    p /= p.sum(axis=-1, keepdims=True)
    indices = np.arange(p.shape[-1], dtype=p.dtype)
    return (p * indices).sum(axis=-1)


def _escores_por_acao(t, n_actions=N_ACTIONS):
    """Reduz a saída da política a **um escore por ação**, com as ações no último eixo.

    Cobre os três formatos que os construtores de `snakeai.nets.registry` produzem:

    * `(lote, ações)` — devolvido como está;
    * `(lote, políticas, ações)` (LBC) — devolvido como está, e o `argmax` do chamador
      passa a comparar a escolha de **cada** cabeça da população. Conferir todas é mais
      forte que conferir só a `indice_alvo`, que o exportador não conhece;
    * `(lote, ações, átomos)` (C51) — colapsado por `_q_de_logits_c51`.

    A desambiguação é por posição do eixo com `N_ACTIONS`, com o **último** ganhando o
    desempate: `(lote, 3, 3)` é a população de três políticas do LBC, não um C51 de três
    átomos — que seria uma configuração sem sentido (o C51 existe para ter resolução).
    """
    t = np.asarray(t, dtype=np.float32)
    if t.ndim == 2:
        return t
    if t.ndim >= 3 and t.shape[-1] == n_actions:
        return t
    if t.ndim == 3 and t.shape[1] == n_actions:
        return _q_de_logits_c51(t)
    raise ValueError(
        f"saída de forma {t.shape} sem um eixo de {n_actions} ações reconhecível — "
        "se for um formato novo de política, ensine-o a `_escores_por_acao`"
    )


def _indice_da_politica(candidatos, referencia):
    """Qual das saídas do `.tflite` é a política — casando a **forma** com a do Keras.

    A regra antiga era "a que tem `N_ACTIONS` colunas", e ela erra dos dois jeitos:

    * **não acha** a saída certa quando a política é `(lote, ações, átomos)` (C51: a
      última dimensão são os átomos), e caía no `cand[0]` sem avisar;
    * **acha duas** no ACER, cujas saídas `logits` e `Q(s,·)` têm a mesma forma, e no LBC,
      onde `(lote, 3, 3)` e `(lote, 3)` casam as duas. A ordem das saídas do
      `Interpreter` não é a ordem das saídas do `keras.Model` — o SavedModel as nomeia
      `output_0`, `output_1`… e o conversor pode reordená-las. Pegar a primeira é sortear.

    Aqui a forma decide, e quando ela empata o **valor** desempata: a saída correta é a
    que se parece com a do Keras. Isso é exatamente o que a paridade afirma, então usar o
    critério para escolher não enfraquece nada — se nenhuma das candidatas se parecer com
    a referência, todas reprovam igual.
    """
    forma = tuple(referencia.shape[1:])
    iguais = [i for i, c in enumerate(candidatos) if tuple(c.shape[1:]) == forma]
    if not iguais:
        formas = ", ".join(str(tuple(c.shape[1:])) for c in candidatos)
        raise ValueError(
            f"nenhuma saída do .tflite tem a forma da política do Keras {forma} — "
            f"saídas disponíveis: {formas}"
        )
    if len(iguais) == 1:
        return iguais[0]
    return min(iguais, key=lambda i: float(np.abs(candidatos[i] - referencia).max()))


def conferir_paridade(modelo, blob_tflite, board_size=10, n=200, seed=0, canais=None):
    """O `.tflite` escolhe a mesma ação que o `.keras`, em `n` estados aleatórios?

    Não basta comparar os logits: o que importa para o jogo é a **ação escolhida**. Uma
    diferença numérica de quantização é aceitável; uma ação diferente não é.

    Os dois lados passam pela **mesma** redução (`_escores_por_acao`) — reduzir só um deles
    é comparar coisas de formas diferentes, que é como isto quebrava no Rainbow.
    """
    import tensorflow as tf

    rng = np.random.default_rng(seed)
    canais = canais or canais_do_modelo(modelo)
    x = rng.normal(size=(n, board_size, board_size, canais)).astype(np.float32)

    saida = modelo(x, training=False)
    logits_keras = np.asarray(saida[0] if isinstance(saida, (list, tuple)) else saida,
                              dtype=np.float32)

    itp = tf.lite.Interpreter(model_content=blob_tflite)
    itp.allocate_tensors()
    entrada = itp.get_input_details()[0]
    saidas = itp.get_output_details()

    indice, logits_lite = None, []
    for i in range(n):
        itp.set_tensor(entrada["index"], x[i: i + 1])
        itp.invoke()
        cand = [itp.get_tensor(o["index"]) for o in saidas]
        if indice is None:
            indice = _indice_da_politica(cand, logits_keras[i: i + 1])
        logits_lite.append(cand[indice][0])
    logits_lite = np.asarray(logits_lite, dtype=np.float32)

    escolha_keras = _escores_por_acao(logits_keras).argmax(-1)
    escolha_lite = _escores_por_acao(logits_lite).argmax(-1)
    return {
        "acoes_iguais": float((escolha_keras == escolha_lite).mean()),
        "erro_max_logits": float(np.abs(logits_keras - logits_lite).max()),
    }


def export_model(modelo, out_dir="export", board_size=10, formatos=("fp16", "int8"),
                 validar=True):
    """Exporta e **mede**: tamanho, latência e paridade de ação.

    Devolve um dicionário pronto para virar linha do `MODELS.md`.

    Uma falha da **conferência** não derruba a exportação: ela vira
    `{"erro": ...}` no relatório, no lugar de `acoes_iguais`. Os arquivos já estão em
    disco quando ela roda, e esta função é a penúltima célula de um notebook que gastou
    horas de GPU — deixar a validação levar o treino junto foi exatamente o que aconteceu
    com o Rainbow. O relatório é impresso, então a falha continua visível; o que ela não
    faz mais é apagar o resto.
    """
    import tensorflow as tf

    os.makedirs(out_dir, exist_ok=True)
    canais = canais_do_modelo(modelo)
    caminho_keras = os.path.join(out_dir, "modelo.keras")
    modelo.save(caminho_keras)

    resultado = {
        "params": int(modelo.count_params()),
        "keras_kb": round(os.path.getsize(caminho_keras) / 1024, 1),
        "canais": canais,
        "tf_ms": round(medir_latencia(lambda x: modelo(x, training=False), board_size,
                                      canais=canais), 4),
    }

    sm_dir = os.path.join(out_dir, "saved_model")
    if os.path.isdir(sm_dir):
        shutil.rmtree(sm_dir)
    modelo.export(sm_dir, format="tf_saved_model")

    for nome in formatos:
        conv = tf.lite.TFLiteConverter.from_saved_model(sm_dir)
        if nome != "fp32":
            conv.optimizations = [tf.lite.Optimize.DEFAULT]
        if nome == "fp16":
            conv.target_spec.supported_types = [tf.float16]
        blob = conv.convert()

        caminho = os.path.join(out_dir, f"modelo_{nome}.tflite")
        with open(caminho, "wb") as f:
            f.write(blob)
        resultado[f"{nome}_kb"] = round(len(blob) / 1024, 1)

        itp = tf.lite.Interpreter(model_content=blob)
        itp.allocate_tensors()
        ent = itp.get_input_details()[0]
        xi = np.zeros(ent["shape"], dtype=np.float32)

        def roda(_x, _itp=itp, _ent=ent):
            _itp.set_tensor(_ent["index"], xi)
            _itp.invoke()

        resultado[f"{nome}_ms"] = round(medir_latencia(roda, board_size,
                                                       canais=canais), 4)

        if validar:
            try:
                resultado[f"{nome}_paridade"] = conferir_paridade(modelo, blob, board_size,
                                                                  canais=canais)
            except Exception as e:                # noqa: BLE001 — ver docstring
                resultado[f"{nome}_paridade"] = {"erro": f"{type(e).__name__}: {e}"}
            if resultado[f"{nome}_kb"] < resultado["params"] / 4096:
                resultado[f"{nome}_alerta"] = (
                    "arquivo pequeno demais para esse número de parâmetros — "
                    "provável perda de pesos na conversão"
                )

    return resultado


# --- snakeai/plot.py ---
"""O gráfico da arena — onde os algoritmos finalmente ficam lado a lado.

Regras de leitura que este módulo impõe, e o porquê de cada uma:

* **Um eixo só.** Score de avaliação contra passos de ambiente. Nada de segundo eixo y:
  duas escalas empilhadas inventam correlação que não existe nos dados.
* **Cor é identidade, não posição.** Cada algoritmo recebe um slot fixo da paleta, sempre
  o mesmo. Filtrar a arena não repinta os sobreviventes — quem aprendeu que "PPO é azul"
  continua certo no gráfico seguinte.
* **Mediana com faixa interquartil**, nunca uma semente só. Uma curva de RL de execução
  única não é resultado, é anedota.
* **Curvas legadas em painel próprio.** Elas vêm de `comparable=False` e são medidas em
  *episódios*, não em passos de ambiente. Plotá-las no mesmo eixo x seria fabricar um eixo
  comum que não existe — o mesmo pecado do gráfico de dois eixos y, com outra roupa. Elas
  ganham um painel ao lado, com o próprio eixo rotulado, em cinza tracejado.
* **Piso e teto sempre visíveis.** Sem o piso aleatório de 1,21 desenhado, qualquer curva
  parece aprendizado; com ele, dá para ver quem só está tendo sorte.
* **Rótulo direto no fim de cada curva**, além da legenda. Três das cores da paleta clara
  ficam abaixo de 3:1 de contraste com o fundo, e a regra é que nesse caso a identidade
  não pode depender só da cor.

A paleta é a de referência do sistema de dataviz, validada para daltonismo nos dois modos
(pior par adjacente ΔE 9,1 no claro e 8,4 no escuro).
"""


import numpy as np

__all__ = ["PALETA", "VARIANTE_PRINCIPAL", "cores_por_algoritmo", "e_principal",
           "separa_principais", "arena_figure", "arena_familias",
           "arena_tempo", "arena_vitorias", "arena_melhores", "arena_table",
           "plot_run",
           "mesmo_hardware"]

# ---------------------------------------------------------------------- paleta
PALETA = {
    "light": {
        "surface": "#fcfcfb",
        "plane": "#f9f9f7",
        "ink": "#0b0b0b",
        "ink2": "#52514e",
        "muted": "#898781",
        "grid": "#e1e0d9",
        "axis": "#c3c2b7",
        "series": ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4",
                   "#008300", "#4a3aa7", "#e34948"],
        "legado": "#898781",
    },
    "dark": {
        "surface": "#1a1a19",
        "plane": "#0d0d0d",
        "ink": "#ffffff",
        "ink2": "#c3c2b7",
        "muted": "#898781",
        "grid": "#2c2c2a",
        "axis": "#383835",
        "series": ["#3987e5", "#d95926", "#199e70", "#c98500", "#d55181",
                   "#008300", "#9085e9", "#e66767"],
        "legado": "#898781",
    },
}

#: Ordem fixa dos slots. Um algoritmo novo entra no fim; ninguém troca de cor por isso.
ORDEM_ALGORITMOS = ["ppo", "dqn", "rainbow", "a2c", "acer", "alphazero",
                    "muzero", "acktr", "acektr", "dreamerv3", "lbc", "soap",
                    "dqn-legacy"]

#: A variante que **o notebook principal de cada algoritmo** produz na configuração padrão.
#: É a definição de "resultado principal" deste repositório: um braço por algoritmo, o que
#: `NN_algo.ipynb` roda sem tocar em nada.
#:
#: Existe porque o gráfico e a tabela respondem perguntas diferentes. A tabela é o registro
#: e leva tudo; o gráfico é a **comparação entre algoritmos**, e uma ablação ali dentro
#: compete visualmente com o braço que ela deveria explicar — o `ppo · esparso` desenha uma
#: segunda curva azul 17 pontos abaixo do PPO, e quem olha de longe lê "o PPO é instável"
#: em vez de "este é o controle de orçamento de gradiente". Ablação tem par próprio, e o
#: par está na tabela.
#:
#: Os nomes seguem o padrão dos agentes: a base é `cfg.net` na maioria, `base` no DQN,
#: `completo` no Rainbow, o preset no DreamerV3 e o parâmetro de busca no AlphaZero e no
#: MuZero. Qualquer desvio vira `+marca` ou `_sufixo` e, por construção, **deixa** de casar
#: com este mapa. Uma execução que não casa não some em silêncio: `arena --all` lista as que
#: ficaram fora do gráfico, pelo mesmo motivo do `COMPARABILITY.md`.
VARIANTE_PRINCIPAL = {
    "ppo": "resnet_small",
    "dqn": "base",
    "rainbow": "completo",
    "a2c": "resnet_small",
    "acer": "resnet_small",
    "alphazero": "sims32",
    "muzero": "unroll5",
    "acktr": "resnet_small",
    "acektr": "resnet_small",
    "dreamerv3": "dreamer_small",
    "lbc": "resnet_small",
    "soap": "resnet_small",
}


def e_principal(r):
    """A execução é o braço principal do algoritmo dela?

    `"default"` e `""` contam: é o que o `AgentBase` dá a quem não declara variante
    nenhuma, e não declarar é a definição de padrão. O mapa cobre os agentes que
    **derivam** o nome da configuração — todos os doze hoje.
    """
    return r.variant in ("default", "") or VARIANTE_PRINCIPAL.get(r.algo) == r.variant


def separa_principais(registros):
    """`(principais, ablacoes)` — a divisão que o gráfico usa e a tabela não."""
    principais = [r for r in registros if e_principal(r)]
    ablacoes = [r for r in registros if not e_principal(r)]
    return principais, ablacoes


#: Famílias, na ordem em que os painéis aparecem. Existem porque a arena passou de oito
#: algoritmos e **oito é o limite honesto de uma paleta categórica**: a nona cor seria
#: indistinguível de alguma das oito sob daltonismo. A saída não é gerar mais uma cor, é
#: mudar a forma do gráfico — *small multiples*, um painel por família.
#:
#: O agrupamento é o de sempre em RL, e não uma conveniência visual: o que o algoritmo
#: aprende (política, valor, ou um modelo do mundo) é a divisão que explica por que as
#: curvas têm formatos diferentes.
FAMILIAS = [
    ("política", "gradiente de política",
     ["ppo", "a2c", "acktr", "acektr", "acer", "lbc", "soap"]),
    ("valor", "função de valor", ["dqn", "rainbow"]),
    ("modelo", "modelo do mundo e busca", ["alphazero", "muzero", "dreamerv3"]),
]


def familia_de(algo):
    for chave, _, membros in FAMILIAS:
        if algo in membros:
            return chave
    return "outros"

PISO_ALEATORIO = 1.21
SCORE_PERFEITO = 97

#: Limiar padrão da coluna "passos até". 40 é bem acima do piso (1,21) e bem abaixo do teto
#: (97): alto o bastante para exigir que o agente jogue de verdade, baixo o bastante para
#: a maioria alcançar dentro do orçamento — um limiar que quase ninguém atinge não ordena
#: nada.
LIMIAR_PADRAO = 40.0


def cores_por_algoritmo(algoritmos, mode="light"):
    """Mapeia algoritmo -> cor, em ordem fixa. Nunca cicla nem gera hue nova.

    Passar do oitavo algoritmo é um erro deliberado: a nona cor seria
    indistinguível de alguma das oito sob daltonismo. Nesse ponto o gráfico
    precisa virar *small multiples*, não ganhar mais uma cor.
    """
    p = PALETA[mode]["series"]
    conhecidos = [a for a in ORDEM_ALGORITMOS if a in algoritmos]
    novos = sorted(a for a in algoritmos if a not in ORDEM_ALGORITMOS)
    ordenados = conhecidos + novos
    if len(ordenados) > len(p):
        raise ValueError(
            f"{len(ordenados)} algoritmos para {len(p)} slots de cor. "
            "Use `arena_figure(..., familias=True)` — small multiples por família — "
            "ou agrupe a cauda em 'outros'. Não gere cor nova."
        )
    return {a: p[i] for i, a in enumerate(ordenados)}


def cores_por_familia(mode="light"):
    """Cor de cada algoritmo **dentro do painel da sua família**.

    Nos *small multiples*, cada painel é uma unidade de leitura com no máximo quatro
    séries coloridas; as outras famílias aparecem em cinza, só para dar contexto. Duas
    famílias podem repetir um matiz — o que é seguro porque elas nunca aparecem coloridas
    no mesmo painel, e cada curva colorida ganha rótulo direto.

    A cor é presa ao algoritmo pela posição dele dentro da família, que é fixa. Filtrar
    execuções não repinta ninguém.
    """
    p = PALETA[mode]["series"]
    return {a: p[i] for _, _, membros in FAMILIAS for i, a in enumerate(membros)}


# ------------------------------------------------------------------ agregação
def agrega_sementes(registros, pontos=60):
    """Junta as sementes de uma mesma `(algo, variante)` numa mediana com faixa IQR.

    As sementes raramente avaliam nos mesmos passos, então interpolamos todas numa
    grade log-espaçada comum antes de tirar os quantis. A grade para no menor
    `max(step)` entre as sementes — extrapolar seria inventar dado.
    """
    curvas = []
    for r in registros:
        x, y = r.eval_curve()
        if x.size >= 2:
            curvas.append((x, y))
    if not curvas:
        return None

    x_min = max(1, max(c[0][0] for c in curvas))
    x_max = min(c[0][-1] for c in curvas)
    if x_max <= x_min:
        return None

    grade = np.unique(np.geomspace(x_min, x_max, pontos).astype(np.int64))
    empilhado = np.stack([np.interp(grade, x, y) for x, y in curvas])
    return {
        "x": grade,
        "mediana": np.median(empilhado, axis=0),
        "q1": np.percentile(empilhado, 25, axis=0),
        "q3": np.percentile(empilhado, 75, axis=0),
        "n_sementes": len(curvas),
    }


def mesmo_hardware(registros):
    """Todas as execuções vieram da mesma máquina? Devolve `(bool, conjunto)`.

    O eixo de tempo só significa alguma coisa dentro de um mesmo hardware. Uma curva feita
    numa P100 do Kaggle e outra numa T4 do Colab colocadas lado a lado em horas comparam os
    aceleradores, não os algoritmos — e o gráfico não avisaria.
    """
    hw = {r.hardware for r in registros}
    return len(hw) <= 1, hw


def agrega_tempo(registros, pontos=60):
    """Como `agrega_sementes`, no eixo de horas de GPU."""
    curvas = []
    for r in registros:
        h, y = r.eval_curve_tempo()
        ok = np.isfinite(h) & (h > 0)
        if ok.sum() >= 2:
            curvas.append((h[ok], y[ok]))
    if not curvas:
        return None

    x_min = max(1e-4, max(c[0][0] for c in curvas))
    x_max = min(c[0][-1] for c in curvas)
    if x_max <= x_min:
        return None
    grade = np.geomspace(x_min, x_max, pontos)
    emp = np.stack([np.interp(grade, h, y) for h, y in curvas])
    return {"x": grade, "mediana": np.median(emp, axis=0),
            "q1": np.percentile(emp, 25, axis=0), "q3": np.percentile(emp, 75, axis=0),
            "n_sementes": len(curvas)}


def _agrupa(registros):
    grupos = {}
    for r in registros:
        grupos.setdefault((r.algo, r.variant), []).append(r)
    return grupos


# -------------------------------------------------------------------- figuras
def arena_familias(registros, mode="light", figsize=(14.5, 4.8), titulo=None,
                   x_log=True, mostrar_legado=True, so_principais=True):
    """*Small multiples*: um painel por família, com as demais em cinza ao fundo.

    Esta é a forma que a arena assume quando passa de oito algoritmos. Ela não é um
    consolo por não caber tudo num painel — é melhor para a pergunta que a arena de fato
    responde. Sobrepor nove curvas com faixa interquartil produz um emaranhado onde a
    comparação relevante ("o Rainbow supera o DQN?") fica *mais* difícil, não menos.

    Cada painel mostra a família em cor e **todas as outras curvas em cinza claro**, na
    mesma escala. Sem esse fundo, três painéis lado a lado seriam três gráficos
    independentes e a comparação entre famílias se perderia — que é justamente o que a
    arena existe para permitir.
    """
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter

    p = PALETA[mode]
    cores = cores_por_familia(mode)
    comparaveis = [r for r in registros if r.oficial]
    if so_principais:
        comparaveis, _ = separa_principais(comparaveis)

    agregados = {}
    for (algo, variante), rs in sorted(_agrupa(comparaveis).items()):
        ag = agrega_sementes(rs)
        if ag is not None:
            agregados[(algo, variante)] = ag

    presentes = [f for f in FAMILIAS
                 if any(a in f[2] for a, _ in agregados)] or FAMILIAS
    legado = [r for r in registros if not r.comparable] if mostrar_legado else []

    # O painel legado entra com largura menor e **eixo x próprio**: ele mede episódios, e
    # pendurá-lo no eixo de passos seria fabricar um eixo comum que não existe.
    larguras = [1.0] * len(presentes) + ([0.62] if legado else [])
    fig = plt.figure(figsize=figsize, facecolor=p["plane"])
    gs = fig.add_gridspec(1, len(larguras), width_ratios=larguras, wspace=.08)
    axes = [fig.add_subplot(gs[0])]
    axes += [fig.add_subplot(gs[i], sharex=axes[0], sharey=axes[0])
             for i in range(1, len(presentes))]
    ax_leg = fig.add_subplot(gs[-1], sharey=axes[0]) if legado else None

    topo = max((max(ag["mediana"]) for ag in agregados.values()), default=0.0)
    topo = max(topo * 1.3, PISO_ALEATORIO * 4)

    for i, (ax, (chave, rotulo, membros)) in enumerate(zip(axes, presentes)):
        ax.set_facecolor(p["surface"])
        ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)
        if i == len(presentes) - 1:
            # rotulada uma vez, e no painel mais vazio: repetir a referência nos três
            # seria ruído, e à esquerda ela cai em cima do início das curvas
            ax.annotate(f"piso aleatório · {PISO_ALEATORIO:.2f}".replace(".", ","),
                        xy=(0.98, PISO_ALEATORIO), xycoords=("axes fraction", "data"),
                        xytext=(0, 5), textcoords="offset points",
                        color=p["muted"], fontsize=8, va="bottom", ha="right")

        # contexto: todo o resto, em cinza, atrás
        for (algo, _), ag in agregados.items():
            if algo not in membros:
                ax.plot(ag["x"], ag["mediana"], color=p["legado"], lw=1.2,
                        alpha=.45, zorder=2, solid_capstyle="round")

        rotulos = []
        for (algo, variante), ag in agregados.items():
            if algo not in membros:
                continue
            cor = cores[algo]
            nome = algo if variante in ("default", "") else f"{algo} · {variante}"
            ax.fill_between(ag["x"], ag["q1"], ag["q3"], color=cor, alpha=.16,
                            linewidth=0, zorder=3)
            ax.plot(ag["x"], ag["mediana"], color=cor, lw=2.0, zorder=4,
                    label=f"{nome}  (n={ag['n_sementes']})", solid_capstyle="round")
            rotulos.append((ag["x"][-1], ag["mediana"][-1], nome, cor))

        for x, y, nome, _ in _sem_colisao(rotulos):
            ax.annotate(nome, xy=(x, y), xytext=(5, 0), textcoords="offset points",
                        color=p["ink2"], fontsize=8.5, va="center", ha="left", zorder=5)

        ax.set_title(rotulo, color=p["ink2"], fontsize=10.5, loc="left", pad=10)
        if x_log:
            ax.set_xscale("log")
        ax.set_ylim(0, topo)
        if not agregados:
            ax.set_xlim(1e4, 1e7)
        else:
            # espaço à direita para o rótulo direto de cada curva não sair do painel
            ax.margins(x=.30)
        ax.grid(True, which="major", color=p["grid"], lw=0.8, zorder=0)
        ax.set_axisbelow(True)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)
        for lado in ("left", "bottom"):
            ax.spines[lado].set_color(p["axis"])
        ax.tick_params(colors=p["muted"], labelsize=9, length=0)
        ax.xaxis.set_major_formatter(FuncFormatter(_formata_passos))
        ax.set_xlabel("passos de ambiente", color=p["ink2"], fontsize=9.5)
        if i:
            ax.tick_params(labelleft=False)
        if rotulos:
            leg = ax.legend(loc="upper left", frameon=False, fontsize=8.5,
                            labelcolor=p["ink2"], handlelength=1.6)
            for t in leg.get_texts():
                t.set_color(p["ink2"])

    if ax_leg is not None:
        _painel_legado(ax_leg, legado, p, ylim=(0, topo))
        ax_leg.tick_params(labelleft=False)

    axes[0].set_ylabel("score na avaliação (1.000 episódios, greedy)",
                       color=p["ink2"], fontsize=10)
    fig.suptitle(titulo or "snake-arena · por família de algoritmo",
                 color=p["ink"], fontsize=13, x=.006, ha="left", y=.985)
    fig.text(.006, .015,
             "cada painel colore uma família e mantém as demais curvas em cinza, na mesma "
             "escala; nove algoritmos não cabem numa paleta categórica. O painel do legado "
             "tem eixo x próprio, em episódios — ver docs/COMPARABILITY.md.",
             color=p["muted"], fontsize=8)
    fig.subplots_adjust(left=.058, right=.985, top=.83, bottom=.16)
    return fig, tuple(axes) + ((ax_leg,) if ax_leg is not None else ())


def arena_figure(registros, mode="light", figsize=(12.5, 6.2), titulo=None,
                 mostrar_legado=True, x_log=True, familias="auto", so_principais=True):
    """A figura principal do benchmark. Devolve `(fig, (ax, ax_legado))`.

    `familias="auto"` (o padrão) troca para *small multiples* assim que o número de
    algoritmos passa dos oito slots de cor. É automático de propósito: a alternativa
    seria a arena quebrar — ou, pior, ganhar uma nona cor — no dia em que o nono
    algoritmo termina de treinar.

    `registros` é uma lista de `snakeai.record.RunRecord` — tipicamente
    `record.load_all("runs")` mais as curvas legadas convertidas.

    `so_principais=True` (o padrão) desenha **um braço por algoritmo** — o que o notebook
    principal produz na configuração padrão, conforme `VARIANTE_PRINCIPAL`. As ablações
    saem do gráfico e continuam na tabela: aqui a pergunta é *quem vai mais longe com os
    mesmos dados*, e uma ablação desenhada ao lado do seu controle, na mesma cor, responde
    outra. Elas não somem em silêncio — a contagem vai no rodapé da figura e a lista, no
    `arena --all`.

    O painel grande tem só as execuções `comparable=True`, no eixo oficial de passos de
    ambiente. As legadas, quando existem, vão para um painel estreito à direita com o
    **próprio eixo em episódios** — porque é isso que elas medem, e fingir o contrário
    seria exatamente o erro que este repositório foi criado para consertar.
    """
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter

    p = PALETA[mode]
    comparaveis = [r for r in registros if r.oficial]
    ablacoes = []
    if so_principais:
        comparaveis, ablacoes = separa_principais(comparaveis)
    legado = [r for r in registros if not r.comparable] if mostrar_legado else []

    algos = {r.algo for r in comparaveis}
    if familias is True or (familias == "auto" and len(algos) > len(p["series"])):
        return arena_familias(registros, mode=mode, titulo=titulo, x_log=x_log,
                              mostrar_legado=mostrar_legado, so_principais=so_principais)

    cores = cores_por_algoritmo(algos, mode)

    fig = plt.figure(figsize=figsize, facecolor=p["plane"])
    if legado:
        gs = fig.add_gridspec(1, 2, width_ratios=(3.4, 1), wspace=.22)
        ax = fig.add_subplot(gs[0])
        ax_leg = fig.add_subplot(gs[1])
    else:
        ax = fig.add_subplot(1, 1, 1)
        ax_leg = None
    ax.set_facecolor(p["surface"])

    # --- referências primeiro, para ficarem atrás dos dados
    ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)
    ax.annotate(f"piso aleatório com máscara · {PISO_ALEATORIO:.2f}".replace(".", ","),
                xy=(0.995, PISO_ALEATORIO), xycoords=("axes fraction", "data"),
                xytext=(0, 5), textcoords="offset points",
                color=p["muted"], fontsize=8.5, va="bottom", ha="right")

    # --- as curvas que competem
    rotulos = []
    for (algo, variante), rs in sorted(_agrupa(comparaveis).items()):
        ag = agrega_sementes(rs)
        if ag is None:
            continue
        cor = cores[algo]
        nome = algo if variante in ("default", "") else f"{algo} · {variante}"
        ax.fill_between(ag["x"], ag["q1"], ag["q3"], color=cor, alpha=.16,
                        linewidth=0, zorder=3)
        ax.plot(ag["x"], ag["mediana"], color=cor, lw=2.0, zorder=4,
                label=f"{nome}  (n={ag['n_sementes']})", solid_capstyle="round")
        rotulos.append((ag["x"][-1], ag["mediana"][-1], nome, cor))

    # o teto do eixo y vem de TODOS os dados, inclusive os legados: os dois painéis
    # compartilham a escala de score, e calcular só a partir das curvas oficiais faz o
    # painel da direita ser cortado quando a arena ainda está vazia. Ele é calculado
    # **antes** dos rótulos porque é a escala deles: ver `_sem_colisao`.
    topo_oficial = max((y for _, y, _, _ in rotulos), default=0.0)
    topo_legado = max(
        (max(c["train_score_mean"] for c in r.curve) for r in legado), default=0.0
    ) if legado else 0.0
    topo = max(topo_oficial * 1.3, topo_legado * 1.15, PISO_ALEATORIO * 4)

    # --- rótulo direto no fim de cada curva (a "relief rule" do contraste)
    for x, y, nome, cor in _sem_colisao(rotulos, minimo=0.033, escala=topo):
        ax.annotate(nome, xy=(x, y), xytext=(6, 0), textcoords="offset points",
                    color=p["ink2"], fontsize=9, va="center", ha="left", zorder=5)

    # --- eixos e cromo
    if x_log:
        ax.set_xscale("log")
    ax.set_xlabel("passos de ambiente", color=p["ink2"], fontsize=10)
    ax.set_ylabel("score na avaliação (1.000 episódios, greedy)",
                  color=p["ink2"], fontsize=10)
    ax.set_title(titulo or "snake-arena · mesmo ambiente, mesmo orçamento, mesma régua",
                 color=p["ink"], fontsize=13, pad=14, loc="left")

    ax.grid(True, which="major", color=p["grid"], lw=0.8, ls="-", zorder=0)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(p["axis"])
        ax.spines[lado].set_linewidth(1.0)
    ax.tick_params(colors=p["muted"], labelsize=9, length=0)
    ax.xaxis.set_major_formatter(FuncFormatter(_formata_passos))

    ax.set_ylim(0, topo)
    if rotulos:
        ax.margins(x=.18)
    else:
        # arena vazia: um eixo x de 1 a 10 e um retângulo em branco não comunicam nada
        ax.set_xlim(1e4, 1e7)
        ax.annotate(
            "nenhuma execução oficial ainda\n\n"
            "as curvas entram aqui quando forem treinadas no orçamento do contrato",
            xy=(0.5, 0.55), xycoords="axes fraction", ha="center", va="center",
            color=p["muted"], fontsize=11, linespacing=1.6)

    if len(rotulos) >= 2:
        leg = ax.legend(loc="upper left", frameon=False, fontsize=9,
                        labelcolor=p["ink2"], handlelength=1.6)
        for t in leg.get_texts():
            t.set_color(p["ink2"])

    # --- painel legado: eixo próprio, unidade própria, mesma escala de score
    # o rodapé é uma linha por assunto: emendar os dois numa só passa da largura da
    # figura e o texto sai cortado na borda, que foi como ele nasceu
    linhas_rodape = [_nota_de_ablacoes(ablacoes)]
    if ax_leg is not None:
        _painel_legado(ax_leg, legado, p, ylim=ax.get_ylim())
        linhas_rodape.append(
            "Os dois painéis não compartilham eixo x — e não podem. À esquerda, "
            "passos de ambiente no jogo novo; à direita, episódios no jogo de 2019, "
            "com outra recompensa e score de treino em vez de avaliação.")
        fig.subplots_adjust(left=.075, right=.985, top=.88, bottom=.155)
    else:
        fig.tight_layout()
        if any(linhas_rodape):
            fig.subplots_adjust(bottom=.16)
    for i, linha in enumerate([t for t in linhas_rodape if t]):
        fig.text(0.012, 0.030 - i * 0.020, linha, color=p["muted"], fontsize=8)
    return fig, (ax, ax_leg)


def _nota_de_ablacoes(ablacoes):
    """O que ficou fora do gráfico, dito no próprio gráfico.

    Uma figura que esconde execuções sem avisar afirma que aquilo é tudo o que existe — o
    mesmo defeito que o `COMPARABILITY.md` chama de pior que incluir. O rodapé dá a
    contagem; a tabela dá os nomes, os números e o par de comparação de cada uma.
    """
    quantas = len({(r.algo, r.variant) for r in ablacoes})
    if not quantas:
        return ""
    if quantas == 1:
        fora = "1 configuração de ablação fica de fora dele e está na tabela"
    else:
        fora = (f"{quantas} configurações de ablação ficam de fora dele e estão na tabela")
    return f"O gráfico mostra o braço principal de cada algoritmo: {fora}, com o controle de cada uma."


def _painel_legado(ax, legado, p, ylim=None):
    """As curvas históricas, no eixo delas: episódios de treino.

    Compartilham a escala y com o painel principal — score é score, essa parte é
    conversível. O eixo x é que não é, e por isso está separado.
    """
    ax.set_facecolor(p["surface"])
    ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)

    melhor = (None, -1.0)
    for r in legado:
        x = np.array([c["episodes"] for c in r.curve], dtype=np.float64)
        y = np.array([c["train_score_mean"] for c in r.curve], dtype=np.float64)
        if y.size > 400:
            # 10 mil episódios num painel estreito viram um borrão cinza; a janela
            # larga mostra a tendência, que é o que o painel de contexto precisa dizer.
            k = max(1, y.size // 40)
            nucleo = np.ones(k) / k
            y = np.convolve(y, nucleo, mode="valid")
            x = x[k - 1:]
        ax.plot(x, y, color=p["legado"], lw=1.2, ls=(0, (4, 3)), alpha=.6, zorder=2)
        if y.max() > melhor[1]:
            melhor = (r.variant, float(y.max()), float(x[int(y.argmax())]))

    if melhor[0]:
        # rótulo ancorado no canto, não no ponto: no painel estreito um rótulo junto
        # ao máximo sai pela borda direita
        ax.annotate(f"melhor: {melhor[0]}\nmédia móvel {melhor[1]:.1f}".replace(".", ","),
                    xy=(0.04, 0.97), xycoords="axes fraction",
                    color=p["ink2"], fontsize=8.5, ha="left", va="top",
                    linespacing=1.5)

    ax.set_title("legado · 2019", color=p["ink2"], fontsize=10, loc="left", pad=14)
    ax.set_xlabel("episódios de treino", color=p["muted"], fontsize=9)
    if ylim:
        ax.set_ylim(ylim)
    ax.grid(True, color=p["grid"], lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(p["axis"])
    ax.tick_params(colors=p["muted"], labelsize=8.5, length=0)
    ax.xaxis.set_major_formatter(__import__("matplotlib").ticker.FuncFormatter(_formata_passos))


def _ate_o_limiar(registros, limiar):
    """Mediana dos passos até `limiar`, e quantas sementes chegaram lá.

    Quem não chegou **não** entra na mediana como um número grande inventado: fica de fora
    e o `n` denuncia. Uma mediana calculada sobre metade das sementes que chegaram, sem
    dizer que foi metade, seria a pior das duas opções.
    """
    passos = [r.passos_ate(limiar) for r in registros]
    chegaram = [p for p in passos if p is not None]
    return {"passos_ate": int(np.median(chegaram)) if chegaram else None,
            "sementes_ate": len(chegaram), "limiar": limiar}


def _sem_colisao(rotulos, minimo=0.045, escala=None):
    """Empurra rótulos que ficariam sobrepostos, preservando a ordem vertical.

    `escala` é o **teto do eixo**, e passá-la é o que faz a separação valer em pixels em
    vez de em pontos de score. Sem ela a referência é a faixa dos próprios rótulos — que
    encolhe justamente quando as curvas convergem, ou seja, o mínimo fica menor no único
    caso em que ele importa. Com os seis braços principais terminando entre 47 e 82, PPO,
    ACKTR e ACER escreviam um por cima do outro.
    """
    if not rotulos:
        return []
    ordenado = sorted(rotulos, key=lambda t: t[1])
    ys = [t[1] for t in ordenado]
    faixa = max(ys[-1] - ys[0], 1e-9)
    minimo = minimo * (escala if escala else faixa)
    for i in range(1, len(ys)):
        if ys[i] - ys[i - 1] < minimo:
            ys[i] = ys[i - 1] + minimo
    return [(x, ys[i], nome, cor) for i, (x, _, nome, cor) in enumerate(ordenado)]


def _formata_passos(v, _pos=None):
    if v >= 1e6:
        return f"{v / 1e6:g} M"
    if v >= 1e3:
        return f"{v / 1e3:g} mil"
    return f"{v:g}"


def plot_run(record, mode="light", figsize=(11, 3.4)):
    """Diagnóstico de uma execução: treino (com exploração) contra avaliação (honesta).

    As duas subindo juntas = aprendeu. A de treino subindo sozinha = está explorando com
    sorte, e o número honesto não acompanha.
    """
    import matplotlib.pyplot as plt

    p = PALETA[mode]
    fig, ax = plt.subplots(figsize=figsize, facecolor=p["plane"])
    ax.set_facecolor(p["surface"])

    treino = [(c["global_step"], c["train_score_mean"]) for c in record.curve
              if c.get("train_score_mean") is not None]
    if treino:
        x, y = zip(*treino)
        ax.plot(x, y, color=p["muted"], lw=1.4, label="treino (com exploração)")

    x, y = record.eval_curve()
    if x.size:
        ax.plot(x, y, color=p["series"][0], lw=2.0, label="avaliação (greedy)")

    ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0)
    ax.set_xlabel("passos de ambiente", color=p["ink2"], fontsize=10)
    ax.set_ylabel("score", color=p["ink2"], fontsize=10)
    ax.set_title(record.run_id, color=p["ink"], fontsize=12, loc="left", pad=10)
    ax.grid(True, color=p["grid"], lw=0.8)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(p["axis"])
    ax.tick_params(colors=p["muted"], labelsize=9, length=0)
    leg = ax.legend(frameon=False, fontsize=9)
    for t in leg.get_texts():
        t.set_color(p["ink2"])
    fig.tight_layout()
    return fig, ax


# --------------------------------------------------------------------- tabela
def arena_vitorias(registros, mode="light", figsize=(8.6, 4.4), titulo=None,
                   so_principais=True):
    """Como cada episódio **termina** — e a taxa de vitória dentro disso.

    Por que este painel existe
    --------------------------
    "Melhor" não é uma coisa só. A média e a taxa de vitória são dois funcionais da mesma
    distribuição — `E[X]` e `P(X = perfeito)` — e **elas discordam nestes dados**: o
    Rainbow é o penúltimo em média e ainda assim fecha o tabuleiro **nove vezes mais** que
    o A2C, que tem 15 pontos a mais de score (19,9% contra 2,2%). Publicar só a média
    deixaria isso invisível.

    O que cada uma joga fora explica a discordância. A taxa de vitória é um limiar no
    extremo: um episódio de 96 conta igual a um de 3, e é por isso que o A2C — que joga
    bem e morre na casa dos 90 — tem 69,61 de média e 2,2% de vitória. A média usa o
    episódio inteiro, mas não distingue "sempre 78" de "metade perfeito, metade zero".

    Por isso a barra não é a taxa de vitória sozinha: é a **repartição inteira** das causas
    de fim, com a vitória como primeiro segmento. É a mesma leitura que o log de treino já
    faz por iteração — score sozinho é ambíguo, 1,2 ponto pode ser "bate em tudo" ou "anda
    em círculo até morrer de fome" — trazida para o fim da execução.

    Ele **não substitui** o painel oficial, pela mesma razão que o `arena_tempo` não
    substitui: é outra pergunta. Esta aqui é *qual eu levaria para o jogo*; a oficial é
    *quem aprende mais com os mesmos dados*.
    """
    import matplotlib.pyplot as plt

    p = PALETA[mode]
    comparaveis = [r for r in registros if r.oficial]
    if so_principais:
        comparaveis, _ = separa_principais(comparaveis)

    linhas = []
    for (algo, variante), rs in _agrupa(comparaveis).items():
        partes = [_mediana_do_final(rs, k) for k in
                  ("fim_tabuleiro_cheio", "fim_colisao", "fim_fome")]
        if any(v is None for v in partes):
            continue                      # protocolo antigo, sem as chaves de causa
        # as três medianas são tomadas por semente e independentes, então a soma foge de
        # 1 por frações de ponto. Normalizar é honesto porque o que a barra afirma é a
        # **proporção** entre as causas; o número impresso é a mediana, não o normalizado.
        total = sum(partes) or 1.0
        linhas.append({
            "nome": algo if variante in ("default", "") else f"{algo} · {variante}",
            "vitoria": partes[0],
            "fatias": [v / total for v in partes],
            "media": _mediana_do_final(rs, "score_mean") or 0.0,
            "n": len(rs),
        })

    linhas.sort(key=lambda d: d["vitoria"])
    fig, ax = plt.subplots(figsize=figsize, facecolor=p["plane"])
    ax.set_facecolor(p["surface"])

    cores = (p["series"][2], p["series"][1], p["muted"])
    nomes = ("tabuleiro cheio", "colisão", "fome")
    y = list(range(len(linhas)))
    for i, linha in enumerate(linhas):
        esquerda = 0.0
        for fatia, cor, nome in zip(linha["fatias"], cores, nomes):
            ax.barh(i, fatia * 100, left=esquerda * 100, color=cor, height=.62,
                    zorder=3, label=nome if i == 0 else None,
                    edgecolor=p["surface"], linewidth=1.2)
            esquerda += fatia
        # o número fica **fora** da barra: dentro ele some quando a fatia é 2%, que é
        # justamente o caso que esta figura existe para mostrar
        ax.annotate(f"{linha['vitoria'] * 100:.1f}%".replace(".", ","),
                    xy=(101, i), xytext=(0, 0), textcoords="offset points",
                    color=p["ink"], fontsize=9.5, va="center", ha="left", zorder=5)
        ax.annotate(f"média {linha['media']:.2f}".replace(".", ","),
                    xy=(119, i), color=p["muted"], fontsize=9, va="center", ha="left",
                    zorder=5)

    ax.set_yticks(y, [linha["nome"] for linha in linhas],
                  color=p["ink2"], fontsize=9.5)
    ax.set_xlim(0, 145)
    ax.set_xticks([0, 25, 50, 75, 100], ["0", "25%", "50%", "75%", "100%"])
    ax.set_xlabel("como os 1.000 episódios de avaliação terminam",
                  color=p["ink2"], fontsize=10, loc="left")
    ax.set_title(titulo or "snake-arena · quem fecha o tabuleiro",
                 color=p["ink"], fontsize=13, pad=14, loc="left")
    ax.grid(True, axis="x", color=p["grid"], lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for lado in ("top", "right", "left"):
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color(p["axis"])
    # a linha do eixo para nos 100%: ela não tem o que dizer embaixo dos rótulos
    ax.spines["bottom"].set_bounds(0, 100)
    ax.tick_params(colors=p["muted"], labelsize=9, length=0)

    if linhas:
        leg = ax.legend(loc="lower right", bbox_to_anchor=(1.0, 1.0), ncols=3,
                        frameon=False, fontsize=9, handlelength=1.2)
        for t in leg.get_texts():
            t.set_color(p["ink2"])

    for i, linha in enumerate((
            "A ordem aqui não é a ordem do painel oficial — e é isso que a figura tem a "
            "dizer.",
            "Média e taxa de vitória medem coisas diferentes da mesma distribuição; "
            "nenhuma das duas é 'a qualidade do modelo'.")):
        fig.text(.012, .042 - i * .024, linha, color=p["muted"], fontsize=8)
    fig.subplots_adjust(left=.20, right=.985, top=.80, bottom=.24)
    return fig, ax


#: Os três regimes que a arena sabe medir, na ordem em que fazem sentido lidos juntos.
#: Cada um responde uma pergunta diferente **sobre o mesmo modelo**, e nenhum é "o
#: resultado": ver `docs/COMPARABILITY.md`.
REGIMES = (
    ("final", "rede pura · último passo", "como o algoritmo terminou"),
    ("melhor", "rede pura · melhor checkpoint", "o melhor que ele produziu"),
    ("busca", "com busca", "o que você levaria para jogar"),
)


def _mediana_da_busca(registros):
    """`(mediana, minimo, maximo, n, sims)` das entradas **oficiais** de `busca`.

    Devolve `None` quando nenhuma semente do grupo tem medição válida. `n` é quantas
    sementes contribuíram, que quase nunca é o total do grupo — medir com busca custa
    horas e normalmente só uma semente foi medida sob o protocolo inteiro. É por isso que
    o `n` volta daqui em vez de ser assumido: uma barra sustentada por uma semente não
    pode ser desenhada como as outras.
    """
    valores, sims = [], set()
    for r in registros:
        melhor = r.melhor_com_busca()
        if melhor is None:
            continue
        valores.append(float(melhor["score_mean"]))
        if melhor.get("num_simulations") is not None:
            sims.add(int(melhor["num_simulations"]))
    if not valores:
        return None
    v = np.array(valores, dtype=np.float64)
    return float(np.median(v)), float(v.min()), float(v.max()), len(v), sorted(sims)


def _mediana_de(registros, campo, chave="score_mean"):
    """`(mediana, minimo, maximo, n)` de `record.<campo>[chave]` entre sementes."""
    valores = [r_c[chave] for r in registros
               if (r_c := getattr(r, campo, None) or {}).get(chave) is not None]
    if not valores:
        return None
    v = np.array(valores, dtype=np.float64)
    return float(np.median(v)), float(v.min()), float(v.max()), len(v)


def arena_melhores(registros, mode="light", figsize=(10.4, 5.6), titulo=None,
                   so_principais=True):
    """Os três regimes lado a lado: como terminou, o melhor que produziu, e com busca.

    Por que este painel existe
    --------------------------
    A pergunta "quais são os melhores modelos nas suas melhores tentativas" não tem uma
    resposta só, e a tentação é responder com **um máximo** — o maior número que qualquer
    semente de qualquer algoritmo produziu em qualquer regime. Isso seria um estimador
    enviesado de um jeito específico e evitável: o máximo cresce com o número de sorteios,
    então ele premia quem **rodou mais sementes**, não quem é melhor. O AlphaZero tem três
    sementes e o `acktr/resnet_small_regua_antiga` tem uma; comparar os máximos dos dois é
    comparar 3 sorteios com 1.

    Então aqui cada barra é a **mediana entre sementes**, dentro do regime, exatamente
    como a arena oficial faz. O que varia entre as barras é a *pergunta*, não a
    estatística. A linha fina sobre a barra é o intervalo entre as sementes.

    O viés que sobra, e que fica escrito na figura
    ----------------------------------------------
    `melhor` é um máximo sobre os ~20 pontos de avaliação da execução, então ele é
    otimista **por construção**, mesmo com a mediana entre sementes por cima. O tamanho
    do viés não é uniforme: com 1000 episódios o erro padrão é `desvio/√1000`, que vale
    ~0,25 para o AlphaZero (desvio 7,9) e ~0,88 para o MuZero (desvio 27,9). Sobre um
    platô de ~8 pontos, `E[máx]` fica ~2 desvios padrão acima da média — ou seja, meio
    ponto para um, cerca de dois pontos para o outro. **Quanto mais instável o algoritmo,
    mais o `melhor` o favorece.**

    Isso não invalida a coluna: uma queda como a do `rainbow/completo/seed1` (final 43,50,
    melhor 86,13) é 42 pontos, muito além de qualquer seleção sobre ruído — é colapso de
    verdade, e é exatamente o que a coluna existe para mostrar. Mas uma diferença de dois
    ou três pontos entre `final` e `melhor` não significa nada.

    A barra `busca` é a única que **não** divide eixo com as outras duas: ela gasta
    dezenas de avaliações de rede por jogada contra uma do PPO. Ela está aqui porque é o
    que se leva para jogar, e está marcada com o orçamento de busca que a produziu.
    """
    import matplotlib.pyplot as plt

    p = PALETA[mode]
    comparaveis = [r for r in registros if r.oficial]
    if so_principais:
        comparaveis, _ = separa_principais(comparaveis)

    linhas = []
    for (algo, variante), rs in _agrupa(comparaveis).items():
        fin = _mediana_de(rs, "final")
        if fin is None:
            continue
        mel = _mediana_de(rs, "melhor")
        bus = _mediana_da_busca(rs)
        linhas.append({
            "nome": algo if variante in ("default", "") else f"{algo} · {variante}",
            "sementes": len(rs),
            "final": fin, "melhor": mel, "busca": bus,
            # a ordenação é pela melhor tentativa disponível — é a pergunta da figura
            "topo": max(v[0] for v in (fin, mel, bus and bus[:4]) if v),
        })
    linhas.sort(key=lambda d: d["topo"])

    fig, ax = plt.subplots(figsize=figsize, facecolor=p["plane"])
    ax.set_facecolor(p["surface"])

    cores = {"final": p["series"][0], "melhor": p["series"][2], "busca": p["series"][3]}
    altura, folga = 0.24, 0.02
    deslocamento = {"final": -(altura + folga), "melhor": 0.0, "busca": altura + folga}

    # a legenda é registrada no primeiro grupo que TEM cada regime, não na primeira
    # linha: a barra com busca só aparece em um ou dois algoritmos, e amarrá-la a `i == 0`
    # deixaria a figura com uma cor sem entrada na legenda
    rotulado = set()
    for i, linha in enumerate(linhas):
        for campo, rotulo, _pergunta in REGIMES:
            dado = linha[campo]
            if dado is None:
                continue
            mediana, minimo, maximo, n = dado[:4]
            y = i - deslocamento[campo]
            ax.barh(y, mediana, color=cores[campo], height=altura, zorder=3,
                    label=None if campo in rotulado else rotulo,
                    # uma barra sustentada por menos sementes que o grupo não pode ser
                    # desenhada igual às outras — ela é hachurada e leva o `n` no rótulo
                    hatch=None if n >= linha["sementes"] else "///",
                    edgecolor=p["surface"], linewidth=1.0)
            rotulado.add(campo)
            if maximo > minimo:
                ax.plot([minimo, maximo], [y, y], color=p["ink2"], lw=1.1, zorder=4,
                        solid_capstyle="butt")
            texto = f"{mediana:.2f}".replace(".", ",")
            if campo == "busca" and dado[4]:
                texto += " · " + "/".join(str(x) for x in dado[4]) + " sims"
            if n < linha["sementes"]:
                texto += f" · n={n}"
            # o rótulo vai depois do que estiver mais à direita, barra ou bigode — senão
            # ele cai em cima da linha do intervalo justamente onde ela é grande
            ax.annotate(texto, xy=(max(mediana, maximo), y), xytext=(5, 0),
                        textcoords="offset points", color=p["ink2"], fontsize=8.5,
                        va="center", ha="left", zorder=5)

    ax.axvline(SCORE_PERFEITO, color=p["muted"], lw=1.0, ls=(0, (4, 3)), zorder=2)
    # embaixo, e não em cima: a faixa de cima é da legenda e a primeira linha é a mais
    # longa — é justamente lá que o rótulo do teto colidiria com o número da barra
    ax.annotate(f"tabuleiro cheio · {SCORE_PERFEITO}",
                xy=(SCORE_PERFEITO, 0.0), xycoords=("data", "axes fraction"),
                xytext=(-5, 4), textcoords="offset points", color=p["muted"],
                fontsize=8.5, ha="right", va="bottom", zorder=5)

    ax.set_yticks(range(len(linhas)), [linha["nome"] for linha in linhas],
                  color=p["ink2"], fontsize=9.5)
    ax.set_ylim(-0.6, len(linhas) - 0.4)
    ax.set_xlim(0, SCORE_PERFEITO * 1.42)
    ax.set_xlabel("score de avaliação · mediana entre sementes, 1.000 episódios, greedy",
                  color=p["ink2"], fontsize=10, loc="left")
    ax.set_title(titulo or "snake-arena · a mesma execução, três perguntas",
                 color=p["ink"], fontsize=13, pad=26, loc="left")
    ax.grid(True, axis="x", color=p["grid"], lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for lado in ("top", "right", "left"):
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color(p["axis"])
    ax.tick_params(colors=p["muted"], labelsize=9, length=0)

    if linhas:
        leg = ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.0), ncols=3,
                        frameon=False, fontsize=9, handlelength=1.2)
        for t in leg.get_texts():
            t.set_color(p["ink2"])

    notas = (
        "Cada barra é a MEDIANA entre sementes, não o máximo: o máximo premia quem rodou "
        "mais sementes. A linha fina é o intervalo entre elas.",
        "`melhor` é um máximo sobre ~20 avaliações, então é otimista por construção — e "
        "mais para quem oscila (~2 pontos) que para quem é estável (~0,5).",
        "`com busca` NÃO divide eixo com as outras duas: gasta dezenas de avaliações de "
        "rede por jogada contra uma do PPO. Hachura = menos sementes que o grupo.",
    )
    for i, nota in enumerate(notas):
        fig.text(.012, .058 - i * .022, nota, color=p["muted"], fontsize=7.6)
    fig.subplots_adjust(left=.22, right=.99, top=.86, bottom=.24)
    return fig, ax


def _mediana_do_final(registros, chave):
    """Mediana entre sementes de um campo de `final` — a estatística oficial da arena."""
    valores = [r.final.get(chave) for r in registros if r.final.get(chave) is not None]
    return float(np.median(valores)) if valores else None


def arena_tempo(registros, mode="light", figsize=(7.4, 5.0), titulo=None,
                limiar=LIMIAR_PADRAO, so_principais=True):
    """A arena no eixo de **custo**: score contra horas de GPU. Devolve `(fig, ax)`.

    O eixo oficial — passos de ambiente — iguala os *dados vistos*. É o padrão da
    literatura e é o certo para "quem aprende mais com a mesma experiência". Mas ele
    esconde uma diferença enorme: o AlphaZero roda uma busca em árvore a cada passo e custa
    ordens de grandeza mais que o DQN para chegar ao mesmo ponto no eixo x. Comparar ali dá
    a ele computação de graça.

    Este painel mostra a outra metade da verdade, e **não substitui** o oficial: são duas
    perguntas diferentes, e a resposta de uma não vale para a outra.

    Quando as execuções vêm de hardwares diferentes o gráfico **diz isso na cara**, porque
    aí ele compara aceleradores e não algoritmos — e essa é a forma mais fácil de ler um
    número errado com confiança.
    """
    import matplotlib.pyplot as plt

    p = PALETA[mode]
    comparaveis = [r for r in registros if r.oficial]
    if so_principais:
        comparaveis, _ = separa_principais(comparaveis)

    algos = {r.algo for r in comparaveis}
    if len(algos) > len(p["series"]):
        # A mesma regra do painel principal: acima de oito, a saída é mudar a forma do
        # gráfico, não gerar cor nova. Aqui isso vira um painel de tempo por família.
        return arena_tempo_familias(registros, mode=mode, titulo=titulo,
                                    so_principais=so_principais)
    # `cores_por_algoritmo`, e **não** `cores_por_familia`: num painel único a cor por
    # posição-dentro-da-família repete matiz entre famílias, e três curvas azuis no mesmo
    # eixo é exatamente a ambiguidade que a paleta existe para evitar.
    cores = cores_por_algoritmo(algos, mode)

    fig, ax = plt.subplots(figsize=figsize, facecolor=p["plane"])
    ax.set_facecolor(p["surface"])
    ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)

    rotulos, topo = [], 0.0
    for (algo, variante), rs in sorted(_agrupa(comparaveis).items()):
        ag = agrega_tempo(rs)
        if ag is None:
            continue
        cor = cores[algo]
        nome = algo if variante in ("default", "") else f"{algo} · {variante}"
        ax.fill_between(ag["x"], ag["q1"], ag["q3"], color=cor, alpha=.16, linewidth=0,
                        zorder=3)
        ax.plot(ag["x"], ag["mediana"], color=cor, lw=2.0, zorder=4,
                label=f"{nome}  (n={ag['n_sementes']})", solid_capstyle="round")
        rotulos.append((ag["x"][-1], ag["mediana"][-1], nome, cor))
        topo = max(topo, float(ag["mediana"].max()))

    for x, y, nome, _ in _sem_colisao(rotulos):
        ax.annotate(nome, xy=(x, y), xytext=(6, 0), textcoords="offset points",
                    color=p["ink2"], fontsize=9, va="center", ha="left", zorder=5)

    ax.set_xscale("log")
    ax.set_xlabel("horas de GPU (inclui as avaliações periódicas)",
                  color=p["ink2"], fontsize=10)
    ax.set_ylabel("score na avaliação (1.000 episódios, greedy)",
                  color=p["ink2"], fontsize=10)
    ax.set_title(titulo or "snake-arena · o mesmo resultado, no eixo do custo",
                 color=p["ink"], fontsize=13, pad=14, loc="left")
    ax.set_ylim(0, max(topo * 1.3, PISO_ALEATORIO * 4))
    ax.margins(x=.22)
    ax.grid(True, which="major", color=p["grid"], lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(p["axis"])
    ax.tick_params(colors=p["muted"], labelsize=9, length=0)
    if rotulos:
        leg = ax.legend(loc="upper left", frameon=False, fontsize=9)
        for t in leg.get_texts():
            t.set_color(p["ink2"])

    igual, hw = mesmo_hardware(comparaveis)
    aviso = ("mesmo hardware em todas as execuções: " + (next(iter(hw)) if hw else "—")
             if igual else
             "⚠ HARDWARES DIFERENTES (" + " · ".join(sorted(hw)) +
             "): este eixo está comparando aceleradores, não algoritmos")
    fig.text(.012, .015, aviso, color=p["muted"] if igual else p["ink"], fontsize=8.5)
    fig.subplots_adjust(left=.115, right=.97, top=.9, bottom=.145)
    return fig, ax


def arena_tempo_familias(registros, mode="light", figsize=(14.5, 4.6), titulo=None,
                         so_principais=True):
    """`arena_tempo` acima de oito algoritmos: um painel por família, o resto em cinza."""
    import matplotlib.pyplot as plt

    p = PALETA[mode]
    cores = cores_por_familia(mode)
    comparaveis = [r for r in registros if r.oficial]
    if so_principais:
        comparaveis, _ = separa_principais(comparaveis)

    agregados = {}
    for chave, rs in sorted(_agrupa(comparaveis).items()):
        ag = agrega_tempo(rs)
        if ag is not None:
            agregados[chave] = ag

    presentes = [f for f in FAMILIAS if any(a in f[2] for a, _ in agregados)] or FAMILIAS
    fig, axes = plt.subplots(1, len(presentes), figsize=figsize, sharex=True, sharey=True,
                             facecolor=p["plane"])
    axes = np.atleast_1d(axes)
    topo = max((float(a["mediana"].max()) for a in agregados.values()), default=0.0)

    for i, (ax, (_, rotulo, membros)) in enumerate(zip(axes, presentes)):
        ax.set_facecolor(p["surface"])
        ax.axhline(PISO_ALEATORIO, color=p["muted"], lw=1.0, zorder=1)
        for (algo, _), ag in agregados.items():
            if algo not in membros:
                ax.plot(ag["x"], ag["mediana"], color=p["legado"], lw=1.2, alpha=.45,
                        zorder=2)
        for (algo, variante), ag in agregados.items():
            if algo not in membros:
                continue
            nome = algo if variante in ("default", "") else f"{algo} · {variante}"
            ax.plot(ag["x"], ag["mediana"], color=cores[algo], lw=2.0, zorder=4,
                    label=f"{nome}  (n={ag['n_sementes']})", solid_capstyle="round")
        ax.set_xscale("log")
        ax.set_ylim(0, max(topo * 1.3, PISO_ALEATORIO * 4))
        ax.set_title(rotulo, color=p["ink2"], fontsize=10.5, loc="left", pad=10)
        ax.set_xlabel("horas de GPU", color=p["ink2"], fontsize=9.5)
        ax.grid(True, color=p["grid"], lw=0.8, zorder=0)
        ax.set_axisbelow(True)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)
        ax.tick_params(colors=p["muted"], labelsize=9, length=0, labelleft=(i == 0))
        if any(a in membros for a, _ in agregados):
            leg = ax.legend(loc="upper left", frameon=False, fontsize=8.5)
            for t in leg.get_texts():
                t.set_color(p["ink2"])

    axes[0].set_ylabel("score na avaliação", color=p["ink2"], fontsize=10)
    fig.suptitle(titulo or "snake-arena · o mesmo resultado, no eixo do custo",
                 color=p["ink"], fontsize=13, x=.006, ha="left", y=.985)
    igual, hw = mesmo_hardware(comparaveis)
    fig.text(.006, .015,
             ("mesmo hardware: " + (next(iter(hw)) if hw else "—")) if igual else
             "⚠ HARDWARES DIFERENTES (" + " · ".join(sorted(hw)) +
             "): este eixo compara aceleradores, não algoritmos",
             color=p["muted"] if igual else p["ink"], fontsize=8.5)
    fig.subplots_adjust(left=.058, right=.99, top=.83, bottom=.17)
    return fig, tuple(axes)


def arena_table(registros, markdown=True, limiar=LIMIAR_PADRAO):
    """A tabela de resultados — a visão que o gráfico não dá.

    Existe também porque três cores da paleta clara ficam abaixo de 3:1 de contraste:
    a regra manda oferecer rótulos visíveis **ou** a visão em tabela. Aqui temos as duas.
    """
    linhas = []
    for (algo, variante), rs in sorted(_agrupa([r for r in registros if r.oficial]).items()):
        finais = [r.final for r in rs if r.final]
        if not finais:
            continue
        medias = np.array([f["score_mean"] for f in finais], dtype=np.float64)
        passos = max((r.curve[-1]["global_step"] for r in rs if r.curve), default=0)
        linhas.append({
            "algo": algo,
            "variante": variante,
            "rede": rs[0].net,
            "params": rs[0].params,
            "sementes": len(rs),
            "passos": int(passos),
            "score_mean": float(np.median(medias)),
            "score_spread": float(medias.max() - medias.min()) if len(medias) > 1 else 0.0,
            "score_median": float(np.median([f.get("score_median", np.nan) for f in finais])),
            "score_max": int(max(f.get("score_max", 0) for f in finais)),
            "win_rate": float(np.median([f.get("win_rate", 0.0) for f in finais])),
            # coluna à parte, como o filtro de flood-fill e a busca do AlphaZero: o
            # melhor checkpoint responde "o melhor que este algoritmo produziu", que não
            # é a mesma pergunta que "como ele terminou"
            "melhor_mean": float(np.median(
                [r.melhor["score_mean"] for r in rs if r.melhor])) if any(
                    r.melhor for r in rs) else None,
            # a terceira coluna, pelo mesmo motivo: o agente medido com a máquina que ele
            # usa para jogar. Só entradas que cumprem o protocolo do contrato
            # (`busca_oficial`); as espiadas de 200 episódios ficam gravadas e fora daqui
            "busca": _mediana_da_busca(rs),
            # A leitura HORIZONTAL da curva: em vez de "quanto marcou no fim", "quantos
            # passos precisou para chegar a `limiar`". Sai dos mesmos dados e responde à
            # outra pergunta — eficiência amostral no sentido estrito.
            **_ate_o_limiar(rs, limiar),
            "horas": float(np.median([r.meta.get("wall_s_total", np.nan) / 3600
                                      for r in rs])),
        })
    linhas.sort(key=lambda d: -d["score_mean"])

    if not markdown:
        return linhas

    out = [
        "| algoritmo | rede | params | sementes | passos | score (last) | melhor ckpt | "
        + f"com busca | passos até {limiar:.0f} | horas | amplitude | mediana/ep | máx | cheio |",
        "|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
        f"| _piso aleatório_ | — | — | — | 0 | **{PISO_ALEATORIO:.2f}** | — | — | — | — | — | 1 | — | 0% |".replace(".", ","),
    ]
    for d in linhas:
        nome = d["algo"] if d["variante"] in ("default", "") else f"{d['algo']} · {d['variante']}"
        melhor = f"{d['melhor_mean']:.2f}" if d["melhor_mean"] is not None else "—"
        if d["busca"] is None:
            busca = "—"
        else:
            busca = f"{d['busca'][0]:.2f}"
            if d["busca"][4]:
                busca += f" ({'/'.join(str(x) for x in d['busca'][4])} sims)"
            if d["busca"][3] < d["sementes"]:
                busca += f" · n={d['busca'][3]}"
        if d["passos_ate"] is None:
            ate = "não chegou"
        else:
            ate = f"{d['passos_ate']:,}"
            if d["sementes_ate"] < d["sementes"]:
                ate += f" ({d['sementes_ate']}/{d['sementes']})"
        horas = f"{d['horas']:.1f}" if np.isfinite(d["horas"]) else "—"
        out.append(
            f"| {nome} | `{d['rede']}` | {d['params']:,} | {d['sementes']} | "
            f"{d['passos']:,} | **{d['score_mean']:.2f}** | {melhor} | {busca} | {ate} | "
            f"{horas} | "
            f"±{d['score_spread']:.2f} | "
            f"{d['score_median']:.0f} | {d['score_max']} | {d['win_rate']:.1%} |"
        )
    out.append(f"\nScore perfeito no 10×10: **{SCORE_PERFEITO}**.")
    out.append(
        f"\n**passos até {limiar:.0f}** é a curva lida na horizontal em vez da vertical: "
        "em vez de *quanto marcou no fim*, *quantos passos precisou para chegar lá*. Sai "
        "dos mesmos dados e responde à outra pergunta — menor é melhor. A resolução é a "
        "cadência de avaliação, e não há interpolação: o passo mostrado é um em que a "
        "medição de fato aconteceu. `(k/n)` significa que só `k` das `n` sementes "
        "chegaram, e as que não chegaram ficam **fora** da mediana em vez de entrar como "
        "um número inventado."
    )
    out.append(
        "\n**horas** é tempo de parede da execução inteira, útil só entre execuções do "
        "mesmo hardware. O eixo de passos iguala os *dados vistos*; ele não iguala o "
        "*esforço*, e a diferença entre os dois é enorme para quem faz busca em árvore."
    )
    out.append(
        "\nA coluna **score (last)** é o número oficial: o modelo do último passo, que é o "
        "estado final do algoritmo. O valor é a **mediana entre as sementes** do score "
        "médio de cada uma — não a média entre elas. É a mesma estatística que o gráfico "
        "desenha como linha, com o intervalo entre sementes como faixa, e com três "
        "sementes ela é o que uma semente divergente não consegue arrastar. Os documentos "
        "de ablação (`ORCAMENTO_DE_GRADIENTE.md`, `CANAL_DE_FOME.md`) reportam **média e "
        "desvio**, porque lá a pergunta é o tamanho de um efeito, não a ordem de um "
        "ranking: os dois números convivem, e cada um diz qual é. **mediana/ep** é outra "
        "coisa ainda — a mediana entre *episódios*, não entre sementes. **melhor ckpt** é "
        "o melhor que aquela execução produziu em algum momento — fica à parte porque "
        "premia quem foi medido mais vezes, pela mesma razão que a busca do AlphaZero e o "
        "filtro de flood-fill ficam fora da curva."
    )
    out.append(
        "\n**com busca** é o agente medido com a máquina que ele de fato usa para jogar — "
        "a árvore do AlphaZero e do MuZero — no mesmo protocolo de 1.000 episódios. Ela "
        "fica numa coluna separada, e não na curva, porque **não divide eixo**: uma "
        "jogada com 32 simulações gasta dezenas de avaliações de rede contra uma do PPO, "
        "e desenhá-las juntas daria computação de graça a quem busca. `n=k` marca quantas "
        "sementes foram medidas sob o protocolo inteiro; medir com busca custa horas, "
        "então quase sempre é menos que o total. Medições parciais — menos episódios que "
        "o contrato, ou que estouraram o teto de tempo — ficam gravadas em `busca` e "
        "**não** aparecem aqui: uma amostra que acabou por tempo é enviesada para "
        "episódios curtos, que são justamente os ruins."
    )
    return "\n".join(out)


# --- snakeai/nets/resnet.py ---
"""Tronco residual totalmente convolucional — a rede do PPO.

No espírito do AlphaZero, mas minúsculo. Convoluções 3×3 com `padding="same"` num
tabuleiro 10×10 dão campo receptivo global depois de ~5 camadas, então 3 blocos residuais
já enxergam o tabuleiro inteiro — **sem jogar fora a posição**, que é onde as redes com
pooling do repositório antigo se perdiam.

Sobre normalização: PPO e BatchNorm se dão mal. As estatísticas do rollout não batem com
as do minibatch de update, e o valor aprendido fica dependente do tamanho do lote. Usamos
**GroupNorm**, que normaliza por amostra e não tem esse problema — e que funciona igual
para DQN, o que mantém a comparação limpa.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

from keras import layers

__all__ = ["PRESETS", "residual_block", "resnet", "TRONCOS_RESIDUAIS"]

#: nome -> (largura, número de blocos residuais)
PRESETS = {
    "resnet_tiny": (32, 2),     # ~40k params com a cabeça
    "resnet_small": (48, 3),    # ~135k — o ponto doce
    "resnet_base": (64, 4),     # ~320k
}


def residual_block(x, largura, nome):
    y = layers.Conv2D(largura, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal", name=f"{nome}_c1")(x)
    y = layers.GroupNormalization(groups=8, name=f"{nome}_n1")(y)
    y = layers.Activation("relu", name=f"{nome}_a1")(y)
    y = layers.Conv2D(largura, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal", name=f"{nome}_c2")(y)
    y = layers.GroupNormalization(groups=8, name=f"{nome}_n2")(y)
    out = layers.Add(name=f"{nome}_add")([x, y])
    return layers.Activation("relu", name=f"{nome}_a2")(out)


def resnet(x, preset="resnet_small", nome=None):
    """Tronco residual. Devolve o mapa de features `(B, B, largura)`, **sem achatar**.

    Não achatar é de propósito: as cabeças convolucionais 1×1 de `heads.py` aproveitam a
    estrutura espacial, e achatar cedo seria desperdiçá-la.
    """
    if preset not in PRESETS:
        raise ValueError(f"preset desconhecido: {preset!r}. Use um de {list(PRESETS)}")
    largura, blocos = PRESETS[preset]
    nome = nome or preset

    x = layers.Conv2D(largura, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal", name=f"{nome}_stem_c")(x)
    x = layers.GroupNormalization(groups=8, name=f"{nome}_stem_n")(x)
    x = layers.Activation("relu", name=f"{nome}_stem_a")(x)
    for i in range(blocos):
        x = residual_block(x, largura, f"{nome}_res{i}")
    return x


TRONCOS_RESIDUAIS = {
    nome: (lambda x, _p=nome: resnet(x, preset=_p)) for nome in PRESETS
}


# --- snakeai/nets/classic.py ---
"""Os troncos convolucionais do `colab-rl`, portados para Keras 3 e corrigidos.

Estes são os corpos de rede que produziram as curvas históricas. Estão aqui para que a
pergunta "quanto do ganho é o algoritmo e quanto é a arquitetura?" tenha resposta medida
em vez de opinião.

Duas coisas que a portabilidade revelou, e que valem mais que o código
------------------------------------------------------------------------

**1. "CNN2" significava duas coisas diferentes no mesmo repositório.**

O `colab-rl` tinha as CNNs definidas em dois lugares, com a mesma numeração e conteúdo
diferente:

===========  ==============================  ==================================
nome         em `models/utilities/networks.py`  nos notebooks
===========  ==============================  ==================================
``CNN1``     16→32, **quebrada** (`return model`)  32→64→64 (Rainbow)
``CNN2``     16→32→32, **quebrada**             32→64→64 com regularização L2
``CNN3``     32→64→64 (Rainbow)                 3 blocos VGG com max-pooling
``CNN4``     não existia                        idem CNN3, com dropout
===========  ==============================  ==================================

Ou seja: o notebook chamado *"DQN (RMSprop - CNN2 - KL-Divergence)"* usava um tronco que
**não é** a `CNN2` do pacote. Um leitor que fosse ao `networks.py` entender o experimento
leria a rede errada. Aqui as redes têm nome descritivo (`cnn_rainbow`, `cnn_alphazero`,
`cnn_vgg`, `cnn_vgg_dropout`) e os apelidos numéricos apontam para as definições **dos
notebooks**, que são as que de fato rodaram.

**2. As redes com pooling destroem o tabuleiro.**

`cnn_vgg` e `cnn_vgg_dropout` aplicam três `MaxPooling2D(2, 2)` seguidos. Num tabuleiro
10×10 isso é ``10 → 5 → 2 → 1``: a saída do tronco tem **uma única célula**. Toda a
informação de *onde* as coisas estão no tabuleiro é jogada fora antes da cabeça densa —
sobra só "existe corpo em algum lugar", "existe comida em algum lugar".

Essas arquiteturas foram desenhadas para imagens 224×224, onde três poolings deixam 28×28.
Copiadas para 10×10, elas colapsam. É uma explicação forte para o platô dos notebooks que
as usavam, e por isso `cnn_vgg_sem_pool` existe: mesma rede, sem os poolings, para medir
exatamente quanto custou.

Os troncos são fiéis ao original de propósito. A correção fica na cabeça (`heads.py`) e
nas variantes explicitamente marcadas.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, regularizers

__all__ = [
    "cnn_rainbow",
    "cnn_alphazero",
    "cnn_vgg",
    "cnn_vgg_dropout",
    "cnn_vgg_sem_pool",
    "TRONCOS_CLASSICOS",
    "APELIDOS_LEGADOS",
]


def cnn_rainbow(x, nome="cnn_rainbow"):
    """32→64→64 com kernels 3×3, 2×2, 1×1, sem padding.

    Da implementação do Rainbow do @Kaixhin. É a `CNN1` dos notebooks e a `CNN3` do
    `networks.py` — a mesma rede com dois nomes.
    """
    x = layers.Conv2D(32, 3, activation="relu", name=f"{nome}_c1")(x)
    x = layers.Conv2D(64, 2, activation="relu", name=f"{nome}_c2")(x)
    x = layers.Conv2D(64, 1, activation="relu", name=f"{nome}_c3")(x)
    return layers.Flatten(name=f"{nome}_flat")(x)


def cnn_alphazero(x, l2const=1e-4, nome="cnn_alphazero"):
    """A mesma pilha da `cnn_rainbow`, com regularização L2 e ativação separada.

    É a `CNN2` **dos notebooks** — a que rodou no experimento "CNN2 - KL-Divergence".
    """
    reg = regularizers.l2(l2const)
    for i, (filtros, k) in enumerate(((32, 3), (64, 2), (64, 1)), start=1):
        x = layers.Conv2D(filtros, k, kernel_regularizer=reg, name=f"{nome}_c{i}")(x)
        x = layers.Activation("relu", name=f"{nome}_a{i}")(x)
    return layers.Flatten(name=f"{nome}_flat")(x)


def _blocos_vgg(x, nome, dropout=0.0, pooling=True):
    plano = ((16, 2), (32, 2), (64, 3))
    for b, (filtros, convs) in enumerate(plano, start=1):
        for c in range(1, convs + 1):
            x = layers.Conv2D(filtros, 3, activation="relu", padding="same",
                              name=f"{nome}_b{b}_c{c}")(x)
            if dropout:
                x = layers.Dropout(dropout, name=f"{nome}_b{b}_d{c}")(x)
        if pooling:
            x = layers.MaxPooling2D(2, strides=2, name=f"{nome}_b{b}_pool")(x)
    return layers.Flatten(name=f"{nome}_flat")(x)


def cnn_vgg(x, nome="cnn_vgg"):
    """Três blocos no estilo VGG com max-pooling. É a `CNN3` dos notebooks.

    **Atenção:** os três poolings reduzem um tabuleiro 10×10 a 1×1. Ver o cabeçalho do
    módulo. Mantida fiel ao original porque é o que produziu as curvas históricas.
    """
    return _blocos_vgg(x, nome, dropout=0.0, pooling=True)


def cnn_vgg_dropout(x, nome="cnn_vgg_dropout", taxa=0.1):
    """`cnn_vgg` com dropout de 0,1 após cada convolução. É a `CNN4` dos notebooks."""
    return _blocos_vgg(x, nome, dropout=taxa, pooling=True)


def cnn_vgg_sem_pool(x, nome="cnn_vgg_sem_pool"):
    """`cnn_vgg` sem os max-poolings — a variante de ablação.

    Não existia no repositório antigo. Existe aqui para responder, com número, quanto do
    platô daquelas execuções veio de colapsar o tabuleiro a uma célula.
    """
    return _blocos_vgg(x, nome, dropout=0.0, pooling=False)


#: Nome descritivo -> função de tronco.
TRONCOS_CLASSICOS = {
    "cnn_rainbow": cnn_rainbow,
    "cnn_alphazero": cnn_alphazero,
    "cnn_vgg": cnn_vgg,
    "cnn_vgg_dropout": cnn_vgg_dropout,
    "cnn_vgg_sem_pool": cnn_vgg_sem_pool,
}

#: Apelidos numéricos do repositório antigo. Apontam para as definições **dos
#: notebooks**, que são as que realmente rodaram (ver o cabeçalho do módulo).
APELIDOS_LEGADOS = {
    "cnn1": "cnn_rainbow",
    "cnn2": "cnn_alphazero",
    "cnn3": "cnn_vgg",
    "cnn4": "cnn_vgg_dropout",
}


# --- snakeai/nets/heads.py ---
"""Cabeças de rede — dueling, noisy e distribucional (C51).

São os componentes que separam um DQN simples de um Rainbow. Ficam separados dos troncos
de propósito: qualquer cabeça encaixa em qualquer tronco, e é isso que permite perguntar
"quanto o dueling vale?" com o resto do experimento congelado.

Todas foram reescritas para Keras 3. A `NoisyDense` do repositório antigo herdava de
`Dense` e mexia nos internals dela (`self.kernel`, `build` reimplementado), o que quebra
em qualquer versão moderna; esta é uma `Layer` própria, com `add_weight` e `keras.random`.
"""


import contextlib
import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

__all__ = ["NoisyDense", "CentraNaMedia", "dueling_head", "distributional_head", "q_de_distribuicao",
           "ruido_ligado"]


@keras.saving.register_keras_serializable(package="snakeai")
class NoisyDense(layers.Layer):
    """Camada densa com ruído fatorado nos pesos (Fortunato et al., 2017).

    Substitui a exploração ε-greedy por ruído aprendido: a rede começa barulhenta e vai
    reduzindo o próprio σ conforme fica confiante. A vantagem sobre o ε-greedy é que a
    exploração passa a ser **dependente do estado** — o agente explora onde ainda não sabe,
    não uniformemente.

    Uma decisão importante: **o ruído é desligado quando `training=False`**. O protocolo de
    avaliação do contrato é greedy e determinístico; se a rede sorteasse ruído durante o
    benchmark, o mesmo modelo daria números diferentes a cada execução e a comparação entre
    algoritmos perderia o sentido. Alguns trabalhos mantêm o ruído na avaliação — aqui não,
    e a escolha está registrada porque muda o número publicado.

    Só que **a coleta não é a avaliação**, e amarrar o ruído a `training` juntava as duas:
    a política de comportamento saía determinística, e um Rainbow com `eps=0` (porque "a
    exploração é responsabilidade da rede") passava o treino inteiro sem explorar nada. O
    atributo `ruido` desempata: `None` segue o `training`, `True` força ruído, `False`
    força determinismo. Use o gerenciador `ruido_ligado` — ele é para uso **eager**, na
    coleta; dentro de uma `tf.function` o valor vira constante no traçado.

    Parâmetros
    ----------
    units : int
        Dimensão de saída.
    sigma0 : float
        Escala inicial do ruído, dividida por `sqrt(entrada)`. 0,5 é o valor do paper.
    """

    #: Sorteia um ruído **por linha do lote** em vez de um por passada.
    #:
    #: O paper (Fortunato et al.) sorteia `ε` uma vez por passada e o compartilha com o
    #: lote inteiro — e isso é fiel quando existe **um** ambiente, que é o caso dele. Com
    #: `num_envs=64` a mesma passada decide a ação dos 64, então os 64 seguem a *mesma*
    #: política perturbada: são 64 cópias correlacionadas, não 64 exploradores. Medido, o
    #: tamanho efetivo cai para ~8 de 64.
    #:
    #: As implementações distribuídas do próprio lineage (Ape-X, R2D2) dão a cada ator a
    #: sua rede e o seu ruído — ou seja, um sorteio por ambiente é a **vetorização
    #: correta** do paper, não um desvio dele. Fica desligado por padrão mesmo assim,
    #: porque muda a política de comportamento e isso é decisão declarada.
    por_amostra: bool = False

    def __init__(self, units, activation=None, sigma0=0.5, seed=None, **kw):
        super().__init__(**kw)
        self.units = int(units)
        self.activation = keras.activations.get(activation)
        self.sigma0 = float(sigma0)
        self.seed = seed
        self.seed_generator = keras.random.SeedGenerator(seed)
        #: `None` = segue `training`; `True`/`False` forçam. Ver `ruido_ligado`.
        self.ruido = None

    def build(self, input_shape):
        entrada = int(input_shape[-1])
        limite = 1.0 / (entrada ** 0.5)
        sigma_ini = self.sigma0 / (entrada ** 0.5)

        self.w_mu = self.add_weight(
            shape=(entrada, self.units), name="w_mu",
            initializer=keras.initializers.RandomUniform(-limite, limite))
        self.w_sigma = self.add_weight(
            shape=(entrada, self.units), name="w_sigma",
            initializer=keras.initializers.Constant(sigma_ini))
        self.b_mu = self.add_weight(
            shape=(self.units,), name="b_mu",
            initializer=keras.initializers.RandomUniform(-limite, limite))
        self.b_sigma = self.add_weight(
            shape=(self.units,), name="b_sigma",
            initializer=keras.initializers.Constant(sigma_ini))
        self._entrada = entrada

    @staticmethod
    def _f(x):
        """`sign(x) * sqrt(|x|)` — a transformação que fatora o ruído no paper."""
        return ops.sign(x) * ops.sqrt(ops.abs(x))

    def call(self, inputs, training=False):
        ativo = self.ruido if self.ruido is not None else training

        if ativo and self.por_amostra:
            # Um ε por linha do lote. Materializar 512 matrizes de peso seria proibitivo,
            # mas a fatoração dispensa: com w[b] = w_mu + w_sigma·(ε_in[b] ⊗ ε_out[b]),
            #     (x[b] @ w[b])_j = (x[b] @ w_mu)_j + ε_out[b,j]·((x[b]·ε_in[b]) @ w_sigma)_j
            # e o viés segue o mesmo ε_out. Custa dois matmuls em vez de um.
            lote = ops.shape(inputs)[0]
            eps_in = self._f(keras.random.normal((lote, self._entrada),
                                                 seed=self.seed_generator))
            eps_out = self._f(keras.random.normal((lote, self.units),
                                                 seed=self.seed_generator))
            y = ops.matmul(inputs, self.w_mu) + self.b_mu
            y = y + eps_out * (ops.matmul(inputs * eps_in, self.w_sigma) + self.b_sigma)
            return self.activation(y) if self.activation is not None else y

        if ativo:
            eps_in = self._f(keras.random.normal((self._entrada,),
                                                 seed=self.seed_generator))
            eps_out = self._f(keras.random.normal((self.units,),
                                                  seed=self.seed_generator))
            w = self.w_mu + self.w_sigma * ops.outer(eps_in, eps_out)
            b = self.b_mu + self.b_sigma * eps_out
        else:
            w, b = self.w_mu, self.b_mu

        y = ops.matmul(inputs, w) + b
        return self.activation(y) if self.activation is not None else y

    def compute_output_shape(self, input_shape):
        return (*input_shape[:-1], self.units)

    def ruido_medio(self):
        """σ médio dos pesos — cai conforme a rede fica confiante. Bom de registrar."""
        return float(ops.convert_to_numpy(ops.mean(ops.abs(self.w_sigma))))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "units": self.units,
            "activation": keras.activations.serialize(self.activation),
            "sigma0": self.sigma0,
            "seed": self.seed,
        })
        return cfg


@keras.saving.register_keras_serializable(package="snakeai")
class CentraNaMedia(layers.Layer):
    """`x − média(x)` ao longo de um eixo. Camada registrada, **não** `Lambda`.

    O `Lambda` com uma função anônima não sobrevive a `save`/`load` no Keras 3: o
    desserializador recusa recarregar um lambda Python por risco de execução arbitrária, e
    exige `safe_mode=False`. `snakeai/nets/muzero.py` já tinha aprendido isso; a cabeça
    dueling e a do C51 não.

    O custo do descuido não é teórico. `AgentBase.avaliar_melhor()` recarrega o checkpoint
    `best` **no fim do treino**, então o `ValueError` chegava depois do orçamento inteiro
    gasto — 8.931 s de GPU numa execução do Rainbow, perdida na última linha. E só o
    Rainbow batia nisto: é o único agente com `dueling=True` **e** `n_atoms>0`, os dois
    caminhos que usavam `Lambda`.
    """

    def __init__(self, eixo=-1, **kw):
        super().__init__(**kw)
        self.eixo = int(eixo)

    def call(self, x):
        return x - ops.mean(x, axis=self.eixo, keepdims=True)

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        cfg = super().get_config()
        cfg["eixo"] = self.eixo
        return cfg


def _densa(tipo, unidades, ativacao=None, nome=None):
    if tipo == "noisy":
        return NoisyDense(unidades, activation=ativacao, name=nome)
    return layers.Dense(unidades, activation=ativacao, name=nome)


def dueling_head(x, n_actions, largura=256, densa="dense", nome="dueling"):
    """`Q(s,a) = V(s) + A(s,a) − média_a A(s,a)`.

    A subtração da média é o que torna a decomposição identificável: sem ela, somar uma
    constante a `V` e subtraí-la de `A` daria o mesmo `Q`, e as duas correntes poderiam
    derivar sem que a perda percebesse.

    O original usava a **média**; o paper também oferece o **máximo**. Ficamos na média,
    que é o padrão do Rainbow.
    """
    a = _densa(densa, largura, "relu", f"{nome}_a_h")(x)
    a = _densa(densa, n_actions, None, f"{nome}_a")(a)
    v = _densa(densa, largura, "relu", f"{nome}_v_h")(x)
    v = _densa(densa, 1, None, f"{nome}_v")(v)

    a_centrada = CentraNaMedia(eixo=-1, name=f"{nome}_center")(a)
    return layers.Add(name=f"{nome}_q")([v, a_centrada])


def distributional_head(x, n_actions, n_atoms=51, largura=256, densa="dense",
                        dueling=False, nome="c51"):
    """Cabeça categórica do C51: distribuição sobre `n_atoms` valores por ação.

    Em vez de estimar `Q(s,a)` — a média do retorno — o C51 estima a distribuição inteira.
    O ganho não é só estatístico: aprender uma distribuição dá um sinal de treino mais
    rico por transição, e é a peça que mais contribui no Rainbow.

    Devolve **logits** de forma `(lote, n_ações, n_átomos)`. A softmax e a projeção sobre
    o suporte ficam no agente, onde o `v_min`/`v_max` é conhecido.
    """
    if dueling:
        a = _densa(densa, largura, "relu", f"{nome}_a_h")(x)
        a = _densa(densa, n_actions * n_atoms, None, f"{nome}_a")(a)
        a = layers.Reshape((n_actions, n_atoms), name=f"{nome}_a_r")(a)

        v = _densa(densa, largura, "relu", f"{nome}_v_h")(x)
        v = _densa(densa, n_atoms, None, f"{nome}_v")(v)
        v = layers.Reshape((1, n_atoms), name=f"{nome}_v_r")(v)

        a_centrada = CentraNaMedia(eixo=1, name=f"{nome}_center")(a)
        return layers.Add(name=f"{nome}_logits")([v, a_centrada])

    h = _densa(densa, largura, "relu", f"{nome}_h")(x)
    h = _densa(densa, n_actions * n_atoms, None, f"{nome}_d")(h)
    return layers.Reshape((n_actions, n_atoms), name=f"{nome}_logits")(h)


def q_de_distribuicao(logits, suporte):
    """Colapsa a distribuição categórica em `Q(s,a)` — só para escolher a ação.

    `logits`: `(lote, n_ações, n_átomos)`. `suporte`: `(n_átomos,)`.
    """
    p = ops.softmax(logits, axis=-1)
    return ops.sum(p * ops.reshape(suporte, (1, 1, -1)), axis=-1)


def suporte_c51(v_min=-10.0, v_max=10.0, n_atoms=51):
    """Os `n_atoms` valores igualmente espaçados em `[v_min, v_max]`.

    Com recompensa `+1`/`−1` e γ = 0,995, o retorno de um episódio de Snake fica bem
    dentro de `[−2, 60]` — a faixa padrão de `[−10, 10]` do Atari é estreita demais aqui.
    O agente escolhe a sua; este é só o utilitário.
    """
    import numpy as np

    return np.linspace(v_min, v_max, n_atoms, dtype=np.float32)


@contextlib.contextmanager
def ruido_ligado(modelo, ativo=True, por_amostra=False):
    """Liga o ruído das `NoisyDense` de `modelo` dentro do bloco, e devolve como estava.

    Existe porque **coletar não é avaliar**. `NoisyDense.call` amarra o ruído a
    `training`, e ligar `training=True` na coleta traria junto tudo o que esse sinalizador
    significa nos outros troncos — o `Dropout` do `cnn_classic`, por exemplo. Este
    gerenciador mexe só nas camadas ruidosas.

    Uso **eager**, na escolha da ação. Dentro de uma `tf.function` o atributo é lido no
    traçado e vira constante no grafo, que não é o que se quer.
    """
    camadas = [c for c in _camadas(modelo) if isinstance(c, NoisyDense)]
    antes = [(c.ruido, c.por_amostra) for c in camadas]
    for c in camadas:
        c.ruido = ativo
        c.por_amostra = bool(por_amostra)
    try:
        yield camadas
    finally:
        for c, (r, pa) in zip(camadas, antes):
            c.ruido, c.por_amostra = r, pa


def _camadas(modelo):
    """Todas as camadas de `modelo`, inclusive as aninhadas em submodelos."""
    vistas, pilha = [], list(getattr(modelo, "layers", []))
    while pilha:
        c = pilha.pop()
        vistas.append(c)
        pilha.extend(getattr(c, "layers", []))
    return vistas


# --- snakeai/nets/registry.py ---
"""O registro de redes — qualquer tronco, para qualquer algoritmo, por string.

É isto que transforma "qual arquitetura é melhor?" numa ablação medida: o agente recebe
`net="cnn_vgg"` ou `net="resnet_small"` e o resto do experimento não muda. Sem isso, cada
comparação de rede viraria um notebook novo, que é como o repositório antigo acabou com
seis DQNs que ninguém conseguia comparar.

Todo modelo construído aqui obedece ao contrato: entrada `(B, B, 5)` egocêntrica,
saída de política com 3 ações relativas.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers


__all__ = [
    "TRONCOS",
    "listar_troncos",
    "build_backbone",
    "build_actor_critic",
    "build_actor_critic_populacao",
    "build_option_actor_critic",
    "build_q_network",
    "build_policy_q",
    "resumo",
]

TRONCOS = {**TRONCOS_RESIDUAIS, **TRONCOS_CLASSICOS}

#: A cabeça densa do repositório antigo tinha **3136** unidades. O número não é arbitrário
#: — é exatamente o achatamento da `cnn_rainbow` num tabuleiro 10×10 (7×7×64), o mesmo
#: valor do DQN do Atari por coincidência de kernels. Só que uma camada de 3136 sobre uma
#: entrada de 3136 são **9,8 milhões de parâmetros**, e ela era replicada nas duas
#: correntes do dueling. Sobre a `cnn_vgg`, que entrega 64 features, a mesma camada liga
#: 64 entradas a 3136 unidades: quase toda a capacidade do modelo depois de o tronco já
#: ter descartado a informação espacial.
#: O padrão aqui é 256. `LARGURA_DENSA_LEGADA` continua disponível para reproduzir o
#: original quando a fidelidade importar mais que o bom senso.
LARGURA_DENSA_LEGADA = 3136
LARGURA_DENSA_PADRAO = 256


def listar_troncos():
    """Nomes aceitos, incluindo os apelidos numéricos do repositório antigo."""
    return sorted(TRONCOS) + sorted(APELIDOS_LEGADOS)


def _resolve(nome):
    if nome in TRONCOS:
        return TRONCOS[nome], nome
    if nome in APELIDOS_LEGADOS:
        canonico = APELIDOS_LEGADOS[nome]
        return TRONCOS[canonico], canonico
    raise ValueError(
        f"tronco desconhecido: {nome!r}. Disponíveis: {listar_troncos()}"
    )


def build_backbone(entrada, net="resnet_small"):
    """Aplica o tronco `net` a um tensor de entrada. Devolve `(saida, nome_canonico)`."""
    fn, canonico = _resolve(net)
    return fn(entrada), canonico


def _entrada(board_size, canais=N_CHANNELS):
    """A entrada do tronco. `canais` só sai de 5 numa ablação declarada.

    O contrato fixa 5 canais, e mudar isso muda a **entrada da rede** — nenhuma curva de 5
    canais é comparável a uma de 6. O parâmetro existe para `VecSnake(canal_fome=True)`,
    que é uma ablação `comparable=False`, e não para configuração casual.
    """
    return keras.Input(shape=(board_size, board_size, canais), name="board")


def _e_espacial(t):
    """True se o tronco devolveu um mapa `(H, W, C)` em vez de um vetor achatado."""
    return len(t.shape) == 4


def build_actor_critic(board_size=10, net="resnet_small", largura_densa=None,
                       n_actions=N_ACTIONS, nome=None, canais=N_CHANNELS):
    """Modelo de duas saídas `[logits, valor]` — o que PPO, A2C e ACER consomem.

    Em troncos que preservam a estrutura espacial (as ResNets), as cabeças são
    convoluções 1×1 seguidas de achatamento, como no AlphaZero: mais barato e mais
    informativo que jogar um `Dense` gigante em cima de um mapa achatado. Em troncos
    clássicos, que já achatam, usa-se a cabeça densa mesmo.

    O `Dense` final da política nasce com `kernel_initializer` de ganho pequeno: no início
    do treino a política precisa ser quase uniforme, senão o PPO gasta as primeiras
    iterações desfazendo uma preferência aleatória.
    """
    inp = _entrada(board_size, canais)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)

    if _e_espacial(x):
        p = layers.Conv2D(4, 1, use_bias=False, name="pi_c")(x)
        p = layers.GroupNormalization(groups=2, name="pi_n")(p)
        p = layers.Activation("relu", name="pi_a")(p)
        p = layers.Flatten(name="pi_f")(p)

        v = layers.Conv2D(2, 1, use_bias=False, name="v_c")(x)
        v = layers.GroupNormalization(groups=2, name="v_n")(v)
        v = layers.Activation("relu", name="v_a")(v)
        v = layers.Flatten(name="v_f")(v)
        v = layers.Dense(largura, activation="relu", name="v_d")(v)
    else:
        p = layers.Dense(largura, activation="relu", name="pi_d")(x)
        v = layers.Dense(largura, activation="relu", name="v_d")(x)

    logits = layers.Dense(
        n_actions, name="logits",
        kernel_initializer=keras.initializers.Orthogonal(gain=0.01),
        bias_initializer="zeros",
    )(p)
    valor = layers.Dense(
        1, name="value",
        kernel_initializer=keras.initializers.Orthogonal(gain=1.0),
        bias_initializer="zeros",
    )(v)

    return keras.Model(inp, [logits, valor], name=nome or f"ac_{canonico}")


def build_actor_critic_populacao(board_size=10, net="resnet_small", n_politicas=3,
                                 largura_densa=None, n_actions=N_ACTIONS, nome=None,
                                 canais=N_CHANNELS):
    """`N` pares (política, valor) sobre um tronco **compartilhado** — o que o LBC consome.

    Saída `[logits, valor]` com formas `(lote, N, ações)` e `(lote, N)`: a população
    inteira num forward só. É essa forma que permite ao comportamento do LBC ser uma
    mistura sobre as `N` políticas sem `N` passadas pela rede.

    O tronco compartilhado é um **desvio declarado** do paper, que trata cada política como
    um modelo inteiro e independente (Assumption 1). A razão é o orçamento: o contrato deste
    repositório dá 5 M passos de ambiente a todos os algoritmos, e três ResNets separadas
    triplicariam o custo por passo — o LBC entraria na arena competindo com o mesmo
    orçamento de ambiente e três vezes mais computação, que é a comparação que este
    repositório existe para não fazer. Ver `docs/LBC.md`.

    O que se perde é diversidade de **representação**: as três políticas veem as mesmas
    features. O que se mantém é diversidade de **objetivo** — cada cabeça é treinada com o
    seu próprio γ e o seu próprio alvo V-trace — e é ela que constrói o espaço de
    comportamento não-degenerado do §4.1.
    """
    if int(n_politicas) < 1:
        raise ValueError("a população precisa de pelo menos uma política")
    n_politicas = int(n_politicas)

    inp = _entrada(board_size, canais)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)
    espacial = _e_espacial(x)

    logits_por_politica, valores_por_politica = [], []
    for i in range(n_politicas):
        if espacial:
            p = layers.Conv2D(4, 1, use_bias=False, name=f"pi{i}_c")(x)
            p = layers.GroupNormalization(groups=2, name=f"pi{i}_n")(p)
            p = layers.Activation("relu", name=f"pi{i}_a")(p)
            p = layers.Flatten(name=f"pi{i}_f")(p)

            v = layers.Conv2D(2, 1, use_bias=False, name=f"v{i}_c")(x)
            v = layers.GroupNormalization(groups=2, name=f"v{i}_n")(v)
            v = layers.Activation("relu", name=f"v{i}_a")(v)
            v = layers.Flatten(name=f"v{i}_f")(v)
            v = layers.Dense(largura, activation="relu", name=f"v{i}_d")(v)
        else:
            p = layers.Dense(largura, activation="relu", name=f"pi{i}_d")(x)
            v = layers.Dense(largura, activation="relu", name=f"v{i}_d")(x)

        li = layers.Dense(
            n_actions, name=f"logits_{i}",
            kernel_initializer=keras.initializers.Orthogonal(gain=0.01),
            bias_initializer="zeros",
        )(p)
        vi = layers.Dense(
            1, name=f"value_{i}",
            kernel_initializer=keras.initializers.Orthogonal(gain=1.0),
            bias_initializer="zeros",
        )(v)
        logits_por_politica.append(
            layers.Reshape((1, n_actions), name=f"logits_{i}_r")(li))
        valores_por_politica.append(vi)

    # `Concatenate` recusa uma entrada só — e uma população de tamanho 1 é justamente a
    # ablação "reduzir H" da Fig. 5 do paper, então este caminho tem que existir.
    if n_politicas == 1:
        logits = logits_por_politica[0]
        valor = valores_por_politica[0]
    else:
        logits = layers.Concatenate(axis=1, name="logits")(logits_por_politica)
        valor = layers.Concatenate(axis=-1, name="value")(valores_por_politica)

    return keras.Model(inp, [logits, valor],
                       name=nome or f"lbc{n_politicas}_{canonico}")


def build_option_actor_critic(board_size=10, net="resnet_small", n_opcoes=4,
                              largura_densa=None, n_actions=N_ACTIONS, nome=None,
                              canais=N_CHANNELS):
    """Política com opções — o que o SOAP consome. Três saídas:

    * `logits_a` `(lote, Z, ações)` — a sub-política `π_θ(a|s,z)`, uma por opção;
    * `logits_z` `(lote, Z, ações, Z)` — a transição `π_ψ(z'|s,a,z)`;
    * `valor` `(lote, Z)` — o crítico condicionado à opção corrente.

    A transição depende de `(s, a, z)`, e não só de `s`: é a fatoração que o paper do SOAP
    propõe contra a do Option-Critic, e é ela que permite a uma opção **persistir** por
    conta própria em vez de ser re-sorteada a cada passo. O custo é um tensor de saída
    `Z × A × Z` — com `Z = 4` e `A = 3`, 48 números por estado, que é barato.

    Os logits da sub-política nascem com ganho pequeno, como no `build_actor_critic`: no
    começo do treino toda opção precisa ser quase uniforme. Os da transição também, e por
    um motivo mais forte — uma preferência inicial de troca de opção é um viés que o agente
    gasta as primeiras iterações desfazendo, e enquanto isso a crença `ζ` já colapsou.
    """
    if int(n_opcoes) < 1:
        raise ValueError("é preciso pelo menos uma opção")
    n_opcoes = int(n_opcoes)

    inp = _entrada(board_size, canais)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)

    def projeta(nome_curto, filtros):
        if _e_espacial(x):
            h = layers.Conv2D(filtros, 1, use_bias=False, name=f"{nome_curto}_c")(x)
            h = layers.GroupNormalization(groups=2, name=f"{nome_curto}_n")(h)
            h = layers.Activation("relu", name=f"{nome_curto}_a")(h)
            h = layers.Flatten(name=f"{nome_curto}_f")(h)
            if nome_curto != "pi":
                h = layers.Dense(largura, activation="relu", name=f"{nome_curto}_d")(h)
            return h
        return layers.Dense(largura, activation="relu", name=f"{nome_curto}_d")(x)

    p = projeta("pi", 4)
    q = projeta("psi", 4)
    v = projeta("v", 2)

    logits_a = layers.Dense(
        n_opcoes * n_actions, name="logits_a_d",
        kernel_initializer=keras.initializers.Orthogonal(gain=0.01),
        bias_initializer="zeros",
    )(p)
    logits_a = layers.Reshape((n_opcoes, n_actions), name="logits_a")(logits_a)

    logits_z = layers.Dense(
        n_opcoes * n_actions * n_opcoes, name="logits_z_d",
        kernel_initializer=keras.initializers.Orthogonal(gain=0.01),
        bias_initializer="zeros",
    )(q)
    logits_z = layers.Reshape((n_opcoes, n_actions, n_opcoes),
                              name="logits_z")(logits_z)

    valor = layers.Dense(
        n_opcoes, name="value",
        kernel_initializer=keras.initializers.Orthogonal(gain=1.0),
        bias_initializer="zeros",
    )(v)

    return keras.Model(inp, [logits_a, logits_z, valor],
                       name=nome or f"soap{n_opcoes}_{canonico}")


def build_q_network(board_size=10, net="cnn_rainbow", largura_densa=None,
                    n_actions=N_ACTIONS, dueling=False, noisy=False, n_atoms=0,
                    nome=None, canais=N_CHANNELS):
    """A família DQN inteira num construtor só.

    `dueling`, `noisy` e `n_atoms` são os eixos que separam o DQN base do Rainbow — e são
    ortogonais de propósito, para que cada um possa ser medido isolado. Essa é a resposta
    aos seis notebooks quase idênticos do repositório antigo: uma função, seis chamadas.

    Saída
    -----
    `(lote, n_ações)` no modo normal; `(lote, n_ações, n_atoms)` de **logits** quando
    `n_atoms > 0` (C51).
    """
    inp = _entrada(board_size, canais)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)
    densa = "noisy" if noisy else "dense"

    if _e_espacial(x):
        x = layers.Conv2D(8, 1, use_bias=False, name="q_c")(x)
        x = layers.GroupNormalization(groups=2, name="q_n")(x)
        x = layers.Activation("relu", name="q_a")(x)
        x = layers.Flatten(name="q_f")(x)

    if n_atoms:
        saida = distributional_head(x, n_actions, n_atoms=n_atoms, largura=largura,
                                    densa=densa, dueling=dueling)
    elif dueling:
        saida = dueling_head(x, n_actions, largura=largura, densa=densa)
    else:
        h = _densa(densa, largura, "relu", "q_d")(x)
        saida = _densa(densa, n_actions, None, "q")(h)

    partes = [p for p, on in (("dueling", dueling), ("noisy", noisy),
                              (f"c51x{n_atoms}", bool(n_atoms))) if on]
    sufixo = ("_" + "_".join(partes)) if partes else ""
    return keras.Model(inp, saida, name=nome or f"q_{canonico}{sufixo}")


def build_policy_q(board_size=10, net="resnet_small", largura_densa=None,
                   n_actions=N_ACTIONS, nome=None, canais=N_CHANNELS):
    """Modelo de duas saídas `[logits, Q(s,·)]` — o que o ACER consome.

    Diferente do actor-critic comum: aqui o crítico devolve **um valor por ação**, não um
    escalar. É disso que o Retrace precisa, e `V(s) = Σ_a π(a|s) Q(s,a)` sai de graça —
    sem uma terceira cabeça e sem inconsistência entre V e Q, que é uma fonte clássica de
    bug silencioso em ACER.
    """
    inp = _entrada(board_size, canais)
    x, canonico = build_backbone(inp, net)
    largura = LARGURA_DENSA_PADRAO if largura_densa is None else int(largura_densa)

    if _e_espacial(x):
        p = layers.Conv2D(4, 1, use_bias=False, name="pi_c")(x)
        p = layers.GroupNormalization(groups=2, name="pi_n")(p)
        p = layers.Activation("relu", name="pi_a")(p)
        p = layers.Flatten(name="pi_f")(p)

        q = layers.Conv2D(8, 1, use_bias=False, name="q_c")(x)
        q = layers.GroupNormalization(groups=2, name="q_n")(q)
        q = layers.Activation("relu", name="q_a")(q)
        q = layers.Flatten(name="q_f")(q)
        q = layers.Dense(largura, activation="relu", name="q_d")(q)
    else:
        p = layers.Dense(largura, activation="relu", name="pi_d")(x)
        q = layers.Dense(largura, activation="relu", name="q_d")(x)

    logits = layers.Dense(
        n_actions, name="logits",
        kernel_initializer=keras.initializers.Orthogonal(gain=0.01),
        bias_initializer="zeros",
    )(p)
    q_saida = layers.Dense(n_actions, name="q", bias_initializer="zeros")(q)
    return keras.Model(inp, [logits, q_saida], name=nome or f"acer_{canonico}")


def resumo(board_size=10, largura_densa=None):
    """Tabela comparativa dos troncos: parâmetros e formato de saída.

    Usada no notebook de ablação e no README. Revela, de graça, quais troncos colapsam o
    tabuleiro — a coluna `saída do tronco` mostra `1×1` para os que usam pooling.
    """
    linhas = []
    for nome in sorted(TRONCOS):
        inp = _entrada(board_size)
        saida, _ = build_backbone(inp, nome)
        forma = tuple(saida.shape[1:])
        modelo = build_actor_critic(board_size, nome, largura_densa)
        tronco = keras.Model(inp, saida)
        linhas.append({
            "tronco": nome,
            "saida_tronco": "×".join(str(d) for d in forma),
            "espacial": _e_espacial(saida),
            "params_tronco": tronco.count_params(),
            "params_actor_critic": modelo.count_params(),
        })
    return linhas


# --- snakeai/agents/base.py ---
"""Andaime comum a todos os agentes.

O que fica aqui é o que **precisa** ser idêntico entre algoritmos para que a comparação
valha: a cadência da avaliação, o formato do registro, o critério de "melhor checkpoint",
e os agendamentos lineares. O que varia — como o agente aprende — fica em cada módulo.

Foi essa separação que faltou no repositório antigo: cada notebook tinha o próprio laço de
treino, a própria noção de época e o próprio jeito de avaliar, e por isso as curvas nunca
puderam ser sobrepostas.
"""


import contextlib
import json
import os
from collections import deque
from dataclasses import asdict, dataclass, field

import numpy as np


__all__ = ["BaseConfig", "AgentBase"]


@dataclass
class BaseConfig:
    """Os campos que todo agente do benchmark tem. Cada algoritmo estende com os seus."""

    board_size: int = CONTRATO["board_size"]
    net: str = "resnet_small"
    seed: int = 0

    #: Orçamento oficial. O contrato exige o **mesmo** valor para todos os algoritmos.
    total_steps: int = 5_000_000

    #: Avaliação periódica durante o treino, no protocolo oficial.
    eval_every_steps: int = 250_000
    eval_episodes: int = CONTRATO["eval_episodes"]
    eval_envs: int = 250

    ckpt_dir: str = "checkpoints"
    runs_dir: str = "runs"
    log_every_steps: int = 50_000

    #: Artefatos gerados no fim do treino. O GIF custa segundos e responde a pergunta que
    #: nenhuma curva responde: *como* o agente joga.
    salvar_grafico: bool = True
    #: Sufixo acrescentado à variante. Serve para que uma execução que muda
    #: hiperparâmetros — e portanto **compete**, mas não é a mesma coisa — não divida a
    #: identidade `(algo, variant, seed)` com a do padrão. `load_all` agrupa por essa
    #: tripla, então identidade repetida vira uma curva só, com as duas misturadas.
    sufixo_variante: str = ""

    salvar_gif: bool = True
    gif_seeds: tuple = (7, 21, 42)

    #: Marque `False` numa execução que muda o ambiente ou o protocolo de propósito — uma
    #: ablação. Ela continua sendo gravada e plotada, mas **fora da arena**, e `caveat`
    #: passa a ser obrigatório: uma curva incomparável sem o motivo escrito é pior que
    #: nenhuma curva, porque alguém vai compará-la mesmo assim.
    comparable: bool = True
    caveat: str = ""

    def __post_init__(self):
        if not self.comparable and not self.caveat:
            raise ValueError(
                "comparable=False exige `caveat` dizendo por que esta execução não "
                "compete. Sem isso a curva vira uma armadilha para quem ler depois.")
        if self.board_size != CONTRATO["board_size"]:
            raise ValueError(
                f"board_size={self.board_size} viola o contrato "
                f"({CONTRATO['board_size']}). Mude o contrato conscientemente, "
                "não a execução."
            )


def proximo_multiplo(passo, cadencia):
    """O menor múltiplo de `cadencia` **estritamente acima** de `passo`.

    É a diferença entre uma grade absoluta e uma que reancora: `passo + cadencia` faz cada
    avaliação cair um bloco depois da anterior, e o desvio se acumula — na execução padrão
    do PPO a última avaliação aconteceu 513 mil passos além do ponto nominal, 10% do
    orçamento. Como cada algoritmo avança em blocos de tamanho diferente (8.192 no A2C,
    49.152 no PPO padrão), as grades divergem entre si e a coluna `passos até 40`, que o
    contrato lê **sem interpolação**, passa a comparar medições feitas em lugares
    diferentes. Ver `docs/REVISAO_ALGORITMOS.md` §1.7.
    """
    return (int(passo) // int(cadencia) + 1) * int(cadencia)


class AgentBase:
    """Laço de treino comum: agendamentos, avaliação, checkpoint e registro.

    A subclasse implementa `iterate()` — um passo de aprendizado, que devolve estatísticas
    do rollout — e o resto vem de graça, igual para todo mundo.
    """

    algo = "base"

    #: Tamanho da janela da média móvel do treino, **em episódios**. 500 é da mesma ordem
    #: dos 1.000 da avaliação oficial: grande o bastante para o número não pular com um
    #: episódio de sorte, pequeno o bastante para acompanhar o agente melhorando.
    JANELA_EPISODIOS = 500

    #: Sufixo da variante para execuções fora do contrato de observação. A identidade de
    #: uma execução é `(algo, variant, seed)` — é por ela que `load_all` agrupa, não pelo
    #: caminho. Sem o sufixo, uma execução com `canal_fome=True` fica com a **mesma
    #: identidade** da execução de contrato da mesma rede e semente: hoje elas só não se
    #: misturam porque `comparable=False` as tira da arena, o que é proteção por acidente,
    #: não por construção. Ver `docs/CANAL_DE_FOME.md`.
    SUFIXO_FOME = "_fome"

    def __init__(self, cfg, variant="default"):
        self.cfg = cfg
        if getattr(cfg, "canal_fome", False):
            variant = self._com_sufixo(variant, self.SUFIXO_FOME)
        self.variant = self._com_sufixo(variant, getattr(cfg, "sufixo_variante", ""))
        self.model = None
        self.global_step = 0
        self.episodes = 0
        self.iteration = 0
        self.history = []
        self.evals = []
        self.baseline = None
        self.melhor = -np.inf
        self._proximo_eval = 0
        self._proximo_log = 0
        self._atualizacoes = 0
        #: Janela de episódios recentes para a média móvel do treino. Sem ela, o log
        #: imprime a média dos episódios que por acaso terminaram **naquela** iteração —
        #: uma amostra de tamanho 0 a 3. É o que produzia a sequência
        #: `2,50 · 10,00 · — · — · 2,00 · 11,00`, que parece instabilidade do algoritmo e
        #: é só tamanho de amostra. O `—` é literalmente "nenhum episódio acabou agora".
        #:
        #: A janela é medida em **episódios**, não em iterações, e a diferença não é
        #: cosmética. Uma iteração de PPO são 512 × 96 = 49.152 passos e ~200 episódios;
        #: uma de DQN são ~1.000 passos e 2 ou 3 episódios. Um limite fixo de iterações
        #: cobriria a execução inteira num caso e alguns segundos no outro — e no primeiro
        #: a "média móvel" viraria **média acumulada**, arrastada para baixo pelos
        #: episódios ruins do começo para sempre.
        self._janela = deque()
        #: Totais acumulados desde o início e no instante do log anterior. A janela móvel
        #: descarta pela esquerda, então a diferença entre dois logs **não** dá para ser
        #: reconstruída dela — e é essa diferença que responde "o que aconteceu nos
        #: últimos N episódios", que a média móvel de 500 esconde por construção. Numa
        #: iteração que produz ~80 episódios por log, a móvel arrasta 6 logs de história:
        #: uma degradação em curso aparece nela achatada e com atraso.
        self._acumulado = {"n": 0.0, **{c: 0.0 for c in self.CAMPOS_JANELA}}
        self._acumulado_no_log = dict(self._acumulado)
        self._legenda_impressa = False
        self._registrou_causas = False
        os.makedirs(cfg.ckpt_dir, exist_ok=True)

    # ----------------------------------------------------------- agendamentos
    #: Os campos somados na janela. Todos são **contagens ou somas** por bloco, nunca
    #: médias: só assim a média da janela pode ser ponderada pelo número de episódios de
    #: cada bloco, que é o que impede uma iteração com 2 episódios de pesar igual a uma
    #: com 200.
    CAMPOS_JANELA = ("score", "vitorias", "fome", "colisao", "passos")

    def _registra_episodios(self, media, n, **somas):
        """Guarda `n` episódios de score médio `media` e descarta o que saiu da janela.

        Descarta pela esquerda enquanto o que sobra ainda cobre `JANELA_EPISODIOS`, e
        nunca esvazia: com um algoritmo cuja iteração já produz mais episódios que a
        janela inteira, o certo é a janela ser aquela iteração — e não ficar vazia.
        """
        bloco = {"n": n, "score": media * n}
        bloco.update({c: float(somas.get(c, 0.0)) for c in self.CAMPOS_JANELA
                      if c != "score"})
        self._janela.append(bloco)
        self._acumulado["n"] += n
        for c in self.CAMPOS_JANELA:
            self._acumulado[c] += bloco[c]
        total = sum(b["n"] for b in self._janela)
        while len(self._janela) > 1 and total - self._janela[0]["n"] >= self.JANELA_EPISODIOS:
            total -= self._janela.popleft()["n"]

    def registra_fim(self, info):
        """Contabiliza os episódios que acabaram neste passo, **por causa**.

        Chamado de dentro do laço de coleta de cada agente, com o `info` que `VecSnake.step`
        devolve. É o único lugar onde a causa da morte existe: a memória de treino guarda
        `cont = 0` e nada mais, e depois do reset não há como saber se a cobra bateu ou
        passou fome.

        E a diferença entre as duas é justamente o que o score sozinho esconde. Um agente
        preso em 1,2 pontos pode estar batendo em tudo (não aprendeu a sobreviver) ou
        andando em círculo até morrer de fome (aprendeu a sobreviver e não a comer) — dois
        problemas opostos, com o mesmo número na curva. Foi exatamente essa ambiguidade que
        custou horas no diagnóstico do Dreamer.
        """
        n = int(info["scores"].size)
        if not n:
            return
        self._registrou_causas = True
        self._registra_episodios(
            float(info["scores"].mean()), n,
            vitorias=info["wins"],
            fome=info["starved"],
            # `deaths` conta colisão; vitória e fome não entram nele
            colisao=info["deaths"],
            passos=float(info["lengths"].sum()),
        )

    def media_movel(self):
        """Score médio dos últimos ~`JANELA_EPISODIOS` episódios, ponderado.

        `None` só quando nenhum episódio terminou ainda.
        """
        n = self.episodios_na_janela()
        return sum(b["score"] for b in self._janela) / n if n else None

    def episodios_na_janela(self):
        return sum(b["n"] for b in self._janela)

    def resumo_janela(self):
        """As frações que o score sozinho não conta, sobre a mesma janela de episódios.

        `{}` enquanto nenhum episódio terminou. As causas só aparecem se o agente chamar
        `registra_fim` — quem ainda não chama continua reportando só o score, sem inventar
        um zero que pareceria "nunca morre de fome".
        """
        n = self.episodios_na_janela()
        if not n:
            return {}
        soma = {c: sum(b.get(c, 0.0) for b in self._janela) for c in self.CAMPOS_JANELA}
        r = {"janela_episodios": n, "train_score_mean": soma["score"] / n}
        if self._registrou_causas:
            r.update({
                "win_rate": soma["vitorias"] / n,
                "frac_fome": soma["fome"] / n,
                "frac_colisao": soma["colisao"] / n,
                "passos_por_episodio": soma["passos"] / n,
            })
        return r

    def resumo_bloco(self):
        """As mesmas médias de `resumo_janela`, mas só sobre os episódios **novos**.

        "Novos" = terminados desde o log anterior. É o número que mostra a direção: a
        média móvel de 500 episódios responde "onde o agente está", o bloco responde "para
        onde está indo", e num treino que degrada as duas discordam por muito tempo antes
        de a móvel virar. Prefixadas com `bloco_` no registro para não colidirem com as
        chaves da janela, que são as que a arena e as curvas leem.

        `{}` quando nenhum episódio terminou desde o log anterior.
        """
        n = self._acumulado["n"] - self._acumulado_no_log["n"]
        if n <= 0:
            return {}
        d = {c: self._acumulado[c] - self._acumulado_no_log[c] for c in self.CAMPOS_JANELA}
        r = {"bloco_episodios": int(n), "bloco_train_score_mean": d["score"] / n}
        if self._registrou_causas:
            r.update({
                "bloco_win_rate": d["vitorias"] / n,
                "bloco_frac_fome": d["fome"] / n,
                "bloco_frac_colisao": d["colisao"] / n,
                "bloco_passos_por_episodio": d["passos"] / n,
            })
        return r

    def _marcar_bloco(self):
        """Fecha o bloco atual. Chamado depois de cada log, e só de lá."""
        self._acumulado_no_log = dict(self._acumulado)

    def frac(self):
        """Fração do orçamento já gasta, em [0, 1]. Base de todo agendamento linear."""
        return min(1.0, self.global_step / max(1, self.cfg.total_steps))

    def linear(self, inicio, fim):
        return inicio + self.frac() * (fim - inicio)

    # ----------------------------------------------------------- truncamento
    @staticmethod
    def desfaz_truncamento(info, prox_obs, prox_mask, done):
        """Devolve `(prox_obs, prox_mask, done)` com a morte por fome tratada como o que
        ela é: **truncamento**, não terminação.

        O `VecSnake` marca `done` para fome porque o episódio de fato acaba ali, mas
        exporta `trunc_idx`, `final_obs` e `final_mask` justamente para que o agente possa
        continuar o valor. Quem guarda a transição crua — DQN, Rainbow — tinha dois
        problemas de uma vez: gravava `done=1`, jogando fora o `γ·V(s')`, e gravava como
        `s'` a observação **do episódio seguinte**, porque o ambiente já resetou. O
        segundo é o pior: não havia como corrigir depois, o estado certo não estava no
        buffer.

        Aqui os dois somem: `s'` volta a ser o estado final verdadeiro e `done` volta a
        ser 0, que é exatamente o alvo de TD correto. Ver `docs/REVISAO_ALGORITMOS.md`
        §1.1.

        Não altera as entradas — devolve cópias.
        """
        ti = info.get("trunc_idx")
        if ti is None or len(ti) == 0:
            return prox_obs, prox_mask, done
        prox_obs = np.array(prox_obs, copy=True)
        prox_mask = np.array(prox_mask, copy=True)
        done = np.array(done, copy=True)
        prox_obs[ti] = info["final_obs"]
        prox_mask[ti] = info["final_mask"]
        done[ti] = 0.0
        return prox_obs, prox_mask, done

    @staticmethod
    def bootstrap_truncados(info, recompensas, valores_finais, gamma):
        """Soma `γ·V(s_final)` à recompensa dos episódios truncados por fome.

        A outra metade do tratamento de truncamento, para quem guarda **retornos** em vez
        de transições soltas. O `desfaz_truncamento` serve a quem tem um buffer de
        `(s, a, r, s')` e pode simplesmente devolver o `s'` verdadeiro; num rollout ou num
        segmento, o passo seguinte já pertence a outro episódio, e o valor do estado final
        precisa entrar **na recompensa** — que é como o PPO faz desde sempre.

        O `done` continua 1: a fronteira do episódio é real dentro do buffer, e é ela que
        impede o retorno de atravessar para o episódio seguinte. O que muda é que o
        retorno daquele passo deixa de valer −0,5 e passa a valer −0,5 + γ·V(s_final).

        Devolve uma cópia; sem truncamento, devolve a entrada intacta. Ver
        `docs/REVISAO_ALGORITMOS.md` §1.1.
        """
        ti = info.get("trunc_idx")
        if ti is None or len(ti) == 0:
            return recompensas
        saida = np.array(recompensas, copy=True, dtype=np.float32)
        saida[ti] += float(gamma) * np.asarray(valores_finais, dtype=np.float32)
        return saida

    # -------------------------------------------------------------- avaliação
    def politica(self):
        """A função de política que `snakeai.eval` consome. Sobrescreva se precisar."""
        return keras_policy(self.model)

    def politica_do_modelo(self, modelo):
        """A política de um modelo que veio **de fora** — um checkpoint, tipicamente.

        Existe porque `avaliar_melhor` trocava `self.model` e chamava `avaliar()`, o que
        só funciona para quem joga por `self.model`. O MuZero declara `model` como
        propriedade com setter vazio e o DreamerV3 joga por `self.ator` dentro de uma
        política recorrente: nos dois, a troca não fazia nada e a coluna `melhor` do
        registro virava uma segunda medição do modelo **final**, gravada com o passo do
        checkpoint `best`. Ver `docs/REVISAO_ALGORITMOS.md` §1.4.

        Quem não consegue jogar a partir de um `.keras` sozinho deve levantar
        `NotImplementedError` com o motivo — `avaliar_melhor` transforma isso numa coluna
        ausente e explicada, que é honesto, em vez de um número errado.
        """
        return keras_policy(modelo)

    def avaliar(self, episodes=None, safety=False, politica=None):
        """Roda o protocolo oficial. **Nunca** com exploração — é o número honesto."""
        stats, _ = evaluate(
            politica or self.politica(),
            board_size=self.cfg.board_size,
            episodes=episodes or self.cfg.eval_episodes,
            num_envs=self.cfg.eval_envs,
            greedy=CONTRATO["eval_greedy"],
            safety=safety,
            seed=CONTRATO["eval_seed"],
            # o ambiente de avaliação tem que ter os mesmos canais que o de treino
            canal_fome=getattr(self.env, "canal_fome", False),
        )
        return stats

    def rodar_protocolo(self, escolher, episodes=1000, seed=123, num_envs=None,
                        max_segundos=None, verbose=False):
        """O protocolo oficial com uma regra de escolha que precisa de mais que `(obs, mask)`.

        `snakeai.eval.evaluate` recebe uma política `(obs, mask) → logits`. Uma **busca**
        não cabe nessa interface: o AlphaZero precisa do estado do ambiente para restaurar
        nós da árvore, o MuZero precisa do latente da representação. Este laço é a mesma
        contabilidade do `evaluate` — episódios, semente, greedy, causas de fim — com a
        escolha delegada a `escolher(env, obs, mask) → ações`.

        Existe para que a coluna "com busca" não seja uma segunda implementação do
        protocolo. As duas armadilhas abaixo estavam nas cópias manuais que este método
        substitui, e as duas produzem um número silenciosamente **baixo justamente nas
        vitórias** — que é o regime em que um agente bom passa a maior parte do tempo:

        * **o score sai de `info["scores"]`, não de `env.score` lido antes do passo.** O
          episódio que termina comendo — toda vitória por tabuleiro cheio é assim — perde
          exatamente um ponto na segunda forma. Ver
          `test_eval.py::test_a_winning_episode_scores_the_last_apple`;
        * **`win_rate` sai da amostra coletada**, `(scores == perfeito).mean()`, e não de um
          contador do laço. O laço continua rodando os ambientes que já cumpriram a cota, e
          somar as vitórias deles daria uma taxa que não corresponde aos episódios medidos.

        `max_segundos` existe porque este laço **não tem um custo previsível**: ele roda até
        cada ambiente fechar a cota, e um agente bom faz episódios longos — a coleta de
        1.000 episódios com busca chega a horas. Sem uma trava, a única saída é cancelar a
        célula e perder tudo. Com ela, o que deu tempo de medir volta com
        `completo=False`, que é o campo que o `validate()` já usa para recusar uma
        avaliação parcial: o número existe para você olhar, e não entra na arena por
        engano. `verbose` imprime o progresso, porque uma espera de uma hora sem uma linha
        na tela é indistinguível de um travamento.
        """
        import time as _time
        cfg = self.cfg
        n = num_envs or min(cfg.eval_envs, 64)
        env = VecSnake(n, cfg.board_size, rng=np.random.default_rng(seed),
                       canal_fome=getattr(self.env, "canal_fome", False))
        obs, mask = env.reset()
        por_env = int(np.ceil(episodes / n))
        coletados = [[] for _ in range(n)]
        motivos = {"fome": 0, "colisao": 0, "tabuleiro_cheio": 0}
        perfeito = cfg.board_size * cfg.board_size - 3
        faltam, passos = n, 0
        t0 = _time.time()
        proximo_aviso = 30.0
        esgotou = False

        while faltam > 0:
            obs, mask, _, done, info = env.step(escolher(env, obs, mask))
            passos += n
            gasto = _time.time() - t0
            if verbose and gasto >= proximo_aviso:
                proximo_aviso = gasto + 30.0
                feitos = sum(len(c) for c in coletados)
                alvo = n * por_env
                # Os `n` ambientes correm em sincronia, então os episódios fecham em
                # levas. Extrapolar com uma leva incompleta dá um número absurdo — na
                # primeira rodada o "faltam" chegava a 256 min para um trabalho de 12.
                # Só estima depois que a primeira leva fechou.
                if feitos >= n:
                    falta = (gasto / feitos) * (alvo - feitos)
                    quanto = f"faltam ~{falta / 60:.0f} min"
                else:
                    quanto = (f"primeira leva ainda correndo — a estimativa aparece "
                              f"quando os {n} ambientes fecharem o 1º episódio")
                print(f"    ... {feitos}/{alvo} episódios · {gasto / 60:.1f} min · "
                      f"{quanto}", flush=True)
            if max_segundos is not None and gasto > max_segundos:
                esgotou = True
                break
            truncados = set(info["trunc_idx"].tolist())
            for j, i in enumerate(np.nonzero(done)[0]):
                if len(coletados[i]) >= por_env:
                    continue
                s_final = int(info["scores"][j])
                coletados[i].append(s_final)
                if i in truncados:
                    motivos["fome"] += 1
                elif s_final == perfeito:
                    motivos["tabuleiro_cheio"] += 1
                else:
                    motivos["colisao"] += 1
                if len(coletados[i]) == por_env:
                    faltam -= 1

        scores = np.array([s for l in coletados for s in l][:episodes], dtype=np.int32)
        if scores.size == 0:
            raise RuntimeError(
                "nenhum episódio terminou dentro do tempo — aumente `max_segundos` ou "
                "reduza `num_simulations`")
        total = max(1, sum(motivos.values()))
        return {
            "episodes": int(scores.size),
            "score_mean": float(scores.mean()),
            "score_median": float(np.median(scores)),
            "score_std": float(scores.std()),
            "score_max": int(scores.max()),
            "score_p95": float(np.percentile(scores, 95)),
            "win_rate": float((scores == perfeito).mean()),
            "perfect_possible": perfeito,
            "env_steps_used": int(passos),
            "segundos": round(_time.time() - t0, 1),
            #: `False` quando o tempo acabou antes da cota. O `validate()` recusa uma
            #: avaliação parcial, e é isso que se quer: o número serve para olhar, não
            #: para entrar na arena por engano.
            "completo": not esgotou,
            **{f"fim_{k}": v / total for k, v in motivos.items()},
        }

    def piso(self):
        if self.baseline is None:
            self.baseline = random_baseline(
                self.cfg.board_size, self.cfg.eval_episodes, self.cfg.eval_envs,
                seed=CONTRATO["eval_seed"],
            )
        return self.baseline

    # ------------------------------------------------------------- checkpoint
    def _caminho(self, tag, ext):
        return os.path.join(self.cfg.ckpt_dir, f"{self.algo}_{tag}.{ext}")

    def modelos_extra(self):
        """Modelos/camadas além de `self.model` sem os quais a execução não se reproduz.

        O `salvar()` grava `self.model`, o que basta para quem joga com uma rede só. O
        DreamerV3 não é assim: `self.model` é o **ator**, e um ator sem o modelo do mundo
        não joga nada — a pasta da execução guardava um `.keras` que não reproduz o número
        da curva, e `retomar()` voltava com o RSSM aleatório enquanto o `global_step`
        continuava contando. Ver `docs/REVISAO_ALGORITMOS.md` §1.4.

        Devolve `{nome: modelo}`. Os pesos vão para um `.npz` ao lado do `.keras`, em vez
        de um `.keras` por peça: preserva a identidade dos objetos, e portanto as
        `tf.function` já traçadas que os capturaram.
        """
        return {}

    def _pesos_extra(self):
        return {f"{nome}/{i}": np.asarray(v)
                for nome, m in self.modelos_extra().items()
                for i, v in enumerate(m.weights)}

    def _salvar_extra(self, tag):
        pesos = self._pesos_extra()
        if pesos:
            np.savez(self._caminho(tag, "npz"), **pesos)

    def _carregar_extra(self, tag):
        """Devolve `True` se havia pesos extras para carregar."""
        caminho = self._caminho(tag, "npz")
        extras = self.modelos_extra()
        if not extras or not os.path.exists(caminho):
            return False
        with np.load(caminho) as dados:
            for nome, m in extras.items():
                for i, v in enumerate(m.weights):
                    chave = f"{nome}/{i}"
                    if chave in dados:
                        v.assign(dados[chave])
        return True

    def salvar(self, tag="last"):
        self.model.save(self._caminho(tag, "keras"))
        self._salvar_extra(tag)
        estado = {
            "global_step": self.global_step, "episodes": self.episodes,
            "iteration": self.iteration, "history": self.history,
            "evals": self.evals, "baseline": self.baseline, "melhor": self.melhor,
            "config": asdict(self.cfg), "variant": self.variant,
        }
        with open(self._caminho(tag, "json"), "w", encoding="utf-8") as f:
            json.dump(estado, f, ensure_ascii=False)

    def retomar(self, tag="last"):
        """Retoma do checkpoint. O Colab derruba a sessão — é questão de quando."""
        import keras

        m, s = self._caminho(tag, "keras"), self._caminho(tag, "json")
        if not (os.path.exists(m) and os.path.exists(s)):
            return False
        self.model = keras.models.load_model(m)
        self._carregar_extra(tag)
        self.on_model_reloaded()
        with open(s, encoding="utf-8") as f:
            estado = json.load(f)
        self.global_step = estado["global_step"]
        self.episodes = estado["episodes"]
        self.iteration = estado["iteration"]
        self.history = estado["history"]
        self.evals = estado.get("evals", [])
        self.baseline = estado.get("baseline")
        self.melhor = estado.get("melhor", -np.inf)
        self._proximo_eval = proximo_multiplo(self.global_step,
                                              self.cfg.eval_every_steps)
        self._proximo_log = self.global_step
        return True

    @staticmethod
    def _com_sufixo(variant, sufixo):
        """Acrescenta `sufixo` à variante, sem duplicar quando ela já o traz."""
        if not sufixo:
            return variant
        sufixo = sufixo if sufixo.startswith("_") else f"_{sufixo}"
        return variant if variant.endswith(sufixo) else variant + sufixo

    def on_model_reloaded(self):
        """Gancho: o otimizador antigo aponta para as variáveis do modelo antigo."""

    # ------------------------------------------------------------------ treino
    def iterate(self):
        raise NotImplementedError

    def train(self, verbose=True, ate_passos=None):
        """Roda até o orçamento, avaliando na cadência oficial. Devolve o `RunRecord`."""
        alvo = ate_passos or self.cfg.total_steps
        # O `env_spec` descreve o ambiente que **de fato** rodou, não o contrato: uma
        # execução com `canal_fome=True` gravava `n_channels: 5` no registro, e o
        # arquivo mentia sobre a própria observação. Ele continua idêntico ao contrato
        # em qualquer execução de 5 canais, que é o caso normal.
        env_spec = dict(CONTRATO)
        canais = getattr(getattr(self, "env", None), "n_channels", None)
        if canais:
            env_spec["n_channels"] = int(canais)

        rec = Recorder(self.algo, variant=self.variant, seed=self.cfg.seed,
                       net=self.cfg.net,
                       params=self.model.count_params() if self.model else 0,
                       config=asdict(self.cfg), env_spec=env_spec,
                       root=self.cfg.runs_dir)
        self.piso()

        while self.global_step < alvo:
            stats = self.iterate()
            self.iteration += 1
            # Quantos passos de gradiente o orçamento de ambiente comprou. Fica no
            # metadado porque é o eixo do §2.1 da revisão e não dá para reconstruir do
            # `config` — o early-stop por KL corta épocas. Zero significa "o agente não
            # reporta", não "não atualizou".
            self._atualizacoes += int(stats.get("atualizacoes", 0) or 0)

            # Quem chama `registra_fim` no laço de coleta já contabilizou os episódios com
            # a causa da morte junto; registrar de novo aqui contaria cada um duas vezes e
            # a média móvel ficaria certa por acidente, mas as frações, não.
            m, k = stats.get("train_score_mean"), stats.get("n_episodes") or 0
            if not self._registrou_causas and m is not None and k:
                self._registra_episodios(m, k)

            if self.global_step >= self._proximo_log:
                self._proximo_log = self.global_step + self.cfg.log_every_steps
                # a curva registra a **média móvel**, não a iteração isolada: é o número
                # que responde "o treino está andando?" sem depender de quantos episódios
                # acabaram no exato momento do log
                bloco = self.resumo_bloco()
                ponto = {"episodes": self.episodes,
                         "train_score_mean": self.media_movel(),
                         "train_score_iter": stats.get("train_score_mean"),
                         **{k: v for k, v in stats.items() if k != "train_score_mean"},
                         **self.resumo_janela(), **bloco}
                self.history.append({"global_step": self.global_step, **ponto})
                rec.log(self.global_step, **ponto)
                if verbose:
                    self._imprimir(stats, bloco)
                self._marcar_bloco()

            if self.global_step >= self._proximo_eval:
                self._proximo_eval = proximo_multiplo(self.global_step,
                                                      self.cfg.eval_every_steps)
                av = self.avaliar()
                av["global_step"] = self.global_step
                av["episodes"] = self.episodes
                self.evals.append(av)
                rec.log(self.global_step, eval_score_mean=av["score_mean"],
                        eval_score_p95=av["score_p95"], episodes=self.episodes)
                if verbose:
                    print(f"  [eval] passo {self.global_step:,} · "
                          f"score {av['score_mean']:.2f} "
                          f"(piso {self.baseline:.2f})")
                if av["score_mean"] > self.melhor:
                    self.melhor = av["score_mean"]
                    self.salvar("best")
                self.salvar("last")

        final = self.avaliar()
        rec.log(self.global_step, eval_score_mean=final["score_mean"],
                eval_score_p95=final["score_p95"], episodes=self.episodes)

        # O melhor checkpoint é medido com o **mesmo** protocolo, e não reaproveita o
        # número da avaliação periódica: aquele veio de outra amostra, e comparar duas
        # medições ruidosas favorece sistematicamente quem foi medido mais vezes.
        melhor = self.avaliar_melhor(verbose=verbose)
        rec.finish(final, melhor_stats=melhor,
                   comparable=getattr(self.cfg, "comparable", True),
                   caveat=getattr(self.cfg, "caveat", ""))
        rec.record.meta["baseline"] = self.baseline
        # Onde este número foi produzido. Uma curva do Kaggle e outra do Colab são
        # comparáveis — o contrato garante isso — mas o **tempo de parede** não é, e
        # `wall_s_total` é lido com frequência como se fosse.
        rec.record.meta.update(resumo_plataforma())
        # Quantos canais a rede realmente viu. Fica no metadado porque é a diferença que
        # torna uma curva incomparável com outra, e "comparable=False + caveat em prosa"
        # não é conferível por máquina — este número é.
        if getattr(self, "env", None) is not None:
            rec.record.meta["obs_channels"] = int(
                getattr(self.env, "n_channels", CONTRATO["n_channels"]))
        if self._atualizacoes:
            rec.record.meta["atualizacoes"] = int(self._atualizacoes)
        self.salvar("last")

        # O registro é gravado SEMPRE. Estourar no fim de um treino de horas e perder a
        # curva seria o pior desfecho possível; o portão do contrato age na hora de
        # montar a arena, não na hora de escrever. As violações ficam no metadado e
        # `RunRecord.oficial` passa a ser False.
        problemas = validate(rec.record)
        if problemas:
            rec.record.meta["contract_violations"] = problemas
            if verbose:
                print("\n[contrato] esta execução NÃO entra na arena:")
                for p in problemas:
                    print(f"  - {p}")
        caminho = rec.save(skip_validation=True)
        if verbose:
            print(f"[registro] {caminho}")

        self.artefatos(rec, verbose=verbose)
        return rec

    # ---------------------------------------------------------------- artefatos
    def modelo_melhor(self):
        """O modelo do checkpoint `best`, ou `None` se ele não existe.

        Carrega numa instância separada de propósito: `self.model` continua sendo o do
        último passo, porque é ele que define a curva e o número oficial. Trocar em
        silêncio faria a última avaliação medir uma coisa e a curva outra.
        """
        import keras

        caminho = self._caminho("best", "keras")
        if not os.path.exists(caminho):
            return None
        return keras.models.load_model(caminho)

    @contextlib.contextmanager
    def politica_de_checkpoint(self, tag="best"):
        """Uma política que joga pelo checkpoint `tag`, válida dentro do bloco.

        `None` quando o checkpoint não existe. É um gerenciador de contexto porque há
        agentes — o DreamerV3 — que só conseguem jogar um checkpoint **trocando os pesos
        dos próprios submodelos**, e nesse caso a restauração precisa acontecer mesmo se a
        avaliação levantar.
        """
        m = self.modelo_melhor() if tag == "best" else None
        yield None if m is None else self.politica_do_modelo(m)

    def avaliar_melhor(self, verbose=True):
        """Roda o protocolo oficial sobre o melhor checkpoint. `{}` se não houver.

        Avalia **pelo modelo carregado**, sem tocar em `self.model`: a troca de atributo
        era silenciosamente ineficaz em dois agentes (ver `politica_do_modelo`).
        """
        try:
            with self.politica_de_checkpoint("best") as pol:
                if pol is None:
                    return {}
                stats = self.avaliar(politica=pol)
        except NotImplementedError as e:
            if verbose:
                print(f"  [melhor] não avaliado: {e}")
            return {"indisponivel": str(e),
                    "global_step": int(self._passo_do_melhor())}
        stats["global_step"] = int(self._passo_do_melhor())
        if verbose:
            print(f"  [melhor] checkpoint do passo {stats['global_step']:,} · "
                  f"score {stats['score_mean']:.2f} "
                  f"(último: {self.evals[-1]['score_mean']:.2f})"
                  if self.evals else
                  f"  [melhor] score {stats['score_mean']:.2f}")
        return stats

    def _passo_do_melhor(self):
        caminho = self._caminho("best", "json")
        if os.path.exists(caminho):
            with open(caminho, encoding="utf-8") as f:
                return json.load(f).get("global_step", 0)
        return 0

    def copiar_modelos(self, destino, verbose=True):
        """Leva `last.keras` e `best.keras` para dentro da pasta da execução.

        Os checkpoints vivem em `ckpt_dir`, que é compartilhado e sobrescrito pela
        execução seguinte. Sem esta cópia, o `history.json` afirma um score que ninguém
        consegue reproduzir nem inspecionar depois — e o GIF vira a única evidência de
        como o agente jogava.

        Os dois, e não só o melhor: `last` é o modelo que produziu o número **oficial**,
        então é ele que permite reconferir a curva; `best` é o que se leva para o jogo.
        """
        import shutil

        pasta = os.path.join(destino, "modelos")
        os.makedirs(pasta, exist_ok=True)
        copiados = {}
        for tag in ("last", "best"):
            # o `.npz` acompanha o `.keras`: para o DreamerV3 é ele que carrega o modelo
            # do mundo, e sem ele a pasta guarda um ator que não joga (§1.4 da revisão)
            for ext in ("keras", "npz"):
                origem = self._caminho(tag, ext)
                if os.path.exists(origem):
                    alvo = os.path.join(pasta, f"{tag}.{ext}")
                    shutil.copyfile(origem, alvo)
                    copiados[tag if ext == "keras" else f"{tag}+pesos"] = alvo
        if verbose and copiados:
            mb = sum(os.path.getsize(c) for c in copiados.values()) / 1e6
            print(f"  [modelos] {', '.join(sorted(copiados))} em {pasta} ({mb:.1f} MB)")
        return copiados

    def artefatos(self, rec, verbose=True):
        """Gráfico, GIFs e os modelos — tudo ao lado do `history.json`.

        A pasta da execução tem que ser autossuficiente: quem a recebe consegue ver a
        curva, ver o agente jogando e **rodar o modelo**, sem depender de nenhum estado
        que ficou na máquina de quem treinou.
        """
        import os

        destino = os.path.dirname(rec.save(skip_validation=True))
        saida = {}
        saida["modelos"] = self.copiar_modelos(destino, verbose=verbose)

        if self.cfg.salvar_grafico:
            try:
                import matplotlib
                matplotlib.use("Agg")

                fig, _ = plot_run(rec.record)
                caminho = os.path.join(destino, "curva.png")
                fig.savefig(caminho, dpi=150, facecolor=fig.get_facecolor())
                matplotlib.pyplot.close(fig)
                saida["grafico"] = caminho
            except Exception as e:                      # nunca derrubar o treino por isso
                saida["grafico_erro"] = repr(e)

        if self.cfg.salvar_gif:

            politica = self.politica()
            for seed in self.cfg.gif_seeds:
                try:
                    caminho, score, motivo = render_episode(
                        politica, caminho=os.path.join(destino, f"episodio_s{seed}.gif"),
                        board_size=self.cfg.board_size, seed=seed,
                        canal_fome=getattr(self.env, "canal_fome", False),
                    )
                    saida[f"gif_s{seed}"] = {"caminho": caminho, "score": score,
                                             "fim": motivo}
                    if verbose:
                        print(f"[gif] seed {seed}: score {score}, terminou por {motivo}")
                except Exception as e:
                    saida[f"gif_s{seed}_erro"] = repr(e)

        rec.record.meta["artefatos"] = saida
        rec.save(skip_validation=True)
        return saida

    #: Legenda impressa uma vez, antes da primeira linha de log.
    LEGENDA = ("[log] cada métrica sai como  janela | bloco  — a média móvel dos últimos "
               "~{janela} episódios\n"
               "      à esquerda, e só os episódios encerrados desde o log anterior à "
               "direita.\n"
               "      Elas discordam por muitos logs antes de a móvel virar: a da "
               "esquerda diz onde o\n"
               "      agente está, a da direita diz para onde ele está indo.")

    @staticmethod
    def _par(janela, bloco, fmt, largura):
        """`janela | bloco` no mesmo formato, com `—` quando o bloco está vazio."""
        esq = f"{janela:{fmt}}" if janela is not None else "—"
        dir_ = f"{bloco:{fmt}}" if bloco is not None else "—"
        return f"{esq:>{largura}}|{dir_:<{largura}}"

    def _imprimir(self, stats, bloco=None):
        """Uma linha por log: a janela móvel e o bloco novo, lado a lado.

        O score sozinho é ambíguo: 1,2 pontos pode ser "bate em tudo" ou "anda em círculo
        até morrer de fome", e a curva fica igual nos dois casos. Por isso a linha traz a
        **repartição das causas de fim**, que separa os dois de imediato, mais o
        comprimento médio do episódio, que é o sinal mais precoce de todos — uma cobra que
        aprende a sobreviver alonga os episódios antes de o score subir.

        E cada uma dessas medidas aparece **duas vezes**: sobre a janela de
        `JANELA_EPISODIOS` e sobre os episódios encerrados desde o log anterior. A média
        móvel existe para o número não pular com uma amostra de 3 episódios, mas o preço é
        atraso — com ~80 episódios por log ela carrega seis logs de passado. Numa
        degradação em curso as duas colunas discordam bem antes de a curva virar, e é
        exatamente essa discordância que se quer ver.
        """
        if not self._legenda_impressa:
            print(self.LEGENDA.format(janela=self.JANELA_EPISODIOS))
            self._legenda_impressa = True
        r = self.resumo_janela()
        b = bloco if bloco is not None else self.resumo_bloco()
        n_bloco = b.get("bloco_episodios", 0)
        partes = [
            f"passo {self.global_step:>10,}",
            f"ep {self.episodes:>7,} +{n_bloco:<4}",
            "score " + self._par(self.media_movel(), b.get("bloco_train_score_mean"),
                                 ".2f", 6),
        ]
        if "win_rate" in r:
            partes += [
                "fome " + self._par(r["frac_fome"], b.get("bloco_frac_fome"), ".1%", 6),
                "colisão " + self._par(r["frac_colisao"], b.get("bloco_frac_colisao"),
                                       ".1%", 6),
                "vit " + self._par(r["win_rate"], b.get("bloco_win_rate"), ".1%", 6),
                self._par(r["passos_por_episodio"], b.get("bloco_passos_por_episodio"),
                          ".0f", 4) + " passos/ep",
            ]
        partes.append(f"janela {self.episodios_na_janela()}")
        print(" · ".join(partes))


# --- snakeai/search/dinamica.py ---
"""A dinâmica que a árvore de busca percorre.

Existe para que **um só MCTS** sirva a dois mundos:

* `DinamicaReal` — o `VecSnake`. Exata, gratuita, e é o que faz o AlphaZero fazer sentido
  em Snake.
* `DinamicaAprendida` — a rede de dinâmica do MuZero. Aproximada, cara de treinar, e
  necessária só quando o simulador **não** está disponível durante a busca.

Ter as duas atrás da mesma interface é o que torna a comparação honesta: a diferença entre
AlphaZero e MuZero neste repositório passa a ser exatamente *o que a árvore percorre*, com
o algoritmo de busca, o PUCT e o backup literalmente idênticos.
"""


import numpy as np


__all__ = ["DinamicaReal", "DinamicaAprendida"]


class DinamicaReal:
    """Um passo do jogo de verdade, em lote, a partir de estados arbitrários."""

    usa_mascara = True

    def __init__(self, board_size=10, starve_base=None):
        self.board_size = int(board_size)
        self.starve_base = starve_base
        self._env = None

    def _ambiente(self, n):
        if self._env is None or self._env.n != n:
            self._env = VecSnake(n, self.board_size, starve_base=self.starve_base,
                                 rng=np.random.default_rng(0))
        return self._env

    def passo(self, estados, acoes):
        """`(novos_estados, obs, mask, recompensa, terminal)`.

        Os estados são dicionários de arrays do `VecSnake.get_state()`.
        """
        env = self._ambiente(len(acoes))
        env.set_state(estados)
        obs, mask, rew, done, _ = env.step(np.asarray(acoes, dtype=np.int32))
        return env.get_state(), obs, mask, rew, done

    @staticmethod
    def empilhar(estados_por_arvore):
        return {c: np.stack([e[c] for e in estados_por_arvore])
                for c in estados_por_arvore[0]}

    @staticmethod
    def fatiar(estados, i):
        return {c: estados[c][i].copy() for c in estados}


class DinamicaAprendida:
    """A rede de dinâmica do MuZero: `(estado_oculto, ação) → (estado', recompensa)`.

    Diferenças que importam em relação à dinâmica real, e que estão aqui de propósito para
    ficarem visíveis:

    * **Não há terminação.** O modelo não prevê fim de episódio; ele aprende que morrer
      rende `−1` e segue rolando. É como o MuZero original trata o assunto.
    * **Não há máscara dentro da árvore.** A máscara vale na raiz, onde o estado é real. Da
      raiz para baixo o estado é uma abstração aprendida, e não existe "ação ilegal" nela —
      o modelo tem que aprender sozinho que certas ações rendem `−1`.

    O estado é o tensor oculto `(N, B, B, largura)`.
    """

    usa_mascara = False

    def __init__(self, fn_dinamica):
        self.fn = fn_dinamica

    def passo(self, estados, acoes):
        novo, recompensa = self.fn(estados, np.asarray(acoes, dtype=np.int32))
        n = len(acoes)
        mask = np.ones((n, N_ACTIONS), dtype=bool)
        done = np.zeros(n, dtype=bool)
        # a "observação" de um nó interno **é** o estado oculto: a rede de predição lê dele
        return novo, novo, mask, np.asarray(recompensa, dtype=np.float32), done

    @staticmethod
    def empilhar(estados_por_arvore):
        return np.stack(estados_por_arvore)

    @staticmethod
    def fatiar(estados, i):
        return estados[i].copy()


# --- snakeai/search/mcts.py ---
"""MCTS com PUCT sobre o simulador **real**.

Por que isto é a jogada certa em Snake
---------------------------------------
MuZero e EfficientZero gastam a maior parte da complexidade deles aprendendo um modelo do
mundo — porque em Atari o simulador não está disponível durante a busca. Aqui está: Snake é
determinístico, de informação perfeita, tem 3 ações, e o `VecSnake` faz ~286 mil passos por
segundo. Aprender um modelo do que já se pode simular exatamente seria pagar caro por uma
aproximação pior.

Então este módulo faz busca em árvore com o jogo de verdade. É o AlphaZero sem a parte de
adivinhar a física.

Como a busca fica em lote
-------------------------
MCTS é naturalmente sequencial, e uma avaliação de rede por simulação com lote 1 seria
lentíssimo na GPU. O truque: rodar **N árvores independentes em paralelo**, uma por
ambiente, e sincronizá-las por número de simulação. Na simulação `k`, as N árvores estão
todas esperando avaliar um nó — e aí a rede recebe um lote de N. A busca custa
`num_simulations` chamadas de rede, não `N × num_simulations`.

O estado de cada nó é o dicionário de `VecSnake.get_state()`. Restaurar um nó é escrever
esses arrays de volta num ambiente de busca descartável — barato, e exato.
"""


import numpy as np


__all__ = ["No", "MCTS", "MinMax"]


class No:
    """Um nó da árvore. Guarda o estado do jogo e as estatísticas do PUCT."""

    __slots__ = ("estado", "prior", "visitas", "soma_valor", "filhos", "recompensa",
                 "terminal", "mask", "expandido")

    def __init__(self, prior=0.0):
        self.prior = float(prior)
        self.visitas = 0
        self.soma_valor = 0.0
        self.filhos = {}
        self.estado = None
        self.recompensa = 0.0
        self.terminal = False
        self.mask = None
        self.expandido = False

    @property
    def valor(self):
        return self.soma_valor / self.visitas if self.visitas else 0.0


class MinMax:
    """Faixa `[min, max]` dos Q vistos numa árvore, para normalizar o PUCT.

    Por que existe (e por que o AlphaZero original não precisa dela): em Xadrez e Go o
    valor é uma probabilidade de vitória em `[-1, 1]`, e `c_puct` foi calibrado nessa
    escala. Aqui a recompensa é `+1` por maçã e a cabeça de valor é linear, então o valor
    aprendido é positivo e cresce com o agente — a execução de 5 M passos mede `valor_raiz`
    indo de 0,26 a **3,5**.

    Nessa escala o termo de exploração (`c_puct · P · √N`, ou `8,6 · P` na raiz com 32
    simulações) só vence quando o prior é alto: um filho **não visitado**, cujo Q vale 0
    por convenção, ganha na raiz com `P = 0,7` (bônus 6,0 contra 3,5) e perde com
    `P = 0,15` (bônus 1,3). Descendo, `√N` encolhe e ele perde sempre. O resultado não é
    uma busca parada — é uma busca que só confirma o que a rede já achava, e portanto
    deixa de ser operador de melhoria de política.

    A normalização é a do MuZero (Schrittwieser et al., 2020, Apêndice B): cada Q é
    mapeado para `[0, 1]` pela faixa observada **naquela árvore**, o que devolve `c_puct`
    à escala em que ele foi calibrado. Ver `docs/BUSCA_DEGENERADA.md`.
    """

    __slots__ = ("minimo", "maximo")

    def __init__(self):
        self.minimo, self.maximo = np.inf, -np.inf

    def atualiza(self, valor):
        v = float(valor)
        if v < self.minimo:
            self.minimo = v
        if v > self.maximo:
            self.maximo = v

    def normaliza(self, valor):
        if self.maximo > self.minimo:
            return (float(valor) - self.minimo) / (self.maximo - self.minimo)
        return float(valor)


class MCTS:
    """Busca em árvore com PUCT, em lote sobre N árvores.

    Parâmetros
    ----------
    avaliar : callable
        `avaliar(obs, mask) -> (priors, valores)`, com `priors` já normalizado e
        mascarado. É a rede; o MCTS não conhece Keras.
    board_size, gamma : contrato do ambiente.
    num_simulations : int
        Orçamento de busca por jogada. É o botão que troca computação por qualidade.
    c_puct : float
        Peso da exploração no PUCT. Maior = confia mais no prior, explora mais largo.
    dirichlet_alpha, dirichlet_frac : float
        Ruído na raiz, só durante a coleta. Sem ele a busca fica determinística e o agente
        nunca descobre uma jogada que a rede ainda não gosta — é o análogo do ε-greedy.
    """

    def __init__(self, avaliar, board_size=10, gamma=0.997, num_simulations=32,
                 c_puct=1.5, dirichlet_alpha=0.5, dirichlet_frac=0.25, rng=None,
                 starve_base=None, dinamica=None, fpu="zero", q_normalizado=False,
                 desempate="ordem"):
        self.avaliar = avaliar
        self.board_size = int(board_size)
        #: O ambiente de busca TEM que ser configurado igual ao de treino. Se o
        #: `starve_base` diferir, a árvore simula um jogo com outra regra de fome — e
        #: planeja sobre um mundo que não é o que o agente vai jogar. Não levanta erro:
        #: só produz decisões ligeiramente erradas, o tempo todo.
        self.starve_base = starve_base
        #: O que a árvore percorre. Trocar isto — e só isto — é a diferença entre
        #: AlphaZero e MuZero neste repositório.
        self.dinamica = dinamica or DinamicaReal(board_size, starve_base)
        self.gamma = float(gamma)
        self.num_simulations = int(num_simulations)
        self.c_puct = float(c_puct)
        self.dirichlet_alpha = float(dirichlet_alpha)
        self.dirichlet_frac = float(dirichlet_frac)
        self.rng = rng if rng is not None else np.random.default_rng(0)
        #: O que vale o Q de um filho **ainda não visitado** — o *first play urgency*.
        #: `"zero"` é a convenção do AlphaZero e o padrão histórico deste arquivo;
        #: `"pai"` usa o valor do próprio nó, que é o palpite honesto quando não se
        #: mediu nada ainda. Num jogo de valor estritamente positivo a diferença não é
        #: cosmética: com `"zero"` o filho novo nasce ~V abaixo dos irmãos e a busca
        #: nunca o toca. Ver `docs/BUSCA_DEGENERADA.md`.
        if fpu not in ("zero", "pai"):
            raise ValueError(f"fpu desconhecido: {fpu!r} (use 'zero' ou 'pai')")
        self.fpu = fpu
        #: Normalização min-max do Q dentro da árvore (MuZero, Apêndice B). Ver `MinMax`.
        self.q_normalizado = bool(q_normalizado)
        #: Como resolver empate exato de pontuação no PUCT. `"ordem"` fica com o primeiro
        #: filho do dicionário, que é a primeira ação **liberada pela máscara** —
        #: `np.nonzero` crescente, ou seja, *virar à esquerda*. Não é raro: na primeira
        #: descida de cada nó todos os filhos têm o mesmo Q (nenhum foi visitado) e, com
        #: prior uniforme, o mesmo `u`. O viés é sistemático e sempre para o mesmo lado.
        #: `"aleatorio"` sorteia entre os empatados, o que troca um viés por ruído.
        if desempate not in ("ordem", "aleatorio"):
            raise ValueError(f"desempate desconhecido: {desempate!r}")
        self.desempate = desempate
        self._sim = None      # ambiente de busca, criado sob demanda no tamanho certo
        self._ultimas_raizes = []

    # ---------------------------------------------------------------- ambiente
    def _ambiente(self, n):
        """Compatibilidade: expõe o `VecSnake` da dinâmica real, quando houver."""
        return self.dinamica._ambiente(n)

    def _expandir(self, nos, priors, valores):
        for no, p in zip(nos, priors):
            if no is None:
                continue
            no.expandido = True
            permitidas = np.nonzero(no.mask)[0] if no.mask is not None else range(N_ACTIONS)
            for a in permitidas:
                no.filhos[int(a)] = No(prior=float(p[a]))

    def _q_virgem(self, no, mm):
        """O Q atribuído a um filho que ainda não foi visitado — o *first play urgency*.

        `"zero"` é a convenção do AlphaZero. Sob normalização ele **não** é normalizado de
        novo: `0` já é, por construção, o piso da faixa, que é exatamente o que o MuZero
        faz. Normalizá-lo o jogaria muito abaixo do pior filho medido — a patologia que a
        normalização existe para remover.

        `"pai"` usa o valor do próprio nó. Sob normalização há uma sutileza que custou uma
        revisão para aparecer: o `MinMax` é alimentado com **Q** (`r + γ·V`) e `no.valor` é
        um **V** — mais o valor do nó continua se movendo depois de a faixa registrar o
        dele. Quando a faixa ainda é estreita, uma diferença absoluta minúscula vira um
        número normalizado grande: medido, 9,1% dos FPU saíam acima de 1 e chegavam a
        **+5,15**, o que faz o filho virgem ganhar de todos os irmãos incondicionalmente e
        a busca abrir filhos novos em vez de aprofundar. Prender em `[0, 1]` devolve o
        significado honesto: um filho não medido vale, no máximo, o melhor irmão medido, e
        no mínimo o pior.
        """
        if self.fpu != "pai":
            return 0.0
        if mm is None:
            return no.valor
        return min(1.0, max(0.0, mm.normaliza(no.valor)))

    def _selecionar(self, no, mm=None):
        """PUCT: `Q(s,a) + c · P · √N / (1 + n)`, com `Q(s,a) = r + γ·V(filho)`.

        A recompensa de **chegar** ao filho tem que entrar no Q — e é fácil esquecer,
        porque o nó guarda o valor do estado dele, não do movimento. Sem esse termo, um
        filho alcançado morrendo tem valor 0 (episódio acabou, sem futuro) e parece tão
        atraente quanto um filho seguro: a busca escolhe a morte e fica **pior que
        aleatória**. Foi exatamente o que aconteceu na primeira versão deste arquivo.

        A segunda armadilha mora na outra ponta da mesma linha: o `Q` de um filho **ainda
        não visitado**. `0` é a convenção do AlphaZero e está certa onde o valor é uma
        `tanh` em `[-1, 1]` centrada em zero. Neste jogo o valor aprendido é positivo
        (medido: 3,5 ao fim de 5 M passos) e o bônus de exploração é `c_puct · P · √N`,
        que só cobre essa diferença quando o prior já é alto. Uma ação de que a rede não
        gosta nunca é experimentada fundo o bastante para a busca discordar dela. `fpu` e
        `q_normalizado` existem para isso; ambos nascem desligados. Ver
        `docs/BUSCA_DEGENERADA.md`.
        """
        if not no.filhos:
            return None
        raiz_n = np.sqrt(max(no.visitas, 1))
        melhor, melhor_pont = None, -np.inf
        virgem = self._q_virgem(no, mm)
        empatados = []
        for a, filho in no.filhos.items():
            u = self.c_puct * filho.prior * raiz_n / (1 + filho.visitas)
            if filho.visitas:
                q = filho.recompensa + self.gamma * filho.valor
                if mm is not None:
                    q = mm.normaliza(q)
            else:
                q = virgem
            pont = q + u
            # `>` estrito e igualdade exata: assim o caminho de `desempate="ordem"` fica
            # bit a bit igual ao de antes desta flag existir. Uma banda de tolerância
            # mudaria a escolha em empates *quase* exatos, e a execução de controle está
            # rodando com o comportamento antigo.
            if pont > melhor_pont:
                melhor, melhor_pont, empatados = a, pont, [a]
            elif pont == melhor_pont:
                empatados.append(a)
        if self.desempate == "aleatorio" and len(empatados) > 1:
            # `integers` em vez de `choice`: este é o laço mais quente da busca (chamado
            # ~`num_simulations × N × profundidade` vezes por jogada) e `choice` valida e
            # converte a lista a cada chamada
            return empatados[int(self.rng.integers(len(empatados)))]
        return melhor

    # -------------------------------------------------------------------- busca
    def run(self, estado_raiz, mask_raiz, obs_raiz, adicionar_ruido=False):
        """Roda a busca a partir de N estados. Devolve `(visitas, valores_raiz)`.

        `visitas` é `(N, 3)` — a contagem de visitas por ação, que é a política melhorada
        pela busca. `valores_raiz` é `(N,)`, o valor que a busca atribuiu à posição.
        """
        n = mask_raiz.shape[0]
        din = self.dinamica

        raizes = [No() for _ in range(n)]
        estatisticas = [MinMax() if self.q_normalizado else None for _ in range(n)]
        for i, r in enumerate(raizes):
            r.estado = din.fatiar(estado_raiz, i)
            r.mask = mask_raiz[i]

        priors, valores = self.avaliar(obs_raiz, mask_raiz)
        priors = np.asarray(priors, dtype=np.float64)
        if adicionar_ruido:
            ruido = self.rng.dirichlet([self.dirichlet_alpha] * N_ACTIONS, size=n)
            priors = (1 - self.dirichlet_frac) * priors + self.dirichlet_frac * ruido
            priors = np.where(mask_raiz, priors, 0.0)
            priors /= np.maximum(priors.sum(1, keepdims=True), 1e-12)
        self._expandir(raizes, priors, valores)
        for r, v in zip(raizes, np.asarray(valores).ravel()):
            r.visitas, r.soma_valor = 1, float(v)

        for _ in range(self.num_simulations):
            caminhos = []          # (lista de nós, ação escolhida) por árvore
            pais, acoes = [], []
            for i, raiz in enumerate(raizes):
                mm = estatisticas[i]
                no, caminho = raiz, [raiz]
                a = self._selecionar(no, mm)
                while a is not None and no.filhos[a].expandido and not no.filhos[a].terminal:
                    no = no.filhos[a]
                    caminho.append(no)
                    a = self._selecionar(no, mm)
                caminhos.append((caminho, a))
                pais.append(no)
                acoes.append(a if a is not None else 1)

            # --- um passo da dinâmica, em lote, a partir dos N pais
            estados = din.empilhar([p.estado for p in pais])
            estados_filhos, obs, mask, rew, done = din.passo(estados, acoes)

            folhas, a_avaliar_obs, a_avaliar_mask, idx_avaliar = [], [], [], []
            for i, ((caminho, a), pai) in enumerate(zip(caminhos, pais)):
                if a is None:
                    folhas.append(None)
                    continue
                filho = pai.filhos[a]
                filho.recompensa = float(rew[i])
                filho.terminal = bool(done[i])
                folhas.append(filho)
                if filho.terminal:
                    # ARMADILHA: o `VecSnake` reseta sozinho ao terminar, então `obs[i]` e
                    # `estados_filhos[i]` já são de um episódio NOVO. Guardá-los aqui
                    # plantaria uma partida aleatória dentro da árvore. Um nó terminal não
                    # precisa de estado — vale 0 e nunca é expandido.
                    continue
                filho.estado = din.fatiar(estados_filhos, i)
                filho.mask = mask[i] if din.usa_mascara else None
                a_avaliar_obs.append(obs[i])
                a_avaliar_mask.append(mask[i])
                idx_avaliar.append(i)

            valores_folha = np.zeros(n, dtype=np.float64)
            if idx_avaliar:
                p_f, v_f = self.avaliar(np.stack(a_avaliar_obs), np.stack(a_avaliar_mask))
                p_f = np.asarray(p_f, dtype=np.float64)
                v_f = np.asarray(v_f, dtype=np.float64).ravel()
                self._expandir([folhas[i] for i in idx_avaliar], p_f, v_f)
                for k, i in enumerate(idx_avaliar):
                    valores_folha[i] = v_f[k]

            # --- backup
            # `v` é sempre o valor estimado DO nó que está recebendo o crédito. Subindo,
            # ele vira `recompensa_de_entrar_no_nó + γ·v`. Nó terminal vale 0: o episódio
            # acabou, não há retorno futuro nenhum para descontar.
            for i, (caminho, a) in enumerate(caminhos):
                if a is None:
                    continue
                folha, mm = folhas[i], estatisticas[i]
                v = 0.0 if folha.terminal else float(valores_folha[i])
                for no in reversed([*caminho, folha]):
                    no.visitas += 1
                    no.soma_valor += v
                    if mm is not None:
                        mm.atualiza(no.recompensa + self.gamma * no.valor)
                    v = no.recompensa + self.gamma * v

        #: guardado para inspeção em teste — a árvore some ao fim do `run`
        self._ultimas_raizes = raizes

        visitas = np.zeros((n, N_ACTIONS), dtype=np.float64)
        for i, raiz in enumerate(raizes):
            for a, filho in raiz.filhos.items():
                visitas[i, a] = filho.visitas
        valores_raiz = np.array([r.valor for r in raizes], dtype=np.float32)
        return visitas, valores_raiz

    @staticmethod
    def politica_das_visitas(visitas, temperatura=1.0):
        """Converte contagens de visita em distribuição.

        `temperatura → 0` vira argmax (jogo forte); `1` mantém a proporção das visitas
        (bom para explorar e para o alvo de treino).

        `temperatura` pode ser escalar ou um vetor `(N,)` — uma por árvore. O vetor é o que
        o agendamento canônico do AlphaZero exige: τ alto nos primeiros lances **de cada
        episódio** e frio no resto, e os N ambientes de um lote estão em lances diferentes.
        """
        visitas = np.asarray(visitas, dtype=np.float64)
        t = np.asarray(temperatura, dtype=np.float64)
        t = np.full((visitas.shape[0], 1), float(t)) if t.ndim == 0 else t.reshape(-1, 1)
        # `1/t` com t≈0 estoura antes de o `where` escolher o ramo, então o expoente é
        # calculado com um t seguro e o ramo frio entra depois.
        frio = t <= 1e-6
        p = np.power(np.maximum(visitas, 0.0), 1.0 / np.where(frio, 1.0, t))
        p = np.where(frio, (visitas == visitas.max(axis=1, keepdims=True)).astype(np.float64), p)
        soma = p.sum(axis=1, keepdims=True)
        return np.where(soma > 0, p / np.maximum(soma, 1e-12), 1.0 / visitas.shape[1])


# --- snakeai/nets/muzero.py ---
"""As três redes do MuZero.

O MuZero substitui o simulador por três funções aprendidas:

======================  ==============================================================
rede                    o que faz
======================  ==============================================================
**representação** `h`   observação → estado oculto `s₀`
**dinâmica** `g`        `(s, a)` → `(s', recompensa)`
**predição** `f`        `s` → `(política, valor)`
======================  ==============================================================

O detalhe que define o algoritmo: **`s` não precisa significar nada**. Não há perda
pedindo que o estado oculto reconstrua a observação. As três redes são treinadas só para
que a busca produza boas jogadas — o modelo aprende o que é útil para planejar, não o que
é fiel ao mundo. É a diferença entre o MuZero e um model-based clássico, e é por isso que
ele funciona em domínios onde reconstruir pixels seria impossível.

Duas peças pequenas que decidem se treina
------------------------------------------
* **Normalização do estado oculto para [0, 1]** (min-max por amostra). Sem isso a escala do
  estado cresce a cada aplicação de `g` e o desenrolar de `K` passos explode.
* **Escala de gradiente de ½ na dinâmica**, aplicada a cada passo do desenrolar. Sem ela o
  gradiente que chega em `h` cresce com `K` e o treino fica instável.
"""


import os

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops


__all__ = ["normaliza_oculto", "escala_gradiente",
           "build_representacao", "build_dinamica", "build_predicao"]


def normaliza_oculto(x):
    """Min-max por amostra, para [0, 1]. Segura a escala ao longo do desenrolar."""
    minimo = ops.min(x, axis=(1, 2, 3), keepdims=True)
    maximo = ops.max(x, axis=(1, 2, 3), keepdims=True)
    return (x - minimo) / ops.maximum(maximo - minimo, 1e-5)


def escala_gradiente(x, escala):
    """Deixa o valor intacto e multiplica o gradiente por `escala`.

    `x·s + stop_gradient(x·(1−s))` — o truque padrão do MuZero para que o gradiente que
    atravessa `K` aplicações da dinâmica não cresça com `K`.
    """
    return x * escala + ops.stop_gradient(x) * (1.0 - escala)


@keras.saving.register_keras_serializable(package="snakeai")
class NormalizaOculto(layers.Layer):
    """Camada em vez de `Lambda`: `Lambda` não sobrevive a `save`/`load` sem gambiarra."""

    def call(self, x):
        return normaliza_oculto(x)

    def compute_output_shape(self, input_shape):
        return input_shape


def build_representacao(board_size=10, preset="resnet_small", nome="h"):
    """`observação → estado oculto`. O estado oculto tem a forma espacial do tabuleiro."""
    largura, blocos = PRESETS[preset]
    inp = keras.Input(shape=(board_size, board_size, N_CHANNELS), name="board")
    x = layers.Conv2D(largura, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal", name=f"{nome}_c")(inp)
    x = layers.GroupNormalization(groups=8, name=f"{nome}_n")(x)
    x = layers.Activation("relu", name=f"{nome}_a")(x)
    for i in range(blocos):
        x = residual_block(x, largura, f"{nome}_res{i}")
    x = NormalizaOculto(name=f"{nome}_norm")(x)
    return keras.Model(inp, x, name="representacao")


def build_dinamica(board_size=10, preset="resnet_small", n_actions=N_ACTIONS,
                   nome="g", n_suporte=0):
    """`(estado, ação) → (estado', recompensa)`.

    A ação entra como **planos constantes** concatenados ao estado — um plano de uns no
    canal da ação escolhida, zeros nos outros. É o encoding do MuZero: mantém a estrutura
    convolucional e não obriga a rede a aprender um embedding.
    """
    largura, blocos = PRESETS[preset]
    s = keras.Input(shape=(board_size, board_size, largura), name="estado")
    a = keras.Input(shape=(board_size, board_size, n_actions), name="acao_planos")

    x = layers.Concatenate(name=f"{nome}_cat")([s, a])
    x = layers.Conv2D(largura, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal", name=f"{nome}_c")(x)
    x = layers.GroupNormalization(groups=8, name=f"{nome}_n")(x)
    x = layers.Activation("relu", name=f"{nome}_a")(x)
    for i in range(max(1, blocos - 1)):
        x = residual_block(x, largura, f"{nome}_res{i}")
    novo = NormalizaOculto(name=f"{nome}_norm")(x)

    r = layers.Conv2D(2, 1, use_bias=False, name=f"{nome}_rc")(x)
    r = layers.GroupNormalization(groups=2, name=f"{nome}_rn")(r)
    r = layers.Activation("relu", name=f"{nome}_ra")(r)
    r = layers.Flatten(name=f"{nome}_rf")(r)
    r = layers.Dense(64, activation="relu", name=f"{nome}_rd")(r)
    # `n_suporte > 0` troca a cabeça escalar por logits sobre um suporte discreto —
    # o Apêndice F do MuZero. A saída deixa de ser um número e passa a ser uma
    # distribuição; quem lê converte por esperança. Ver §2.33.
    recompensa = layers.Dense(max(1, n_suporte), name="recompensa")(r)

    return keras.Model([s, a], [novo, recompensa], name="dinamica")


def build_predicao(board_size=10, preset="resnet_small", n_actions=N_ACTIONS,
                   nome="f", n_suporte=0):
    """`estado oculto → (logits de política, valor)`."""
    largura, _ = PRESETS[preset]
    s = keras.Input(shape=(board_size, board_size, largura), name="estado")

    p = layers.Conv2D(4, 1, use_bias=False, name=f"{nome}_pc")(s)
    p = layers.GroupNormalization(groups=2, name=f"{nome}_pn")(p)
    p = layers.Activation("relu", name=f"{nome}_pa")(p)
    p = layers.Flatten(name=f"{nome}_pf")(p)
    logits = layers.Dense(
        n_actions, name="logits",
        kernel_initializer=keras.initializers.Orthogonal(gain=0.01))(p)

    v = layers.Conv2D(2, 1, use_bias=False, name=f"{nome}_vc")(s)
    v = layers.GroupNormalization(groups=2, name=f"{nome}_vn")(v)
    v = layers.Activation("relu", name=f"{nome}_va")(v)
    v = layers.Flatten(name=f"{nome}_vf")(v)
    v = layers.Dense(128, activation="relu", name=f"{nome}_vd")(v)
    valor = layers.Dense(max(1, n_suporte), name="valor")(v)

    return keras.Model(s, [logits, valor], name="predicao")


# --- snakeai/agents/muzero.py ---
"""MuZero — a mesma busca, sobre um modelo aprendido.

O contraste com o AlphaZero deste repositório é a razão de ele estar aqui: **o algoritmo de
busca é literalmente o mesmo objeto**, o `MCTS`. Muda só o que a árvore percorre — a
`DinamicaReal` (o `VecSnake`) vira `DinamicaAprendida` (a rede `g`). Toda diferença de
resultado entre os dois é atribuível a isso, e a nada mais.

Vale dizer com todas as letras: **em Snake, o MuZero deveria perder para o AlphaZero.** O
simulador está disponível, é exato e é rápido; trocá-lo por uma aproximação aprendida só
pode piorar a busca. O MuZero existe para domínios onde o simulador *não* está disponível
durante o jogo — e medi-lo aqui é medir quanto custa não ter o simulador. Esse é um número
interessante, e é uma pergunta que o benchmark pode responder justamente por ter os dois
lado a lado, sob o mesmo contrato.

O desenrolar de K passos
------------------------
O treino não olha transições isoladas. Ele parte de uma posição da trajetória, aplica `h`
uma vez, e depois `g` `K` vezes seguindo as ações que foram realmente tomadas. Em cada um
dos `K+1` passos há três perdas:

* **política** ← visitas do MCTS naquele passo,
* **valor** ← retorno de n passos com bootstrap no valor da busca,
* **recompensa** ← a recompensa que o ambiente de fato deu.

A perda de recompensa é a única âncora que liga o estado oculto ao mundo. Sem ela o modelo
pode inventar qualquer dinâmica internamente consistente e a busca vira ficção.

Os consertos que vieram do AlphaZero
------------------------------------
Como o `MCTS` é **o mesmo objeto**, os três defeitos que a primeira execução de 5 M passos
do AlphaZero revelou estavam aqui também, palavra por palavra (§2.27–§2.29 da revisão):

* o PUCT dava `Q = 0` a um filho ainda não visitado. É a convenção do AlphaZero, correta
  onde o valor é uma `tanh` em `[-1, 1]`; aqui a cabeça é linear e o valor aprendido é
  positivo, então o bônus `c_puct·P·√N` só cobre a diferença onde o prior já é alto — a
  busca passa a **confirmar** a rede em vez de discordar dela, que é o oposto de ser um
  operador de melhoria de política. Conserto: `fpu` e `q_normalizado` — este último é,
  ironicamente, a normalização min-max do próprio paper do MuZero (Apêndice B);
* o alvo de valor não é normalizado e domina o tronco compartilhado. Conserto:
  `valor_symlog`, com a busca continuando a ler a escala real;
* a mesma distribuição temperada escolhia a ação **e** virava o alvo de treino. Conserto:
  `temp_alvo` e `temp_passos`.

Mais o orçamento de gradiente, o decaimento de `lr` e o bootstrap do fim da janela. Ao
contrário do AlphaZero, aqui não havia execução de controle a preservar — o MuZero nunca
rodou sob o contrato — então tudo já nasce ligado. Ver `docs/BUSCA_DEGENERADA.md`.

O que a primeira execução mostrou (§2.31)
-----------------------------------------
`unroll5/seed0` terminou em **49,26** com o melhor ponto em **66,05**, oscilando entre 31,7
e 66,0 enquanto o `train_score` — que é o da **busca** — ficava estável em 58–60. O
professor está bom; quem oscila é o aluno. E `perda_pi` sobe no último terço do orçamento
*enquanto o `lr` desce*, o que descarta passo grande demais e aponta para o alvo.

A causa provável é aritmética. `perda_pi` é uma **soma crua** sobre os `K+1` passos: o passo
0, que sai de `f(h(o))` — a observação real, o único caminho que `politica()` usa na
avaliação —, e `K` passos imaginados, que saem de `f(g^k(...))`. Sem peso entre eles, o
passo 0 vale **14,5%** da perda com `unroll=5`, e **11,0%** com `unroll=10`. O pseudocódigo
do paper escala só os passos imaginados por `1/K` (`normaliza_unroll`), o que põe o passo 0
em ~46% qualquer que seja `K`. O repositório já tinha *a outra* escala de gradiente — a de
1/2 no estado oculto, que controla o que chega em `h` — e não esta.

Fica **desligado** por padrão até a medição, com os braços no `92_muzero_ablacoes`.
"""


import os
from dataclasses import dataclass

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
import numpy as np
import tensorflow as tf


__all__ = ["MuZeroConfig", "MuZero"]


@dataclass
class MuZeroConfig(BaseConfig):
    net: str = "resnet_small"
    num_envs: int = 64
    rollout: int = 16

    num_simulations: int = 24
    c_puct: float = 1.5
    #: α ∝ 1/(ações legais); a heurística do paper calibra em ~10/n, que daria 3,3 para
    #: **3** ações. Com 0,5 o ruído punha mais de 90% da massa numa única ação em 15% dos
    #: lances. Ver `docs/BUSCA_DEGENERADA.md`.
    dirichlet_alpha: float = 1.0
    dirichlet_frac: float = 0.25

    # ------------------------------------------------------------------------------
    # Os três consertos do §2.27–§2.29, herdados do AlphaZero. **O `MCTS` é o mesmo
    # objeto**, então os defeitos eram os mesmos — e o MuZero nunca rodou sob o contrato,
    # então aqui eles já nascem ligados, sem execução de controle para preservar.
    # ------------------------------------------------------------------------------

    #: §2.27 — o Q de um filho ainda não visitado. `"zero"` é a convenção do AlphaZero e
    #: está certa onde o valor é uma `tanh` em `[-1, 1]`; aqui a cabeça é linear e o valor
    #: aprendido é positivo, então o bônus `c_puct·P·√N` só cobre a diferença onde o prior
    #: já é alto — a busca passa a confirmar a rede em vez de discordar dela.
    fpu: str = "pai"
    #: §2.27 — normalização min-max do Q dentro da árvore (MuZero, Apêndice B). Devolve
    #: `c_puct` à escala em que foi calibrado. Irônico que faltasse justamente aqui.
    q_normalizado: bool = True
    #: Empate exato no PUCT: `"ordem"` fica sempre com o primeiro filho do dicionário.
    desempate: str = "aleatorio"

    #: §2.28 — treinar o valor em symlog em vez da escala crua. O alvo é um retorno
    #: descontado não normalizado que cresce com o agente; a busca continua recebendo o
    #: valor na escala **real**, porque o backup soma `recompensa + γ·valor`.
    valor_symlog: bool = True

    #: §2.29 — temperatura por lance do episódio e alvo de política sem temperar. Ver as
    #: notas homônimas em `alphazero.py`.
    #:
    #: **Atenção ao qual dos dois agendamentos do paper isto é** (§2.34). O Apêndice D
    #: descreve dois: nos jogos de tabuleiro, τ = 1 nos primeiros `k` lances e frio
    #: depois; em **Atari** — *"a variation of this scheme"* — as ações são amostradas da
    #: contagem de visitas *"throughout the duration of each game, instead of just the
    #: first k moves"*, com τ decaindo por **passo de treino** (1,0 na primeira metade,
    #: 0,5 no terceiro quarto, 0,25 no último). `temp_passos` é o primeiro. Snake tem
    #: episódios de ~1.200 lances, então ele deixa 97,5% de cada episódio em τ = 0,25
    #: desde a primeira iteração — muito menos exploração do que o paper faria em Atari,
    #: num jogo de recompensa esparsa. `temp_esquema` traz o segundo.
    temp_passos: int = 30
    temp_alvo: float = 1.0
    #: Fecha o último passo da janela de coleta, que hoje teria alvo sem bootstrap.
    bootstrap_fim_janela: bool = True

    #: Escala os `K` passos imaginados por `1/K`, deixando o passo 0 inteiro — é o
    #: `scale_gradient(loss, 1/K)` do pseudocódigo do paper. **Desligado é a soma crua**,
    #: e a soma crua tem uma consequência que não é óbvia: as perdas são somas sobre
    #: `K+1` termos sem peso, então a fatia do passo 0 — o único que a métrica oficial
    #: mede, porque `politica()` age sobre a observação real — vale **14,5%** da perda de
    #: política com `unroll=5` e **11,0%** com `unroll=10`. Ligado, ela fica em ~46%
    #: qualquer que seja `K`. Medido em `tools/diag_unroll.py`.
    #:
    #: Ou seja: aumentar o desenrolar sem isto **dilui** justamente o termo que decide o
    #: número do contrato.
    #:
    #: O Apêndice G do paper é explícito em ter **duas** escalas de gradiente: "we scale
    #: the loss of each head by 1/K" e "we scale the gradient at the start of the dynamics
    #: function by 1/2". O repositório tinha a segunda e não a primeira. Vale notar que a
    #: leitura literal da prosa — escalar *todos* os `K+1` termos por `1/K` — não mudaria
    #: nada aqui: sob Adam, dividir a perda inteira por uma constante é quase um no-op (o
    #: segundo momento normaliza), sobrando só um `clipnorm` que morde menos. O que muda a
    #: **fatia** do passo 0 é deixá-lo fora da escala, que é o que o pseudocódigo publicado
    #: faz. Ver §2.31.
    normaliza_unroll: bool = False

    gamma: float = 0.997
    #: 10 é o valor do MuZero **puro** (Apêndice G, Atari). O Apêndice H o baixa para 5,
    #: mas como parte do pacote do Reanalyse — trocar só isto não é seguir o paper.
    n_step: int = 10
    #: Passos do desenrolar no treino. É o que obriga o modelo a ser útil por mais de um
    #: passo à frente — com K=1 ele vira um crítico caro.
    unroll: int = 5

    lr: float = 3e-4
    #: Decaimento linear do `lr` até o fim do orçamento, como no PPO e no ACKTR. `0`
    #: mantém constante.
    lr_final: float = 5e-5
    max_grad_norm: float = 5.0
    batch_size: int = 256
    memory_size: int = 50_000
    #: Com 1, os 5 M passos compravam ~4.900 atualizações contra as ~38.300 do PPO. O
    #: passo de gradiente aqui é caro (o desenrolar de `unroll` passos), então 8 sai ~30%
    #: mais lento por iteração — bem mais que os ~5% do AlphaZero. Ver §2.1.
    #:
    #: **O número que isto produz merece ser dito.** `8 × batch_size 256 = 2048` amostras
    #: de gradiente por iteração contra `num_envs 64 × rollout 16 = 1024` passos novos:
    #: **2,0 amostras por estado**. O paper usa 0,1 no MuZero puro e sobe justamente para
    #: **2,0** no MuZero Reanalyse (Apêndice H) — e o Reanalyse existe porque reúso alto
    #: precisa de alvo fresco: ele **refaz a busca** com a rede atual em 80% das
    #: atualizações e usa rede alvo para o bootstrap de valor. Aqui não há nem um nem
    #: outro. Estamos no regime de reúso do Reanalyse **sem** o Reanalyse, e é a hipótese
    #: mais forte para a oscilação depois da do §2.31. Ver `docs/REVISAO_ALGORITMOS.md`.
    epochs_por_iter: int = 8

    temp_inicio: float = 1.0
    temp_fim: float = 0.25
    temp_frac: float = 0.5
    #: `"lance"` usa `temp_passos` (o esquema de jogo de tabuleiro, o que está aqui hoje).
    #: `"treino"` é o de Atari: amostra durante o episódio inteiro, com τ em degraus por
    #: fração do orçamento. Ver §2.34.
    temp_esquema: str = "lance"

    # ------------------------------------------------------------------------------
    # Reanalyse (Apêndice H). Ver §2.32.
    # ------------------------------------------------------------------------------

    #: Expoente `α` do replay priorizado do Apêndice G — `P(i) ∝ p_i^α`, com
    #: `p_i = |ν_i − z_i|`: a distância entre o valor que a **busca** encontrou na raiz e
    #: o retorno de n passos que o jogo de fato entregou. `0` sorteia uniforme, que é o
    #: que este repositório sempre fez (e é o que o paper faz nos jogos de tabuleiro; em
    #: Atari ele prioriza, com `α = β = 1`).
    #:
    #: Repare que as duas quantidades são fixas no instante da coleta, então a prioridade
    #: **não é atualizada** depois — ao contrário da do DQN, que segue o erro TD atual.
    #: Isso torna a implementação bem mais simples e é o que o paper especifica.
    per: float = 0.0
    #: Expoente `β` da correção de viés de amostragem. `w_i = (1/N · 1/P(i))^β`, e o paper
    #: usa `β = 1` — correção total, sem o *annealing* que o PER do DQN faz.
    per_beta: float = 1.0

    #: Fração de cada minilote cujo alvo de política do **passo 0** é refeito com a rede
    #: atual, antes do passo de gradiente. `0,8` é o número do paper; `0` desliga.
    #:
    #: Por que isto existe: este repositório faz `epochs_por_iter × batch_size` amostras
    #: de gradiente contra `num_envs × rollout` passos novos — **2,0 amostras por
    #: estado**, que é exatamente o número do Reanalyse (o MuZero puro usa 0,1). E o
    #: Reanalyse não é só um número de reúso: ele existe porque reúso alto precisa de
    #: alvo fresco. Sem ele, o alvo de visitas de uma amostra veio de uma rede `g` de
    #: dezenas de iterações atrás e é treinado contra um modelo que já se moveu.
    #:
    #: O alvo refeito é **escrito de volta no buffer**, então o trabalho não se perde: uma
    #: linha refrescada continua fresca nos sorteios seguintes.
    #:
    #: **Escopo, dito com todas as letras.** Só o passo 0 é refeito, porque só a
    #: observação do passo 0 é guardada — os passos `1..K` do desenrolar precisariam das
    #: observações seguintes. Isso cobre exatamente o termo que a métrica oficial mede
    #: (§2.31) e deixa de fora os imaginados. O alvo de **valor** também não é refeito:
    #: `z` é um retorno de n passos com bootstrap, e refazê-lo exigiria a rede alvo do
    #: Apêndice H mais o estado em `t+n`, que o buffer não guarda. Isto é, portanto, o
    #: Reanalyse **da política**, e não o Apêndice H inteiro.
    reanalise: float = 0.0
    #: Orçamento de busca do Reanalyse. `0` usa `num_simulations`, que é o que o paper
    #: faz. Baixar é o botão de custo — e é um desvio, porque produz um alvo de qualidade
    #: menor que o da coleta.
    reanalise_sims: int = 0

    # ------------------------------------------------------------------------------
    # Apêndice F — a cabeça categórica e a transformação de escala. Ver §2.33.
    # ------------------------------------------------------------------------------

    #: Átomos do suporte discreto do valor e da recompensa. `0` mantém a cabeça escalar
    #: com erro quadrático, que é o que este repositório sempre teve.
    #:
    #: O paper não faz regressão escalar: ele projeta o alvo num **suporte discreto** com
    #: two-hot (um alvo de 3,7 vira peso 0,3 no átomo 3 e 0,7 no átomo 4) e treina a
    #: cabeça com **entropia cruzada**, lendo o número de volta pela esperança da softmax.
    #: São 601 átomos para uma faixa de `[-300, 300]` em Atari; aqui a faixa é a do
    #: `symlog`, `[-LIMITE, +LIMITE]`, então 121 átomos dão o mesmo espaçamento relativo.
    #:
    #: Por que isto pode importar aqui: com a cabeça escalar, `perda_v` é um MSE cuja
    #: escala depende do alvo e cujo gradiente cresce com o erro — e §2.31 mediu o valor
    #: em `[6,7; 17,5]` de banda na escala real, que é o número que a árvore soma no
    #: backup. Entropia cruzada sobre um suporte fixo tem gradiente limitado e calibra
    #: uma distribuição em vez de um ponto.
    n_suporte: int = 0
    #: Teto do suporte, **na escala real**. O paper usa 601 átomos em `[-300, 300]`
    #: porque um retorno de Atari é grande; copiar esses números aqui daria resolução de
    #: ~3 pontos perto de zero, num jogo cujo valor medido vive entre 0 e ~11. Dimensionar
    #: pelo domínio é o que preserva o espírito: com 60 e 121 átomos, a resolução perto de
    #: zero fica em ~0,07 e perto de 10 em ~0,75, para as duas transformações.
    teto_suporte: float = 60.0
    #: `"symlog"` é o que está aqui desde §2.28 (DreamerV3). `"h"` é a do paper —
    #: `h(x) = sign(x)(√(|x|+1) − 1 + εx)`, do R2D2, com `ε = 0,001`. As duas comprimem;
    #: a `h` cresce como √x e a `symlog` como log x, então a `h` preserva mais resolução
    #: longe de zero, que é onde o valor deste jogo vive quando o agente é bom.
    transformacao: str = "symlog"

    #: **0,25 é o valor do paper**, não um chute: o Apêndice H baixa o alvo de valor para
    #: 0,25 contra 1,0 de política e recompensa, e diz por quê — "avoid overfitting of the
    #: value function". Subir isto é ir contra o paper, não em direção a ele.
    coef_valor: float = 0.25
    coef_recompensa: float = 1.0

    sims_avaliacao: int = 24


class MuZero(AgentBase):
    algo = "muzero"

    def __init__(self, cfg: MuZeroConfig = None, variant=None):
        cfg = cfg or MuZeroConfig()
        super().__init__(cfg, variant=variant or f"unroll{cfg.unroll}")
        keras.utils.set_random_seed(cfg.seed)

        self.h = build_representacao(cfg.board_size, cfg.net)
        self.g = build_dinamica(cfg.board_size, cfg.net, n_suporte=cfg.n_suporte)
        self.f = build_predicao(cfg.board_size, cfg.net, n_suporte=cfg.n_suporte)
        # os átomos vivem na escala **transformada**, e são constantes do grafo
        if cfg.n_suporte > 0:
            teto = float(self._comprime(tf.constant(cfg.teto_suporte)).numpy())
            self.atomos = tf.constant(
                np.linspace(-teto, teto, cfg.n_suporte, dtype=np.float32))
            self.passo_suporte = float(2 * teto / (cfg.n_suporte - 1))
        else:
            self.atomos, self.passo_suporte = None, 0.0
        self.largura = PRESETS[cfg.net][0]

        self.optimizer = keras.optimizers.Adam(cfg.lr, clipnorm=cfg.max_grad_norm)
        self.optimizer.build(self._variaveis())

        self.env = VecSnake(cfg.num_envs, cfg.board_size,
                            rng=np.random.default_rng(cfg.seed))
        self.obs, self.mask = self.env.reset()
        self.rng = np.random.default_rng(cfg.seed + 1)

        self.mcts = MCTS(self._avaliar_oculto, board_size=cfg.board_size,
                         gamma=cfg.gamma, num_simulations=cfg.num_simulations,
                         c_puct=cfg.c_puct, dirichlet_alpha=cfg.dirichlet_alpha,
                         dirichlet_frac=cfg.dirichlet_frac,
                         fpu=cfg.fpu, q_normalizado=cfg.q_normalizado,
                         desempate=cfg.desempate,
                         dinamica=DinamicaAprendida(self._passo_dinamica),
                         rng=np.random.default_rng(cfg.seed + 2))

        forma = (cfg.board_size, cfg.board_size, N_CHANNELS)
        M, K = cfg.memory_size, cfg.unroll
        self._buf_obs = np.zeros((M, *forma), dtype=np.float32)
        self._buf_mask = np.ones((M, N_ACTIONS), dtype=bool)
        self._buf_act = np.zeros((M, K), dtype=np.int32)
        self._buf_pi = np.zeros((M, K + 1, N_ACTIONS), dtype=np.float32)
        self._buf_z = np.zeros((M, K + 1), dtype=np.float32)
        self._buf_r = np.zeros((M, K), dtype=np.float32)
        #: Quais passos do desenrolar são reais. Ver a nota em `collect`.
        self._buf_vivo = np.zeros((M, K + 1), dtype=np.float32)
        #: `|ν − z|` do Apêndice G, gravado na coleta e nunca mais tocado.
        self._buf_prio = np.zeros(M, dtype=np.float32)
        self._pos, self._cheio = 0, 0

    def _variaveis(self):
        return (self.h.trainable_variables + self.g.trainable_variables
                + self.f.trainable_variables)

    def on_model_reloaded(self):
        self.optimizer = keras.optimizers.Adam(self.cfg.lr,
                                               clipnorm=self.cfg.max_grad_norm)
        self.optimizer.build(self._variaveis())

    # -------------------------------------------------------- modelo -> busca
    @property
    def model(self):
        """O `AgentBase` salva `self.model`; para o MuZero o que interessa é `h`+`f`."""
        return self._modelo_politica()

    @model.setter
    def model(self, _):
        pass          # o estado real vive em h, g, f

    def _modelo_politica(self):
        """Modelo `observação → [logits, valor]`, para a política pura e o checkpoint."""
        if getattr(self, "_mp", None) is None and hasattr(self, "h"):
            inp = keras.Input(shape=(self.cfg.board_size, self.cfg.board_size,
                                     N_CHANNELS))
            logits, valor = self.f(self.h(inp))
            self._mp = keras.Model(inp, [logits, valor], name="muzero_politica")
        return getattr(self, "_mp", None)

    #: Teto do valor antes do `symexp`, igual ao do AlphaZero: uma cabeça que divergiu
    #: vira número grande e finito em vez de envenenar a árvore inteira.
    LIMITE_SYMLOG = 6.0

    @staticmethod
    def _symlog(x):
        return tf.sign(x) * tf.math.log1p(tf.abs(x))

    @staticmethod
    def _symexp(x):
        x = tf.clip_by_value(x, -MuZero.LIMITE_SYMLOG, MuZero.LIMITE_SYMLOG)
        return tf.sign(x) * tf.math.expm1(tf.abs(x))

    #: `ε` da transformação do paper. Existe para `h` ser invertível e ter derivada
    #: limitada longe de zero; sem ele a inversa explode.
    EPS_H = 1e-3
    #: O teto do `symexp` vale ~402 na escala real (`e⁶ − 1`). `h` cresce como √x, então
    #: o **mesmo** teto real pede um limite diferente: `h(402) ≈ 19,5`. Reusar o 6,0 aqui
    #: cortaria o valor em 47 — um teto que este jogo pode encostar, e um corte que não
    #: levanta exceção nenhuma: só devolve um valor sistematicamente baixo para a árvore.
    LIMITE_H = 19.5

    @staticmethod
    def _h(x):
        """`sign(x)(√(|x|+1) − 1 + εx)` — R2D2, e o que o Apêndice F do MuZero usa."""
        return tf.sign(x) * (tf.sqrt(tf.abs(x) + 1.0) - 1.0) + MuZero.EPS_H * x

    @staticmethod
    def _h_inv(x):
        """A inversa fechada de `_h`. Vale a pena escrever a conta porque errá-la não
        levanta exceção nenhuma — só devolve um valor sistematicamente torto para a
        árvore somar no backup."""
        x = tf.clip_by_value(x, -MuZero.LIMITE_H, MuZero.LIMITE_H)
        e = MuZero.EPS_H
        a = tf.abs(x)
        raiz = tf.sqrt(1.0 + 4.0 * e * (a + 1.0 + e))
        return tf.sign(x) * (tf.square((raiz - 1.0) / (2.0 * e)) - 1.0)

    def _comprime(self, x):
        return self._h(x) if self.cfg.transformacao == "h" else self._symlog(x)

    def _descomprime(self, x):
        return self._h_inv(x) if self.cfg.transformacao == "h" else self._symexp(x)

    def _do_suporte(self, saida):
        """Saída da cabeça → escalar **na escala transformada**.

        É a única costura entre a cabeça escalar e a categórica: com `n_suporte = 0` ela
        só tira a dimensão sobrando; com suporte, é a esperança da softmax sobre os
        átomos. Todo o resto do agente continua vendo um escalar.
        """
        if self.cfg.n_suporte <= 0:
            return tf.squeeze(saida, -1)
        return tf.reduce_sum(tf.nn.softmax(saida) * self.atomos, -1)

    def _dois_quentes(self, x):
        """Escalar na escala transformada → alvo two-hot `(..., n_suporte)`.

        Um alvo de 3,7 num suporte de passo 1 vira 0,3 no átomo 3 e 0,7 no átomo 4 — a
        projeção que faz a esperança da distribuição reproduzir o número exato.
        """
        n = self.cfg.n_suporte
        x = tf.clip_by_value(x, self.atomos[0], self.atomos[-1])
        pos = (x - self.atomos[0]) / self.passo_suporte
        baixo = tf.floor(pos)
        peso = pos - baixo
        i = tf.cast(baixo, tf.int32)
        return (tf.one_hot(i, n) * (1.0 - peso)[..., None]
                + tf.one_hot(tf.minimum(i + 1, n - 1), n) * peso[..., None])

    def _perda_escalar(self, saida, alvo):
        """Entropia cruzada com suporte; erro quadrático sem. Mesma assinatura, para o
        laço do desenrolar não precisar saber qual dos dois está ligado."""
        if self.cfg.n_suporte <= 0:
            return tf.square(tf.squeeze(saida, -1) - alvo)
        return -tf.reduce_sum(self._dois_quentes(alvo) * tf.nn.log_softmax(saida), -1)

    def _recompensa_real(self, r):
        """A recompensa que o backup soma. Espelha `alvo_r` em `_passo`: comprimida só
        quando há suporte."""
        return self._descomprime(r) if self.cfg.n_suporte > 0 else r

    def _valor_real(self, valor):
        """A escala que o MCTS precisa: ele soma `recompensa + γ·valor`, e a recompensa
        é a que a rede `g` prevê, na escala do mundo."""
        return self._descomprime(valor) if self.cfg.valor_symlog else valor

    @tf.function(reduce_retracing=True)
    def _repr_predicao(self, obs, mask):
        s = self.h(obs, training=False)
        logits, valor = self.f(s, training=False)
        logits = tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))
        return s, tf.nn.softmax(logits), self._valor_real(self._do_suporte(valor))

    @tf.function(reduce_retracing=True)
    def _predicao(self, s):
        logits, valor = self.f(s, training=False)
        return tf.nn.softmax(logits), self._valor_real(self._do_suporte(valor))

    @tf.function(reduce_retracing=True)
    def _dinamica_tf(self, s, planos):
        novo, r = self.g([s, planos], training=False)
        return novo, self._recompensa_real(self._do_suporte(r))

    def _planos_de_acao(self, acoes, n):
        b = self.cfg.board_size
        planos = np.zeros((n, b, b, N_ACTIONS), dtype=np.float32)
        planos[np.arange(n), :, :, acoes] = 1.0
        return planos

    def _passo_dinamica(self, estados, acoes):
        """A interface que `DinamicaAprendida` consome."""
        planos = self._planos_de_acao(acoes, len(acoes))
        novo, r = self._dinamica_tf(tf.convert_to_tensor(estados),
                                    tf.convert_to_tensor(planos))
        return novo.numpy(), r.numpy()

    def _avaliar_oculto(self, estados, mask):
        """Priors e valores a partir de estados ocultos — o que o MCTS chama."""
        p, v = self._predicao(tf.convert_to_tensor(np.asarray(estados,
                                                              dtype=np.float32)))
        p = p.numpy()
        m = np.asarray(mask)
        if m.shape == p.shape and not m.all():
            p = np.where(m, p, 0.0)
            p /= np.maximum(p.sum(1, keepdims=True), 1e-12)
        return p, v.numpy()

    def politica(self):
        """Política pura de `h`+`f`, sem busca — a curva oficial do contrato."""
        modelo = self._modelo_politica()

        def fn(obs, mask):
            logits, _ = modelo(obs, training=False)
            return np.where(mask, np.asarray(logits), MASK_NEG).astype(np.float32)
        return fn

    @tf.function(reduce_retracing=True)
    def _representacao(self, obs):
        return self.h(obs, training=False)

    def _busca(self, obs, mask, ruido=False, busca=None):
        """Roda o MCTS a partir da observação: `h` uma vez, depois só a dinâmica.

        Usa `_representacao` e **não** `_repr_predicao`: o `MCTS.run` avalia a raiz por
        conta própria, então pedir priors e valor aqui seria uma segunda passagem da cabeça
        de predição por jogada, jogada fora. Custo, não correção — mas custo justamente no
        lugar onde ele é o argumento da coluna separada.
        """
        s = self._representacao(tf.convert_to_tensor(obs)).numpy()
        arvore = busca if busca is not None else self.mcts
        return arvore.run(s, mask, s, adicionar_ruido=ruido)

    def avaliar_com_busca(self, episodes=1000, num_simulations=None, seed=123,
                          num_envs=None, max_segundos=None, verbose=False):
        """O protocolo oficial, mas escolhendo com MCTS — a **coluna separada** da tabela.

        Mesmo desenho do `AlphaZero.avaliar_com_busca`, com a diferença que é justamente o
        ponto do MuZero: a árvore percorre `g`, não o `VecSnake`. O ambiente aqui só avança
        o jogo de verdade entre as jogadas — a busca nunca o consulta.

        A contabilidade é a de `AgentBase.rodar_protocolo`, compartilhada com o AlphaZero
        para que as duas colunas não possam divergir.
        """
        cfg = self.cfg
        busca = MCTS(self._avaliar_oculto, board_size=cfg.board_size, gamma=cfg.gamma,
                     num_simulations=num_simulations or cfg.sims_avaliacao,
                     c_puct=cfg.c_puct, fpu=cfg.fpu, q_normalizado=cfg.q_normalizado,
                     desempate=cfg.desempate,
                     dinamica=DinamicaAprendida(self._passo_dinamica),
                     rng=np.random.default_rng(seed))

        def escolher(env, obs, mask):
            visitas, _ = self._busca(obs, mask, busca=busca)
            return visitas.argmax(1).astype(np.int32)

        st = self.rodar_protocolo(escolher, episodes=episodes, seed=seed,
                                  num_envs=num_envs, max_segundos=max_segundos,
                                  verbose=verbose)
        st["num_simulations"] = busca.num_simulations
        return st

    # -------------------------------------------------------------------- coleta
    def temperatura(self):
        """Escalar (fração do treino) ou `(N,)` (por lance do episódio, o do paper).

        Ver a nota homônima em `alphazero.py`: com o agendamento por fração do treino,
        metade do orçamento inteiro é jogada com τ = 1, inclusive nas posições apertadas.
        """
        cfg = self.cfg
        if cfg.temp_esquema == "treino":
            # Apêndice D, Atari: os degraus são por passo de TREINO e valem para o
            # episódio inteiro. Um escalar, e de propósito — o lance não entra na conta.
            f = self.frac()
            return 1.0 if f < 0.5 else (0.5 if f < 0.75 else 0.25)
        if cfg.temp_passos > 0:
            return np.where(self.env.steps < cfg.temp_passos,
                            cfg.temp_inicio, cfg.temp_fim).astype(np.float64)
        f = min(1.0, self.frac() / max(cfg.temp_frac, 1e-9))
        return cfg.temp_inicio + f * (cfg.temp_fim - cfg.temp_inicio)

    def collect(self):
        cfg = self.cfg
        T, N, K = cfg.rollout, cfg.num_envs, cfg.unroll

        obs_b = np.empty((T, N, cfg.board_size, cfg.board_size, N_CHANNELS), np.float32)
        mask_b = np.empty((T, N, N_ACTIONS), bool)
        pi_b = np.empty((T, N, N_ACTIONS), np.float32)
        v_b = np.empty((T, N), np.float32)
        act_b = np.empty((T, N), np.int32)
        rew_b = np.empty((T, N), np.float32)
        done_b = np.empty((T, N), np.float32)

        scores, vitorias, temps = [], 0, []
        for t in range(T):
            obs_b[t], mask_b[t] = self.obs, self.mask
            # com `temp_passos` a temperatura depende do lance de cada ambiente, e os N
            # ambientes estão em lances diferentes: tem que ser lida a cada passo
            temp = self.temperatura()
            temps.append(float(np.mean(temp)))
            visitas, valores = self._busca(self.obs, self.mask, ruido=True)
            pi = MCTS.politica_das_visitas(visitas, temp)
            # o alvo de treino não precisa ser a distribuição que escolheu a ação: no
            # AlphaZero/MuZero ele é a contagem de visitas crua
            pi_b[t] = (pi if cfg.temp_alvo <= 0
                       else MCTS.politica_das_visitas(visitas, cfg.temp_alvo))
            v_b[t] = valores
            a = (pi.cumsum(1) > self.rng.random((N, 1))).argmax(1).astype(np.int32)
            act_b[t] = a
            self.obs, self.mask, r, d, info = self.env.step(a)
            self.registra_fim(info)
            if info["trunc_idx"].size:       # fome é truncamento, não terminação
                _, _, v_f = self._repr_predicao(
                    tf.convert_to_tensor(info["final_obs"]),
                    tf.convert_to_tensor(info["final_mask"]))
                r = self.bootstrap_truncados(info, r, v_f.numpy(), cfg.gamma)
            rew_b[t], done_b[t] = r, d.astype(np.float32)
            scores.extend(info["scores"].tolist())
            vitorias += info["wins"]

        # alvo de valor por passo: n passos + bootstrap no valor da busca
        # O `n` encolhe no fim da janela. Com `rollout=16` e `n_step=10`, o estado
        # `t + n_step` está fora do buffer para todo `t >= 6` — e a versão anterior
        # simplesmente **não fazia bootstrap** nesses casos: dez dos dezesseis passos
        # tratavam o fim da coleta como fim de episódio, e num jogo de recompensa esparsa
        # isso é um alvo quase sempre igual a zero, que ainda por cima realimenta a busca.
        # Encurtar o horizonte e fazer bootstrap no último estado disponível troca um
        # pouco de viés de horizonte por um alvo que não é puxado para zero.
        # Ver `docs/REVISAO_ALGORITMOS.md` §2.5.
        # `bootstrap_fim_janela` acrescenta uma linha `T`: o valor da REDE no estado em que
        # a coleta parou. Menos preciso que o resto do vetor, que é valor de busca — e
        # ainda assim melhor que tratar o fim da janela como fim de episódio.
        if cfg.bootstrap_fim_janela:
            _, _, v_fim = self._repr_predicao(tf.convert_to_tensor(self.obs),
                                              tf.convert_to_tensor(self.mask))
            v_boot = np.concatenate([v_b, v_fim.numpy().astype(np.float32)[None]], axis=0)
            limite = T                  # o passo T-1 passa a ter para onde olhar
        else:
            v_boot = v_b
            limite = T - 1              # o padrão do §2.5: um passo sem bootstrap, e só ele

        z = np.zeros((T, N), np.float32)
        for t in range(T):
            g = np.zeros(N, np.float32)
            desc = np.ones(N, np.float32)
            vivo = np.ones(N, bool)
            n = min(cfg.n_step, limite - t)     # 0 só no último passo, e só sem bootstrap
            for k in range(n):
                g += desc * rew_b[t + k] * vivo
                vivo &= done_b[t + k] < 0.5
                desc *= cfg.gamma
            if n > 0:
                g += desc * v_boot[t + n] * vivo
            else:                               # t = T-1: não há estado seguinte aqui
                g += rew_b[t]
            z[t] = g

        # Cada amostra guarda o desenrolar de K passos que vem depois dela — e **nem todo
        # passo é real**. O `VecSnake` reseta sozinho ao terminar, então uma janela que
        # atravessa a morte da cobra continua em índices que pertencem a uma partida NOVA,
        # sorteada, com a cobra em outro lugar. Sem máscara, `g` é treinada a prever a
        # recompensa desse outro jogo — e a perda de recompensa é, segundo o docstring
        # deste módulo, "a única âncora que liga o estado oculto ao mundo". Medido no
        # cenário do contrato (`T=16`, `K=5`), **25% das amostras atravessam pelo menos uma
        # morte**.
        #
        # A mesma máscara resolve o outro buraco: antes, as `K` últimas linhas da janela
        # eram simplesmente descartadas (`validos = T - K`), o que jogava fora **31% dos
        # passos coletados** — que continuavam contados no orçamento de 5 M. Agora os
        # passos que cairiam fora da janela são mascarados em vez de a linha inteira ser
        # jogada fora, e as `T` linhas viram amostra.
        vivo_k = np.ones((T, N, K + 1), np.float32)
        for t in range(T):
            for j in range(1, K + 1):
                dentro = 1.0 if t + j <= T - 1 else 0.0
                anterior = min(t + j - 1, T - 1)
                vivo_k[t, :, j] = (vivo_k[t, :, j - 1]
                                   * (done_b[anterior] < 0.5).astype(np.float32)
                                   * dentro)

        idx = np.arange(T)
        # o índice é grampeado para a leitura não estourar; o que ele lê a mais está
        # zerado pela máscara
        def _janela(fonte, k):
            return fonte[np.minimum(idx + k, T - 1)]

        self._guardar(
            obs_b[idx].reshape(-1, *obs_b.shape[2:]),
            mask_b[idx].reshape(-1, N_ACTIONS),
            np.stack([_janela(act_b, k) for k in range(K)], axis=-1).reshape(-1, K),
            np.stack([_janela(pi_b, k) for k in range(K + 1)], axis=1)
              .transpose(0, 2, 1, 3).reshape(-1, K + 1, N_ACTIONS),
            np.stack([_janela(z, k) for k in range(K + 1)], axis=-1).reshape(-1, K + 1),
            np.stack([_janela(rew_b, k) for k in range(K)], axis=-1).reshape(-1, K),
            vivo_k.reshape(-1, K + 1),
            # `|ν − z|`: o quanto a busca e o jogo discordaram naquele estado. Não é o
            # erro da rede — é o erro do alvo, e por isso não muda depois.
            np.abs(v_b - z).reshape(-1),
        )

        self.global_step += T * N
        self.episodes += len(scores)
        return {
            "train_score_mean": float(np.mean(scores)) if scores else None,
            "n_episodes": len(scores),
            "wins": vitorias,
            "temperatura": float(np.mean(temps)),
            "valor_busca": float(v_b.mean()),
            "memoria": self._cheio,
        }

    def _guardar(self, obs, mask, act, pi, z, r, vivo, prio=None):
        k = len(obs)
        idx = (self._pos + np.arange(k)) % self.cfg.memory_size
        self._buf_obs[idx] = obs
        self._buf_mask[idx] = mask
        self._buf_act[idx] = act
        self._buf_pi[idx] = pi
        self._buf_z[idx] = z
        self._buf_r[idx] = r
        self._buf_vivo[idx] = vivo
        self._buf_prio[idx] = 0.0 if prio is None else prio
        self._pos = int((self._pos + k) % self.cfg.memory_size)
        self._cheio = min(self._cheio + k, self.cfg.memory_size)

    # -------------------------------------------------------------------- treino
    @staticmethod
    def _media_mascarada(x, m):
        """Média só sobre os passos reais. Denominador é a contagem, não o lote inteiro —
        senão a perda encolheria só porque a janela atravessou uma morte."""
        return tf.reduce_sum(x * m) / tf.maximum(tf.reduce_sum(m), 1.0)

    @tf.function(reduce_retracing=True)
    def _passo(self, obs, mask, act, pi_alvo, z, r_alvo, vivo, w, coef_v, coef_r):
        K = tf.shape(act)[1]
        with tf.GradientTape() as tape:
            s = self.h(obs, training=True)
            logits, valor = self.f(s, training=True)
            logits = tf.where(mask, logits, tf.fill(tf.shape(logits), MASK_NEG))
            logp = tf.nn.log_softmax(logits)

            alvo_v = self._comprime(z) if self.cfg.valor_symlog else z
            # a recompensa só é comprimida quando há suporte: sem ele a cabeça é uma
            # regressão na escala do mundo, como sempre foi. Com suporte, alvo e átomos
            # têm de morar na mesma escala — senão a projeção two-hot satura no primeiro
            # átomo e a cabeça aprende a prever a borda.
            alvo_r = self._comprime(r_alvo) if self.cfg.n_suporte > 0 else r_alvo
            # o passo 0 fica separado: é o único que a métrica oficial mede, e sem
            # instrumentá-lo não dá para saber se a destilação falha nele ou nos passos
            # imaginados — a soma sozinha esconde os dois casos
            # `w` é o peso de importância do replay priorizado; sem PER ele é 1 em toda
            # linha e a média ponderada é a média. Ele entra como PESO DA MÁSCARA nos
            # passos do desenrolar, o que é exatamente a mesma conta e evita um segundo
            # caminho de redução para manter em pé.
            perda_pi_0 = -self._media_mascarada(
                tf.reduce_sum(pi_alvo[:, 0] * logp, -1), w)
            perda_v = self._media_mascarada(
                self._perda_escalar(valor, alvo_v[:, 0]), w)
            perda_pi_k = tf.constant(0.0)
            perda_r = tf.constant(0.0)
            peso = (1.0 / self.cfg.unroll) if self.cfg.normaliza_unroll else 1.0

            for k in range(self.cfg.unroll):
                planos = tf.one_hot(act[:, k], N_ACTIONS)[:, None, None, :]
                planos = tf.tile(planos, [1, self.cfg.board_size,
                                          self.cfg.board_size, 1])
                s, rec = self.g([s, planos], training=True)
                # escala de gradiente de 1/2: sem ela o gradiente que chega em `h` cresce
                # com o número de passos do desenrolar
                s = s * 0.5 + tf.stop_gradient(s) * 0.5

                logits_k, valor_k = self.f(s, training=True)
                logp_k = tf.nn.log_softmax(logits_k)
                # `vivo[:, j]` zera o que pertence a uma partida seguinte ou está fora
                # da janela. O passo `k` consome a ação e a recompensa do instante `t+k`
                # (precisa estar vivo lá, `vivo[:, k]`) e produz alvos do instante
                # `t+k+1` (`vivo[:, k + 1]`).
                perda_pi_k += peso * self._media_mascarada(
                    -tf.reduce_sum(pi_alvo[:, k + 1] * logp_k, -1), vivo[:, k + 1] * w)
                perda_v += peso * self._media_mascarada(
                    self._perda_escalar(valor_k, alvo_v[:, k + 1]), vivo[:, k + 1] * w)
                # a âncora do modelo no mundo real: sem ela a dinâmica pode inventar
                # qualquer física internamente consistente — e treiná-la contra a
                # recompensa de outra partida é pior que não treinar
                perda_r += peso * self._media_mascarada(
                    self._perda_escalar(rec, alvo_r[:, k]), vivo[:, k] * w)

            perda_pi = perda_pi_0 + perda_pi_k
            perda = perda_pi + coef_v * perda_v + coef_r * perda_r

        variaveis = (self.h.trainable_variables + self.g.trainable_variables
                     + self.f.trainable_variables)
        grads = tape.gradient(perda, variaveis)
        self.optimizer.apply_gradients(zip(grads, variaveis))
        return perda_pi, perda_v, perda_r, perda_pi_0

    def _busca_reanalise(self):
        """A árvore do Reanalyse. É a **mesma** da coleta salvo se `reanalise_sims` pedir
        outro orçamento — e então ela é construída uma vez e guardada, porque um `MCTS`
        novo por chamada seria reconstruído `epochs_por_iter` vezes por iteração e
        re-semeado em cada uma.

        A `DinamicaAprendida` não é opcional aqui: sem ela a árvore percorreria o
        simulador real, que é o AlphaZero. É o tipo de troca silenciosa que não levanta
        exceção — só devolve outro algoritmo.
        """
        cfg = self.cfg
        if not cfg.reanalise_sims or cfg.reanalise_sims == cfg.num_simulations:
            return self.mcts
        if getattr(self, "_mcts_rea", None) is None:
            self._mcts_rea = MCTS(
                self._avaliar_oculto, board_size=cfg.board_size, gamma=cfg.gamma,
                num_simulations=cfg.reanalise_sims, c_puct=cfg.c_puct,
                fpu=cfg.fpu, q_normalizado=cfg.q_normalizado, desempate=cfg.desempate,
                dinamica=DinamicaAprendida(self._passo_dinamica),
                rng=np.random.default_rng(cfg.seed + 3))
        return self._mcts_rea

    def _reanalisar(self, idx):
        """Refaz a busca com a rede **atual** sobre observações guardadas e reescreve o
        alvo de política do passo 0, no buffer.

        Duas escolhas que merecem estar escritas:

        * **Sem ruído de Dirichlet.** O ruído da raiz existe para explorar durante a
          geração de dados; aqui o que se produz é um alvo. A consequência é real e vale
          nomear: um buffer meio refrescado carrega **duas distribuições de alvo**, uma
          sorteada e uma determinística. Qual das duas é mais aguda depende do estado do
          treino — com a rede treinada, cujo prior já é agudo, o ruído espalha e o refeito
          sai mais afiado; com a rede recém-iniciada, cujo prior é quase uniforme, um
          sorteio de `Dir(1,1,1)` (máximo esperado ~0,61) é mais agudo que ela e a direção
          se inverte. O que vale nos dois casos, e é o que o teste protege, é que o alvo
          refeito é **reprodutível**. Com a escrita de volta, o buffer converge para ele.
        * **Só o passo 0.** É o único cuja observação o buffer guarda — e é o único que a
          métrica oficial mede, porque `politica()` age sobre a observação real (§2.31).
        """
        cfg = self.cfg
        visitas, _valores = self._busca(self._buf_obs[idx], self._buf_mask[idx],
                                        ruido=False, busca=self._busca_reanalise())
        # `temp_alvo <= 0` é o regime do braço `sem_alvo_cru`, em que o alvo é a
        # distribuição temperada que escolheu a ação. Ela depende do lance do episódio,
        # que o buffer não guarda — então aqui o alvo refeito é sempre a contagem crua.
        temp = cfg.temp_alvo if cfg.temp_alvo > 0 else 1.0
        self._buf_pi[idx, 0] = MCTS.politica_das_visitas(visitas, temp)
        return len(idx)

    def _aprender(self):
        cfg = self.cfg
        if self._cheio < cfg.batch_size:
            return None
        lr = cfg.lr
        if cfg.lr_final > 0:
            lr = self.linear(cfg.lr, cfg.lr_final)
            self.optimizer.learning_rate.assign(lr)
        saidas = []
        refeitos = 0
        probs = None
        if cfg.per > 0:
            p = np.power(self._buf_prio[:self._cheio] + 1e-6, cfg.per)
            soma = p.sum()
            # com o buffer recém-criado, ou num trecho em que busca e jogo concordaram
            # exatamente, `p` pode somar zero — e aí o sorteio proporcional não existe.
            # Cair para o uniforme é o comportamento certo, e silencioso de propósito.
            probs = (p / soma) if soma > 0 else None
        n_reanalise = int(round(cfg.reanalise * cfg.batch_size))
        for _ in range(cfg.epochs_por_iter):
            if probs is None:
                i = self.rng.integers(0, self._cheio, size=cfg.batch_size)
                w = np.ones(cfg.batch_size, np.float32)
            else:
                i = self.rng.choice(self._cheio, size=cfg.batch_size, p=probs)
                # `w = (1/(N·P))^β`, normalizado pelo máximo do lote — a mesma convenção
                # do `memory/replay.py` deste repositório. Sem normalizar, a escala da
                # perda passaria a depender de qual amostra caiu no lote.
                w = np.power(1.0 / (self._cheio * probs[i]), cfg.per_beta)
                w = (w / w.max()).astype(np.float32)
            # antes da leitura do lote, e escrevendo no buffer: o alvo refeito vale para
            # este passo de gradiente **e** para os sorteios futuros que caírem na mesma
            # linha. É o que faz a taxa de refresco compor em vez de se perder.
            if n_reanalise > 0:
                refeitos += self._reanalisar(i[:n_reanalise])
            p, v, r, p0 = self._passo(
                tf.convert_to_tensor(self._buf_obs[i]),
                tf.convert_to_tensor(self._buf_mask[i]),
                tf.convert_to_tensor(self._buf_act[i]),
                tf.convert_to_tensor(self._buf_pi[i]),
                tf.convert_to_tensor(self._buf_z[i]),
                tf.convert_to_tensor(self._buf_r[i]),
                tf.convert_to_tensor(self._buf_vivo[i]),
                tf.convert_to_tensor(w),
                cfg.coef_valor, cfg.coef_recompensa,
            )
            saidas.append((float(p), float(v), float(r), float(p0)))
        p, v, r, p0 = (float(np.mean(x)) for x in zip(*saidas))
        return {"perda_pi": p, "perda_v": v, "perda_r": r, "lr": float(lr),
                # `perda_pi_0` é a que corresponde ao que a curva oficial mede;
                # `frac_pi_0` diz quanto dela sobrou dentro da soma
                "perda_pi_0": p0, "frac_pi_0": p0 / max(p, 1e-9),
                # buscas gastas refazendo alvo — o custo do Reanalyse, contra as
                # `num_envs × rollout` da coleta, que é a régua para lê-lo
                "reanalises": refeitos,
                "atualizacoes": cfg.epochs_por_iter}

    def iterate(self):
        stats = self.collect()
        treino = self._aprender()
        if treino:
            stats.update(treino)
        return stats

    # ---------------------------------------------------------------- checkpoint
    def salvar(self, tag="last"):
        for nome, rede in (("h", self.h), ("g", self.g), ("f", self.f)):
            rede.save(os.path.join(self.cfg.ckpt_dir,
                                   f"{self.algo}_{tag}_{nome}.keras"))
        super().salvar(tag)

    def retomar(self, tag="last"):
        caminhos = {n: os.path.join(self.cfg.ckpt_dir, f"{self.algo}_{tag}_{n}.keras")
                    for n in ("h", "g", "f")}
        if not all(os.path.exists(c) for c in caminhos.values()):
            return False
        self.h = keras.models.load_model(caminhos["h"])
        self.g = keras.models.load_model(caminhos["g"])
        self.f = keras.models.load_model(caminhos["f"])
        self._mp = None
        return super().retomar(tag)


ASSINATURA_PACOTE = "adbd46d52abbddf4"

# ==== FIM DO CÓDIGO GERADO ====

## Configuração

Os padrões abaixo são os do **contrato**: tabuleiro 10×10, 5 M passos de orçamento,
avaliação de 1.000 episódios com semente 123. Mexer neles é legítimo para experimentar,
mas o resultado só entra na arena se o contrato for respeitado — o `Recorder` recusa
qualquer outra coisa e diz o motivo.


In [ ]:
# @title Parâmetros
SEMENTE = 0        # @param {type:"integer"}
PASSOS = 5000000   # @param {type:"integer"}
REDE = "resnet_small"  # @param ["resnet_tiny", "resnet_small", "resnet_base", "cnn_rainbow", "cnn_alphazero", "cnn_vgg", "cnn_vgg_dropout", "cnn_vgg_sem_pool"]

# Armazenamento: nada para configurar. Detecta Colab, Kaggle ou máquina local e escolhe a
# pasta que **persiste** em cada um — Drive, /kaggle/working ou o diretório atual. Se a
# montagem do Drive falhar, avisa e segue, em vez de parar.
PASTA = pasta_de_trabalho()

# No Kaggle a sessão nova nasce com /kaggle/working vazio: o que sobreviveu está montado
# somente-leitura em /kaggle/input. Isto traz os checkpoints de volta — e nunca sobrescreve
# um checkpoint desta sessão, senão o treino andaria para trás.
semear_checkpoints(os.path.join(PASTA, "checkpoints"))

cfg = MuZeroConfig(
    seed=SEMENTE,
    net=REDE,
    total_steps=PASSOS,
    ckpt_dir=os.path.join(PASTA, "checkpoints"),
    runs_dir=os.path.join(PASTA, "runs"),
)
print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))

## Treino

**Retomável, e é requisito, não conveniência.** Um treino de 5 M passos não cabe numa
sessão gratuita sem cair pelo menos uma vez. Rode a célula de novo e ela continua do último
checkpoint.

* **Colab** — os checkpoints vão para o Drive e sobrevivem à queda da sessão.
* **Kaggle** — `/kaggle/working` vira a **saída** desta versão. Para continuar depois:
  *Save Version → Save & Run All* (roda headless, sem aba aberta), e na execução seguinte
  *Add Input → Your Work → Notebook Output* apontando para esta. A célula de parâmetros
  recupera os checkpoints sozinha.


In [ ]:
# @title Treinar
agente = MuZero(cfg)
if agente.retomar("last"):
    print("retomando do checkpoint")
print("parâmetros:", f"{agente.model.count_params():,}")

registro = agente.train(verbose=True)

## Veredito — os dois modelos

Duas perguntas diferentes, dois números:

* **`last`** — o modelo do último passo. É ele que entra na curva e na arena, porque é o
  estado final do algoritmo, instabilidade inclusa.
* **`best`** — o melhor checkpoint já visto. É ele que você levaria para o jogo.

Os dois existem porque **RL profundo não melhora monotonicamente**: fora do caso tabular
não há garantia nenhuma, e uma execução pode terminar pior do que já esteve. Na primeira
execução longa do ACKTR, 8 das 21 avaliações tinham um checkpoint anterior melhor que o
modelo daquele momento — numa delas, 21,7 pontos melhor.

Dentro de cada um, três regimes: piso aleatório, política pura e política com o filtro de
segurança. Se a coluna do meio não estiver bem acima do piso, não aprendeu — e aí o
problema é hiperparâmetro ou tempo de treino, não código.


In [ ]:
# @title Veredito
print("=== last · modelo do último passo (é o que entra na arena) ===")
_fome = getattr(agente.env, "canal_fome", False)
resultado = verdict(agente.politica(), episodes=1000, canal_fome=_fome)
print(format_verdict(resultado))

melhor = agente.modelo_melhor()
if melhor is not None:
    print()
    print(f"=== best · checkpoint do passo "
          f"{registro.record.melhor.get('global_step', 0):,} ===")
    _guardado, agente.model = agente.model, melhor
    try:
        print(format_verdict(verdict(agente.politica(), episodes=1000,
                                     canal_fome=_fome)))
    finally:
        agente.model = _guardado

fig, _ = plot_run(registro.record)
plt.show()

## Veredito com busca — a coluna separada do contrato

A curva oficial mede a **rede pura**, greedy, sem nenhuma ajuda. É o que torna as curvas
comparáveis: a busca gasta `num_simulations` avaliações de rede e outros tantos passos de
simulador **por jogada**, contra 1 avaliação do PPO. Somar as duas no mesmo eixo diria "o
AlphaZero ganha do PPO" quando o que aconteceu foi gastar 32× mais computação na hora de
decidir — a mesma razão que manda o filtro de flood-fill para uma coluna própria.

Reportar, no entanto, é obrigação. Um agente que existe para buscar, avaliado só sem
buscar, é meia medição — e a busca é o que você levaria para jogar de verdade, já que em
Snake o simulador está disponível na hora de agir. Então: coluna separada, não coluna
proibida.

O protocolo é o mesmo do contrato — 1.000 episódios, greedy (argmax das visitas), semente
123 — e a busca roda com a **mesma** configuração do treino (`fpu`, `q_normalizado`,
`desempate`, `c_puct`, `gamma`). Dois orçamentos, para mostrar a curva computação ×
qualidade: quanto do resultado vem da rede e quanto vem do lookahead.

**Custo, e por que ele te pega de surpresa.** O laço roda até cada ambiente fechar a cota,
e **um agente bom faz episódios longos** — no AlphaZero de 5 M eles passam de 900 passos.
Com 64 ambientes são ~16 episódios cada, ou seja ~15 mil passos de ambiente, cada um
custando `num_simulations + 1` avaliações de rede. No MuZero é o dobro disso, porque cada
simulação chama `g` **e** `f`, e a árvore nunca poda nós terminais (o modelo aprendido não
prevê fim de episódio). Uma medição de 1.000 episódios com 32 simulações passa de uma hora,
e é exatamente esse o motivo de a primeira tentativa nesta célula ter sido cancelada.

Por isso ela vem com `MINUTOS_MAX`. Ao estourar, o que deu tempo de medir volta marcado
`completo=False` — o mesmo campo que o `validate()` usa para recusar avaliação parcial, de
modo que o número serve para você olhar e **não entra na arena por engano**. O progresso é
impresso a cada 30 s, com estimativa do que falta.

**Comece pequeno se for só olhar.** `EPISODIOS = 200` e **um** orçamento dão a ordem de
grandeza em poucos minutos. Subir `EPISODIOS` é o lever que importa; `AMBIENTES` mais alto
melhora o aproveitamento da GPU mas encarece o laço de árvore em Python na mesma proporção,
então costuma ser quase neutro.

**Mas o padrão é 1.000, porque é o que entra na arena.** O resultado vai para o campo
`busca` do registro — irmão de `final` e `melhor`, não um canto de `meta` —, e a coluna
*com busca* da tabela só aceita entradas com os 1.000 episódios do contrato e
`completo=True`. Uma medição de 200 episódios tem erro padrão ~2,2× o da oficial; uma que
estourou `MINUTOS_MAX` é uma amostra enviesada para episódios **curtos**, que são
justamente os ruins. As duas ficam gravadas e marcadas — o registro guarda o que você
mediu, a arena publica só o que cumpre a régua.


In [ ]:
# @title Veredito com busca
import time

import numpy as np

# O contrato pede 1000. Menos que isso mede e grava, mas nao entra na arena: a coluna
# `com busca` da tabela so aceita o protocolo inteiro. 200/20min da a ordem de grandeza.
EPISODIOS_DO_CONTRATO = 1000
EPISODIOS = 1000              # @param {type:"integer"}
MINUTOS_MAX = 120             # @param {type:"integer"}
ORCAMENTOS = [cfg.sims_avaliacao]   # @param
AMBIENTES = 64                # @param {type:"integer"}
AVALIAR_MELHOR = False        # @param {type:"boolean"}

if not hasattr(agente, "avaliar_com_busca"):
    raise RuntimeError("este agente não busca na hora de agir — a coluna não se aplica")

# reaproveita o `resultado` da célula anterior: sem busca é barato, mas o filtro de
# flood-fill é laço Python e é a parte cara da avaliação
_base = globals().get("resultado")
if _base is None:
    _base = verdict(agente.politica(), episodes=EPISODIOS, com_filtro=False)
_tabela = {**_base, "linhas": list(_base["linhas"])}
_pura = _tabela["linhas"][1]["score_mean"]
_medidas = {}


def _mede_com_busca(_ag, modelo_nome):
    for _i, _sims in enumerate(ORCAMENTOS):
        _t0 = time.time()
        print(f"  {modelo_nome} · {_sims} sims ({EPISODIOS} episódios, teto "
              f"{MINUTOS_MAX} min)...", flush=True)
        _st = _ag.avaliar_com_busca(episodes=EPISODIOS, num_simulations=_sims,
                                    num_envs=AMBIENTES, max_segundos=MINUTOS_MAX * 60,
                                    verbose=True)
        _dt = time.time() - _t0
        # `checkpoint` e `num_simulations` viajam DENTRO da entrada: sem eles o
        # número não é interpretável, e ler a chave para descobrir de onde veio é um
        # contrato implícito que quebra na primeira vez que alguém renomeia a chave
        _st = {**_st, "checkpoint": modelo_nome}
        _medidas[f"{modelo_nome}_sims{_sims}"] = _st
        _rotulo = f"agente + busca ({_sims} sims)"
        if modelo_nome != "last":
            _rotulo += f" · {modelo_nome}"
        if not _st["completo"]:
            _rotulo += " ⚠ parcial"
        _tabela["linhas"].append({"regime": _rotulo, **_st})
        print(f"    score {_st['score_mean']:.2f} · cheio {_st['win_rate']:.1%} · "
              f"{_st['episodes']} episódios · {_dt / 60:.1f} min"
              + ("" if _st["completo"] else "  ⚠ TEMPO ESGOTADO: amostra parcial, "
                                            "`completo=False`, fora da arena"), flush=True)
        if _i == 0 and len(ORCAMENTOS) > 1:
            _resto = sum(ORCAMENTOS[1:]) / max(ORCAMENTOS[0], 1) * _dt
            print(f"     (os orçamentos restantes devem levar ~{_resto / 60:.0f} min)",
                  flush=True)


print(f"rede pura (a curva oficial): {_pura:.2f}   ·   piso {_tabela['piso']:.2f}")
print(f"medindo com busca, {EPISODIOS} episódios por orçamento...", flush=True)
_mede_com_busca(agente, "last")

if AVALIAR_MELHOR:
    # **Não** troque `agente.model` para medir o `best`, que é o que a célula do veredito
    # faz. Ali funciona porque `politica()` lê o atributo a cada chamada, em eager. Aqui a
    # busca passa por uma `tf.function` cujo traço **já capturou as variáveis** do modelo
    # atual: a troca seria silenciosamente ignorada e você mediria o `last` outra vez. No
    # MuZero é ainda mais silencioso — `model` é uma property com setter vazio. Um agente
    # novo tem o cache de traço vazio, e é a única forma honesta de medir o outro
    # checkpoint com busca.
    _ag_best = type(agente)(cfg)
    if _ag_best.retomar("best"):
        _mede_com_busca(_ag_best, "best")
    else:
        print("  sem checkpoint `best` — pulando")

print()
print(format_verdict(_tabela))
print()
for _nome, _st in _medidas.items():
    _s = _st["score_mean"]
    _linha = f"{_nome:>16}: busca {_s:>6.2f}  ·  rede pura {_pura:>6.2f}"
    # as razões só significam alguma coisa longe do zero; cedo no treino ambas são ~0 e
    # dividir uma pela outra imprime um número de sete dígitos que não quer dizer nada
    if _pura <= 0.5 or _s <= 0.5:
        _linha += "  ·  (razões omitidas: alguma das duas ainda está perto de zero)"
    elif _s > _pura:
        _linha += f"  ·  {_s / _pura:.2f}x  ·  a rede captura {_pura / _s:.0%} da busca"
    else:
        _linha += f"  ·  {_s / _pura:.2f}x  ·  a rede está À FRENTE da busca aqui"
    print(_linha)

# `busca` é campo de primeira classe do registro desde o schema 2 — irmão de `final` e
# `melhor`, e não um canto de `meta`. A diferença não é organizacional: o que mora em
# `meta` não passa por `validate()`, e isto **é um resultado**.
registro.record.busca = {**(registro.record.busca or {}), **_medidas}
_oficiais = [k for k, v in _medidas.items()
             if v.get("episodes") == EPISODIOS_DO_CONTRATO and v.get("completo")]
print()
print("gravado em `busca` de", registro.save())
if _oficiais:
    print(f"entram na arena: {', '.join(_oficiais)}")
else:
    print(f"NENHUMA entrada entra na arena — o contrato pede {EPISODIOS_DO_CONTRATO} "
          "episodios completos. O que foi medido fica gravado e marcado como espiada.")

## O agente jogando

Um GIF vale mais que a curva para entender *como* o agente perde. Morrer preso no próprio
corpo e morrer de fome dão a mesma linha no gráfico e são problemas completamente
diferentes.


In [ ]:
# @title GIF
from IPython.display import Image, display

for semente in (7, 21, 42):
    caminho, score, motivo = render_episode(
        agente.politica(), caminho=f"episodio_last_s{semente}.gif", seed=semente,
        canal_fome=getattr(agente.env, "canal_fome", False))
    print(f"last · semente {semente}: score {score}, terminou por {motivo}")
    display(Image(filename=caminho))

## Exportar — os dois

`.keras` para retomar treino, TFLite fp16/int8 para embarcar no jogo. A paridade de **ação**
contra o `.keras` é conferida — diferença numérica de quantização é aceitável, ação
diferente não é.

A conferência é pulada quando a política **tem memória** (o SOAP, com a crença de opção; o
DreamerV3, com o latente do modelo do mundo). Não é um detalhe de implementação: um
`.tflite` que recebe só a observação não consegue reproduzir uma política cuja ação depende
de estado interno, então "as ações batem" seria uma afirmação sobre outra coisa. Os arquivos
continuam sendo gerados e medidos; o que não se afirma é a paridade.

Exporta `last` **e** `best`, em pastas separadas. Exportar é para usar, e o que você leva
para o jogo é o melhor; mas o `last` vai junto porque é ele que corresponde ao número da
arena, e misturar os dois é como se perde a rastreabilidade entre o gráfico e o arquivo.


In [ ]:
# @title Exportar
relatorios = {}
# `apos_passo` é o contrato das políticas com memória (ver `snakeai/eval.py`). Quem o
# expõe não pode ter a paridade de ação conferida contra um `.tflite` sem estado.
_COM_MEMORIA = hasattr(agente.politica(), "apos_passo")
if _COM_MEMORIA:
    print("política com memória: TFLite exportado, paridade de ação não conferida")

relatorios["last"] = export_model(
    agente.model, out_dir=os.path.join(PASTA, "export", "last"),
    validar=not _COM_MEMORIA)

_melhor = agente.modelo_melhor()
if _melhor is not None:
    relatorios["best"] = export_model(
        _melhor, out_dir=os.path.join(PASTA, "export", "best"),
        validar=not _COM_MEMORIA)

print(json.dumps(relatorios, indent=2, ensure_ascii=False))

## Onde ficou o resultado

O `history.json` da execução vai para `runs/<algo>/<variante>/seed<N>/`, junto com a curva e
os GIFs. Essa pasta é o que entra na arena: coloque em `runs/` do repositório e rode
`python -m snakeai.arena --all`.

Ele carrega os dois resultados: `final` (o modelo do último passo, que é o número oficial)
e `melhor` (o melhor checkpoint, com o passo em que apareceu). Junto vão `modelos/last.keras`
e `modelos/best.keras` — a pasta é autossuficiente, quem a recebe consegue rodar o agente
sem depender de nada que ficou nesta máquina.

Sobre versionar isso no GitHub: o registro vai (`history.json`, `curva.png` e os GIFs), os
**pesos não**. Um `.keras` vai de 0,8 MB (`resnet_small`) a 6,7 MB (`cnn_rainbow` com dueling
e C51), e a arena inteira passa de 100 MB só de modelo — binário em git **nunca some do
histórico**, então cada re-execução deixaria mais uma cópia lá para sempre. O `.gitignore` já
tira `runs/**/*.keras` e `runs/**/*.npz`; o lugar deles é um *Release* do GitHub, que é feito
para binário e não entra no clone. Os arquivos continuam na sua pasta — o que muda é só o que
o git carrega.


In [ ]:
# @title Conferir o contrato
CAMINHO_REGISTRO = registro.save(skip_validation=True)
print("registro:", CAMINHO_REGISTRO)

problemas = validate(registro.record)
print("entra na arena?" , "sim" if not problemas else "NÃO:")
for p in problemas:
    print("  -", p)

_f = registro.record.final.get("score_mean")
_m = registro.record.melhor.get("score_mean")
if _f is not None and _m is not None:
    print()
    print(f"last  {_f:.2f}   (passo {registro.record.steps()[-1]:,})")
    print(f"best  {_m:.2f}   (passo {registro.record.melhor.get('global_step', 0):,})")
    if _m > _f:
        print(f"→ a execução terminou {_m - _f:.2f} abaixo do melhor que já esteve. "
              "Normal: RL profundo não melhora monotonicamente.")

## Baixar o resultado

Um `.zip` só, com a pasta inteira da execução — registro, curva, GIFs e o modelo exportado.

**Um arquivo, e não vários downloads**, por dois motivos: o navegador bloqueia downloads
múltiplos disparados em sequência, e a pasta da execução só faz sentido inteira — o
`history.json` sem a curva e sem os GIFs perde metade do que ela responde.

A entrega muda com a plataforma, e o `.zip` existe nos dois casos:

* **Colab** — dispara o download pelo navegador, o que exige a aba aberta. Se ela não
  estiver, a célula imprime o caminho em vez de falhar: o download é conveniência, o
  arquivo é o resultado.
* **Kaggle** — não há o que disparar, e é por isso que ele aguenta execução headless: o
  que está em `/kaggle/working` aparece sozinho no painel **Output**, à direita, e é
  baixável de lá com a aba fechada.


In [ ]:
# @title Baixar tudo num .zip
import shutil

PASTA_EXECUCAO = os.path.dirname(CAMINHO_REGISTRO)

# o export mora fora da pasta da execução; copiamos para dentro antes de zipar,
# senão o .zip sai sem o modelo — que é justamente o que se leva para o jogo
_export = os.path.join(PASTA, "export")
if os.path.isdir(_export):
    shutil.copytree(_export, os.path.join(PASTA_EXECUCAO, "export"), dirs_exist_ok=True)

_nome = "_".join([registro.record.algo, registro.record.variant,
                  f"seed{registro.record.seed}"])
ZIP = shutil.make_archive(os.path.join(PASTA, _nome), "zip", PASTA_EXECUCAO)
print(f"{ZIP}  ({os.path.getsize(ZIP) / 1e6:.1f} MB)")
for _raiz, _, _arqs in os.walk(PASTA_EXECUCAO):
    for _a in sorted(_arqs):
        print("   ", os.path.relpath(os.path.join(_raiz, _a), PASTA_EXECUCAO))

entregar_arquivo(ZIP)